<div style="direction: rtl; line-height: 1.8;">
<h1>NB12g — بازسازی تمیز و آزمایشی Golden Dataset</h1>
<p>این نوت‌بوک مسیر اصلی مرحلهٔ ۱۲ است و از نوت‌بوک‌های 12 تا 12f و آزمایش 12g قبلی، فقط artifactهای معتبر را می‌خواند. هیچ نسخهٔ جدیدی از Golden Dataset منتشر یا فعال نمی‌شود.</p>
<p>هدف این اجرا ساخت یک آزمایش قابل‌بررسی است: هر سؤال از یک quote دقیق در chunk واقعی تولید می‌شود، پاسخ نیز span دقیق همان quote است. بنابراین هیچ سؤال آزاد، answer آزاد یا قالب عمومی وارد آزمایش نمی‌شود.</p>
</div>


<div style="direction: rtl; line-height: 1.8;">
<h2>۱. قرارداد اجرا</h2>
<ul>
<li>Golden Dataset v1 فقط خوانده می‌شود.</li>
<li>نام هیچ خروجی با <code>golden_dataset_vN.jsonl</code> شروع نمی‌شود.</li>
<li>NB13 فعلی پس از اجرای این notebook همچنان v1 را می‌خواند.</li>
<li>همهٔ cacheها با hash محتوا شناخته می‌شوند، نه نام notebook یا state kernel.</li>
</ul>
</div>


In [1]:
# Code Cell 01
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from urllib.parse import urlparse
import hashlib, importlib.util, json, math, os, random, re, time, uuid
import numpy as np

SOURCE_DATASET_FILENAME = "golden_dataset_v1.jsonl"
EXPERIMENT_NAME = "golden_dataset_quote_grounded_experiment"
QUESTION_PROMPT_VERSION = "nb12g_clean_quote_question_v1"
QUESTIONS_PER_QUOTE = 1
MAX_QUOTES_PER_SEED = 3
RANDOM_SEED = 14050621
random.seed(RANDOM_SEED)

cwd = Path.cwd().resolve()
project_root = next((p for p in (cwd, *cwd.parents)
                     if (p / "notebooks" / "phase2").is_dir() and (p / "data" / "phase2").is_dir()), None)
assert project_root is not None, "Open this notebook from inside the RAG_finance repository."

golden_dir = project_root / "notebooks" / "phase2" / "golden_dataset"
v1_path = golden_dir / SOURCE_DATASET_FILENAME
assert v1_path.is_file(), f"Missing source dataset: {v1_path}"
print("Project root:", project_root)
print("Read-only Golden input:", v1_path)
print("Active-dataset creation:", False)
print("\nEXECUTION CONTRACT: PASS")


Project root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Read-only Golden input: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_v1.jsonl
Active-dataset creation: False

EXECUTION CONTRACT: PASS


In [2]:
# Code Cell 02
def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def read_jsonl(path):
    rows=[]
    with path.open(encoding="utf-8") as f:
        for line_no,line in enumerate(f,1):
            if line.strip():
                try: row=json.loads(line)
                except json.JSONDecodeError as e: raise AssertionError(f"Invalid JSONL {path}:{line_no}") from e
                assert isinstance(row,dict)
                rows.append(row)
    return rows

def write_json_atomic(path, value):
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding="utf-8")
    tmp.replace(path)

def write_jsonl_atomic(path, rows):
    tmp=path.with_suffix(path.suffix+".tmp")
    with tmp.open("w",encoding="utf-8") as f:
        for row in rows: f.write(json.dumps(row,ensure_ascii=False)+"\n")
    tmp.replace(path)

def canon(value):
    value=str(value or "").translate(str.maketrans({"ي":"ی","ى":"ی","ك":"ک","ة":"ه","ۀ":"ه","‌":" ","‏":" ","‎":" "})).lower()
    return re.sub(r"\s+", " ", value).strip()

def words(value):
    return re.findall(r"[\w\u0600-\u06ff]+",canon(value))

STOP={"از","به","در","با","برای","که","این","آن","را","و","یا","یک","می","شود","شده","است","هست","بود","باشد","های","ها","خود","چه","چرا","چگونه","درباره","بر","اگر","اما","هم"}
def content_words(value): return {w for w in words(value) if len(w)>2 and w not in STOP}

def parse_json_object(raw):
    raw=(raw or "").strip()
    raw=re.sub(r"^```(?:json)?\s*", "", raw)
    raw=re.sub(r"\s*```$", "", raw)
    dec=json.JSONDecoder()
    for i,c in enumerate(raw):
        if c=="{":
            try:
                item,_=dec.raw_decode(raw[i:])
                if isinstance(item,dict): return item
            except json.JSONDecodeError: pass
    return None

print("Utilities loaded.\nUTILITIES: PASS")


Utilities loaded.
UTILITIES: PASS


<div style="direction: rtl; line-height: 1.8;"><h2>۲. کنترل NB11 و استخراج seedهای معتبر</h2><p>این بخش به state notebookهای قبلی وابسته نیست و hash artifactهای freezeشده را کنترل می‌کند.</p></div>


In [3]:
# Code Cell 03
manifest_paths=sorted(project_root.rglob("evaluation_freeze_manifest_v1.json"))
assert len(manifest_paths)==1, f"Expected one NB11 manifest, found {len(manifest_paths)}"
manifest_path=manifest_paths[0]
manifest=read_json(manifest_path)
assert manifest.get("status")=="FROZEN_FOR_EVALUATION"
artifacts=manifest["artifacts"]
bad=[]
for name,d in artifacts.items():
    p=project_root/d["path"]
    if not p.is_file() or p.stat().st_size != d.get("bytes") or sha256_file(p) != d.get("sha256"):
        bad.append(name)
assert not bad, f"NB11 frozen artifacts failed: {bad}"

child_desc=artifacts["parent_child_children"]
parent_desc=artifacts["parent_child_parents"]
medium_desc=artifacts["structure_aware_medium"]
child_path=project_root/child_desc["path"]
parent_path=project_root/parent_desc["path"]
medium_path=project_root/medium_desc["path"]
print("NB11 manifest:",manifest_path.relative_to(project_root))
print("Validated frozen artifacts:",len(artifacts))
print("\nNB11 FREEZE: PASS")


NB11 manifest: reports/phase2/chunking/evaluation_freeze_manifest_v1.json
Validated frozen artifacts: 15

NB11 FREEZE: PASS


In [4]:
# Code Cell 04
v1=read_jsonl(v1_path)
required={"candidate_id","question","expected_answer","source_seed_id","source_lineage","source_evidence","taxonomy_topics","competency_ids"}
assert all(required <= set(r) for r in v1), "v1 schema is incomplete"
by_seed=defaultdict(list)
for r in v1: by_seed[int(r["source_seed_id"])].append(r)
assert sorted(by_seed)==list(range(1,29))

seeds={}
for seed_id,records in sorted(by_seed.items()):
    evidences={canon(r["source_evidence"]):r["source_evidence"] for r in records}
    answers={canon(r["expected_answer"]):r["expected_answer"] for r in records}
    assert len(evidences)==1 and len(answers)==1
    seeds[seed_id]={
        "seed_id":seed_id,
        "lineage":f"pilot_{seed_id}",
        "approved_evidence":next(iter(evidences.values())),
        "approved_answer":next(iter(answers.values())),
        "taxonomy_topics":sorted({x for r in records for x in r["taxonomy_topics"]}),
        "competency_ids":sorted({x for r in records for x in r["competency_ids"]}),
        "legacy_questions":[r["question"] for r in records],
    }
def seed_split(seed_id):
    bucket=int.from_bytes(hashlib.sha256(f"nb12g-clean-split:{seed_id}".encode()).digest()[:4],"big") % 100
    return "train" if bucket < 60 else "dev" if bucket < 80 else "test"
split_by_seed={seed_id:seed_split(seed_id) for seed_id in seeds}
assert set(split_by_seed.values()) == {"train","dev","test"}
print("v1 records:",len(v1),"| seeds:",len(seeds),"| v1 SHA:",sha256_file(v1_path))
print("Seed-level split:",dict(Counter(split_by_seed.values())))
print("Legacy questions used for generation:",False)
print("\nSEED FREEZE: PASS")


v1 records: 573 | seeds: 28 | v1 SHA: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729
Seed-level split: {'train': 19, 'dev': 3, 'test': 6}
Legacy questions used for generation: False

SEED FREEZE: PASS


<div style="direction: rtl; line-height: 1.8;"><h2>۳. provenance و quoteهای واقعی</h2><p>برخلاف آزمایش قبلی، quoteها قبل از تولید سؤال و به‌شکل deterministic از chunk واقعی انتخاب می‌شوند. مدل اجازه ندارد quote بسازد یا تغییر دهد.</p></div>


In [5]:
# Code Cell 05
registry_paths = [
    path
    for path in project_root.rglob("human_review_registry.json")
    if "before_" not in path.name
]
assert registry_paths, "Human-review registry is missing."
registry_path = max(registry_paths, key=lambda path: path.stat().st_mtime)

source_packet_paths = sorted(
    project_root.rglob("source_packet_pilots23_28_v1.json")
)
assert len(source_packet_paths) == 1, (
    "Expected exactly one source packet for pilots 23–28."
)

registry = read_json(registry_path)
source_packet = read_json(source_packet_paths[0])
completed_cases = registry.get("completed_human_review_cases", [])
source_packets = {
    int(packet["reference_pilot_number"]): packet
    for packet in source_packet.get("pilots", [])
}

assert len(completed_cases) == 22
assert sorted(source_packets) == list(range(23, 29))

provenance = {}

for seed_id, review_case in enumerate(completed_cases, 1):
    document_ids = sorted(
        {
            item.get("document_id")
            for item in review_case.get("source_evidence", [])
            if isinstance(item, dict) and item.get("document_id")
        }
    )

    provenance[seed_id] = {
        "pilot_case_id": review_case["pilot_case_id"],
        "source": "human_review_registry",
        "document_ids": document_ids,
        "raw": review_case,
    }

for seed_id in range(23, 29):
    packet = source_packets[seed_id]
    claim_document_ids = {
        claim.get("content_source", {}).get("document_id")
        for claim in packet.get("claims", [])
        if isinstance(claim.get("content_source"), dict)
    }

    provenance[seed_id] = {
        "pilot_case_id": packet["pilot_case_id"],
        "source": "source_packet",
        "document_ids": sorted(
            {
                document_id
                for document_id in (
                    set(packet.get("accepted_website_document_ids", []))
                    | claim_document_ids
                )
                if document_id
            }
        ),
        "raw": packet,
    }

assert sorted(provenance) == list(range(1, 29))

children = read_jsonl(child_path)
parents = read_jsonl(parent_path)
medium = read_jsonl(medium_path)
parent_by_id = {record["chunk_id"]: record for record in parents}

child_by_document = defaultdict(list)

for chunk in children:
    identifiers = set(chunk.get("member_document_ids", []))

    if chunk.get("source_id"):
        identifiers.add(chunk["source_id"])

    for document_id in identifiers:
        child_by_document[document_id].append(chunk)

def coverage(reference, text):
    reference_words = content_words(reference)

    if not reference_words:
        return 0.0

    return len(reference_words & content_words(text)) / len(reference_words)

source_chunks = {}

for seed_id, seed in seeds.items():
    matching_chunks = []
    seen_chunk_ids = set()

    for document_id in provenance[seed_id]["document_ids"]:
        for chunk in child_by_document.get(document_id, []):
            if chunk["chunk_id"] in seen_chunk_ids:
                continue

            seen_chunk_ids.add(chunk["chunk_id"])
            matching_chunks.append(chunk)

    linkage_basis = "REGISTERED_DOCUMENT_ID"

    if not matching_chunks:
        reviewed_evidence_units = [
            canon(item.get("text", ""))
            for item in provenance[seed_id]["raw"].get(
                "source_evidence", []
            )
            if isinstance(item, dict)
            and len(canon(item.get("text", ""))) >= 100
        ]

        matching_chunks = [
            chunk
            for chunk in medium
            if any(
                evidence_unit in canon(chunk["content"])
                for evidence_unit in reviewed_evidence_units
            )
        ]
        linkage_basis = "EXACT_REVIEWED_EVIDENCE_FALLBACK"

    assert matching_chunks, f"No source chunks found for seed {seed_id}."

    ranked_chunks = sorted(
        (
            (coverage(seed["approved_evidence"], chunk["content"]), chunk)
            for chunk in matching_chunks
        ),
        key=lambda item: (-item[0], item[1]["chunk_id"]),
    )

    best_coverage = ranked_chunks[0][0]
    relative_threshold = best_coverage * 0.72
    selected_chunks = [
        chunk
        for score, chunk in ranked_chunks
        if score >= relative_threshold
    ][:4]

    # Always preserve the best direct-provenance chunk.
    if not selected_chunks:
        selected_chunks = [ranked_chunks[0][1]]

    source_chunks[seed_id] = {
        "basis": linkage_basis,
        "selected": selected_chunks,
        "matching": [chunk for _, chunk in ranked_chunks],
        "best_coverage": float(best_coverage),
    }

print("Seeds with provenance:", len(source_chunks))
print(
    "Weak lexical links:",
    [
        seed_id
        for seed_id, result in source_chunks.items()
        if result["best_coverage"] < 0.18
    ],
)
print("\nPROVENANCE AND SOURCE CHUNKS: PASS")


Seeds with provenance: 28
Weak lexical links: [22]

PROVENANCE AND SOURCE CHUNKS: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۳. انتخاب quote واقعی و ترمیم کنترل‌شده</h2>
<p>ابتدا جمله‌های دقیق از chunkهای متصل به provenance انتخاب می‌شوند. اگر یک جمله به‌تنهایی کافی نباشد، پنجره‌های یک تا سه‌جمله‌ای از همان منبع مستقیم بررسی می‌شوند. هیچ متن ساختگی و هیچ evidence قدیمی به‌عنوان quote جایگزین نمی‌شود.</p>
</div>


In [6]:
# Code Cell 06
SINGLE_SENTENCE_MIN_COVERAGE = 0.15
DIRECT_WINDOW_MIN_COVERAGE = 0.16

def sentences(text):
    parts = re.split(r"(?<=[.!؟])\s+|\n+", str(text))
    return [
        re.sub(r"\s+", " ", part).strip()
        for part in parts
        if len(words(part)) >= 6
    ]

def sentence_windows(text, maximum_sentences=3):
    source_sentences = sentences(text)
    windows = []

    for start_index in range(len(source_sentences)):
        for width in range(1, maximum_sentences + 1):
            end_index = start_index + width

            if end_index > len(source_sentences):
                continue

            window = " ".join(
                source_sentences[start_index:end_index]
            ).strip()

            if 8 <= len(words(window)) <= 110:
                windows.append(window)

    return windows

def quote_record(seed_id, quote_index, score, quote, chunk, basis):
    return {
        "quote_id": f"seed{seed_id:02d}_quote{quote_index:02d}",
        "source_seed_id": seed_id,
        "quote": quote,
        "quote_hash": hashlib.sha256(
            canon(quote).encode("utf-8")
        ).hexdigest(),
        "quote_coverage": float(score),
        "chunk_id": chunk["chunk_id"],
        "parent_chunk_id": chunk.get("parent_chunk_id"),
        "label_basis": basis,
    }

quote_ledger = []
initially_unresolved_seed_ids = []

for seed_id, seed in seeds.items():
    candidates = []

    for chunk in source_chunks[seed_id]["selected"]:
        for sentence in sentences(chunk["content"]):
            score = coverage(seed["approved_evidence"], sentence)

            if score >= SINGLE_SENTENCE_MIN_COVERAGE:
                candidates.append((score, sentence, chunk))

    candidates.sort(
        key=lambda item: (
            -item[0],
            -len(words(item[1])),
            item[2]["chunk_id"],
        )
    )

    chosen = []
    seen_quotes = set()

    for score, quote, chunk in candidates:
        canonical_quote = canon(quote)

        if canonical_quote in seen_quotes:
            continue

        seen_quotes.add(canonical_quote)
        chosen.append((score, quote, chunk))

        if len(chosen) == MAX_QUOTES_PER_SEED:
            break

    if not chosen:
        initially_unresolved_seed_ids.append(seed_id)
        continue

    for quote_index, (score, quote, chunk) in enumerate(chosen, 1):
        quote_ledger.append(
            quote_record(
                seed_id,
                quote_index,
                score,
                quote,
                chunk,
                source_chunks[seed_id]["basis"],
            )
        )

print("Initial exact-source quotes:", len(quote_ledger))
print("Initially unresolved seeds:", initially_unresolved_seed_ids)
print("\nINITIAL QUOTE SELECTION: PASS")


Initial exact-source quotes: 44
Initially unresolved seeds: [1, 7, 11, 14, 20, 22, 26, 27, 28]

INITIAL QUOTE SELECTION: PASS


In [7]:
# Code Cell 07
def is_direct_provenance_chunk(chunk, registered_document_ids):
    chunk_document_ids = set(chunk.get("member_document_ids", []))

    if chunk.get("source_id"):
        chunk_document_ids.add(chunk["source_id"])

    return bool(set(registered_document_ids) & chunk_document_ids)

all_frozen_retrieval_chunks = []
seen_frozen_chunk_ids = set()

for chunk in [*children, *medium]:
    if chunk["chunk_id"] in seen_frozen_chunk_ids:
        continue

    seen_frozen_chunk_ids.add(chunk["chunk_id"])
    all_frozen_retrieval_chunks.append(chunk)

resolved_quote_windows = []
source_review_queue = []

for seed_id in initially_unresolved_seed_ids:
    seed = seeds[seed_id]
    registered_document_ids = provenance[seed_id]["document_ids"]
    window_candidates = []

    direct_chunks = [
        chunk
        for chunk in all_frozen_retrieval_chunks
        if is_direct_provenance_chunk(
            chunk,
            registered_document_ids,
        )
    ]

    for chunk in direct_chunks:
        for window in sentence_windows(chunk["content"]):
            window_candidates.append(
                (
                    coverage(seed["approved_evidence"], window),
                    window,
                    chunk,
                )
            )

    window_candidates.sort(
        key=lambda item: (
            -item[0],
            not item[2]["chunk_id"].startswith(
                "chunk:parent_child:child:"
            ),
            -len(words(item[1])),
            item[2]["chunk_id"],
        )
    )

    eligible_windows = [
        candidate
        for candidate in window_candidates
        if candidate[0] >= DIRECT_WINDOW_MIN_COVERAGE
    ]

    if eligible_windows:
        score, quote, chunk = eligible_windows[0]
        resolved_quote_windows.append(
            quote_record(
                seed_id,
                1,
                score,
                quote,
                chunk,
                "DIRECT_PROVENANCE_WINDOW_REPAIR",
            )
        )
        continue

    best_candidate = window_candidates[0] if window_candidates else None
    source_review_queue.append(
        {
            "source_seed_id": seed_id,
            "pilot_case_id": provenance[seed_id]["pilot_case_id"],
            "quote_id": None,
            "quote": best_candidate[1] if best_candidate else None,
            "question": None,
            "answer_span": None,
            "review_reasons": [
                "direct_source_quote_requires_content_adjudication"
            ],
            "best_direct_window_coverage": (
                float(best_candidate[0]) if best_candidate else 0.0
            ),
            "best_direct_chunk_id": (
                best_candidate[2]["chunk_id"]
                if best_candidate
                else None
            ),
            "human_decision": None,
            "human_notes": None,
        }
    )

all_quote_ledger = [*quote_ledger, *resolved_quote_windows]
all_quote_ledger.sort(
    key=lambda item: (
        item["source_seed_id"],
        item["quote_id"],
    )
)

assert len(
    {item["quote_hash"] for item in all_quote_ledger}
) == len(all_quote_ledger), "Duplicate quote texts were selected."

resolved_seed_ids = sorted(
    {item["source_seed_id"] for item in resolved_quote_windows}
)
manual_review_seed_ids = sorted(
    {item["source_seed_id"] for item in source_review_queue}
)

assert not (set(resolved_seed_ids) & set(manual_review_seed_ids))
assert sorted(
    set(resolved_seed_ids) | set(manual_review_seed_ids)
) == sorted(initially_unresolved_seed_ids)

print("Window-resolved seeds:", resolved_seed_ids)
print("Seeds remaining in source review:", manual_review_seed_ids)
print("Final quote candidates:", len(all_quote_ledger))
print("\nDIRECT QUOTE-WINDOW RESOLUTION: PASS")


Window-resolved seeds: [1, 11, 14, 20, 26, 27, 28]
Seeds remaining in source review: [7, 22]
Final quote candidates: 51

DIRECT QUOTE-WINDOW RESOLUTION: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۴. تولید سؤال و پاسخ کوتاه با cache پایدار</h2>
<p>cache با hash دیتاست، نسخهٔ prompt، مدل و hash خود quote شناخته می‌شود؛ بنابراین تغییر نام notebook یا Restart/Run All باعث اجرای دوبارهٔ درخواست‌های موفق قبلی نمی‌شود.</p>
</div>


In [8]:
# Code Cell 08
from dotenv import load_dotenv
from openai import OpenAI

assert importlib.util.find_spec("openai") is not None
assert importlib.util.find_spec("dotenv") is not None

environment_path = project_root / ".env"
assert environment_path.is_file(), "Project .env file is missing."
load_dotenv(environment_path, override=False)

api_key = os.getenv("ARVAN_AI_API_KEY")
chat_url = os.getenv("ARVAN_CHAT_URL")
chat_model = os.getenv("ARVAN_CHAT_MODEL")

assert api_key
assert chat_url
assert chat_model
assert urlparse(chat_url).scheme == "https"

chat = OpenAI(
    api_key=api_key,
    base_url=chat_url,
    timeout=180,
    max_retries=1,
)

dataset_sha256 = sha256_file(v1_path)
manifest_sha256 = sha256_file(manifest_path)
cache_namespace_payload = {
    "dataset_sha256": dataset_sha256,
    "prompt_version": QUESTION_PROMPT_VERSION,
    "model": chat_model,
}
cache_namespace = hashlib.sha256(
    json.dumps(
        cache_namespace_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:24]

artifact_root = (
    project_root
    / "artifacts"
    / "phase2"
    / "golden_dataset_rebuild_clean"
)
shared_cache_directory = (
    artifact_root
    / "shared_question_cache"
    / cache_namespace
)
shared_cache_directory.mkdir(parents=True, exist_ok=True)

build_identity = {
    "v1_sha256": dataset_sha256,
    "nb11_manifest_sha256": manifest_sha256,
    "prompt_version": QUESTION_PROMPT_VERSION,
    "model": chat_model,
    "quote_hashes": [
        item["quote_hash"]
        for item in all_quote_ledger
    ],
}
build_fingerprint = hashlib.sha256(
    json.dumps(build_identity, sort_keys=True).encode("utf-8")
).hexdigest()

run_directory = artifact_root / "runs" / build_fingerprint
run_directory.mkdir(parents=True, exist_ok=True)

legacy_cache_directories = sorted(
    path
    for path in artifact_root.glob("*/question_cache")
    if path.is_dir()
)

print("Chat model:", chat_model)
print("Build fingerprint:", build_fingerprint)
print("Stable cache namespace:", cache_namespace)
print("Legacy cache directories discovered:", len(legacy_cache_directories))
print("\nMODEL AND CACHE PREFLIGHT: PASS")


Chat model: Qwen3-30B-A3B
Build fingerprint: a0f3cc56c178ee5ab4a8d7229d5747eff347fd72cbac46ba29e06fe95c1ca161
Stable cache namespace: 4bf4113103410cf667037ff8
Legacy cache directories discovered: 1

MODEL AND CACHE PREFLIGHT: PASS


In [9]:
# Code Cell 09
FORBIDDEN_QUESTION_PATTERNS = (
    "طبق متن",
    "بر اساس متن",
    "در این متن",
    "در این بخش",
    "منبع",
    "کدام یک از موارد زیر",
    "چه نکته ای باید بررسی شود",
    "چه مفهومی باید درک شود",
)
ALLOWED_INTENTS = (
    "DIRECT_FACT",
    "EXPLANATION",
    "CONSEQUENCE",
    "COMPARISON",
)

def prompt_for_quote(item):
    return f"""
فقط بر پایهٔ QUOTE زیر یک پرسش فارسی طبیعی و مستقل بساز.
پاسخ باید عبارتی کوتاه باشد که عیناً و بدون تغییر در QUOTE وجود دارد.
هیچ دانش، اصطلاح، مثال، عدد یا موضوع بیرون از QUOTE اضافه نکن.
از عبارت‌های «طبق متن»، «در این بخش» و «منبع» استفاده نکن.
سؤال تستی بدون گزینه نساز.
اگر quote برای پرسشی روشن مناسب نیست، usable را false بگذار.
خروجی فقط یک JSON معتبر باشد:
{{"usable":true,"intent":"DIRECT_FACT|EXPLANATION|CONSEQUENCE|COMPARISON","question":"...؟","answer_span":"..."}}

QUOTE:
{item['quote']}
""".strip()

def validate_payload(payload, item):
    errors = []

    if not isinstance(payload, dict):
        return ["payload_not_object"]

    if payload.get("usable") is not True:
        return ["model_marked_not_usable"]

    question = str(payload.get("question", "")).strip()
    answer = str(payload.get("answer_span", "")).strip()

    if payload.get("intent") not in ALLOWED_INTENTS:
        errors.append("invalid_intent")

    if not question.endswith("؟"):
        errors.append("question_mark")

    if not 4 <= len(words(question)) <= 40:
        errors.append("question_length")

    if any(
        canon(pattern) in canon(question)
        for pattern in FORBIDDEN_QUESTION_PATTERNS
    ):
        errors.append("generic_or_invalid_form")

    if not 1 <= len(words(answer)) <= 28:
        errors.append("answer_length")

    if canon(answer) not in canon(item["quote"]):
        errors.append("answer_not_exact_span")

    if not (content_words(question) & content_words(item["quote"])):
        errors.append("question_has_no_quote_signal")

    return sorted(set(errors))

def call_question(prompt):
    response = chat.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": "Return only one valid JSON object.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0.1,
        max_tokens=800,
    )

    assert response.choices
    raw_response = (
        response.choices[0].message.content or ""
    ).strip()

    return (
        raw_response,
        parse_json_object(raw_response),
        response.choices[0].finish_reason,
    )

def discover_cached_record(quote_hash):
    canonical_path = shared_cache_directory / f"{quote_hash}.json"
    candidate_paths = [canonical_path]
    candidate_paths.extend(
        directory / f"{quote_hash}.json"
        for directory in legacy_cache_directories
    )

    for candidate_path in candidate_paths:
        if candidate_path.is_file():
            return candidate_path, read_json(candidate_path)

    return canonical_path, None

generation = []
cache_reuse_count = 0
live_request_count = 0

print("=== RESUMABLE QUOTE-GROUNDED GENERATION ===")

for position, item in enumerate(all_quote_ledger, 1):
    cache_path, generation_record = discover_cached_record(
        item["quote_hash"]
    )

    cached_errors = (
        validate_payload(generation_record.get("payload"), item)
        if isinstance(generation_record, dict)
        else ["no_cache"]
    )

    if not cached_errors:
        cache_reuse_count += 1
        canonical_cache_path = (
            shared_cache_directory / f"{item['quote_hash']}.json"
        )

        if cache_path != canonical_cache_path:
            write_json_atomic(canonical_cache_path, generation_record)
    else:
        base_prompt = prompt_for_quote(item)
        current_errors = cached_errors

        for attempt in range(1, 4):
            retry_prompt = base_prompt

            if attempt > 1:
                retry_prompt += (
                    "\nخطاهای پاسخ قبلی: "
                    + json.dumps(current_errors, ensure_ascii=False)
                    + "\nفقط JSON معتبر و کاملاً مبتنی بر quote بده."
                )

            try:
                raw_response, payload, finish_reason = call_question(
                    retry_prompt
                )
                live_request_count += 1
                current_errors = validate_payload(payload, item)
            except Exception as error:
                live_request_count += 1
                raw_response = ""
                payload = None
                finish_reason = None
                current_errors = [
                    "model_request_error:" + type(error).__name__
                ]
            generation_record = {
                "quote_id": item["quote_id"],
                "quote_hash": item["quote_hash"],
                "attempt": attempt,
                "model": chat_model,
                "payload": payload,
                "raw_response": raw_response,
                "finish_reason": finish_reason,
                "errors": current_errors,
            }
            write_json_atomic(
                shared_cache_directory / f"{item['quote_hash']}.json",
                generation_record,
            )

            if not current_errors:
                break

    generation.append({**item, **generation_record})
    status = "PASS" if not generation_record["errors"] else "REVIEW"
    print(
        f"Quote {position}/{len(all_quote_ledger)} | "
        f"seed={item['source_seed_id']} | {status}"
    )

assert len(generation) == len(all_quote_ledger)

print("\nCached successful generations reused:", cache_reuse_count)
print("Live model requests:", live_request_count)
print(
    "Machine-valid generations:",
    sum(not item["errors"] for item in generation),
)
print(
    "Generation review items:",
    sum(bool(item["errors"]) for item in generation),
)
print("\nQUOTE-GROUNDED GENERATION: COMPLETE")


=== RESUMABLE QUOTE-GROUNDED GENERATION ===
Quote 1/51 | seed=1 | PASS
Quote 2/51 | seed=2 | PASS
Quote 3/51 | seed=2 | PASS
Quote 4/51 | seed=3 | PASS
Quote 5/51 | seed=3 | PASS
Quote 6/51 | seed=3 | PASS
Quote 7/51 | seed=4 | PASS
Quote 8/51 | seed=4 | PASS
Quote 9/51 | seed=4 | PASS
Quote 10/51 | seed=5 | PASS
Quote 11/51 | seed=5 | PASS
Quote 12/51 | seed=6 | PASS
Quote 13/51 | seed=6 | PASS
Quote 14/51 | seed=8 | PASS
Quote 15/51 | seed=8 | PASS
Quote 16/51 | seed=8 | PASS
Quote 17/51 | seed=9 | PASS
Quote 18/51 | seed=9 | PASS
Quote 19/51 | seed=10 | PASS
Quote 20/51 | seed=10 | PASS
Quote 21/51 | seed=10 | PASS
Quote 22/51 | seed=11 | PASS
Quote 23/51 | seed=12 | PASS
Quote 24/51 | seed=12 | PASS
Quote 25/51 | seed=12 | PASS
Quote 26/51 | seed=13 | PASS
Quote 27/51 | seed=13 | PASS
Quote 28/51 | seed=13 | PASS
Quote 29/51 | seed=14 | PASS
Quote 30/51 | seed=15 | PASS
Quote 31/51 | seed=16 | PASS
Quote 32/51 | seed=16 | PASS
Quote 33/51 | seed=16 | PASS
Quote 34/51 | seed=17 | PA

<div style="direction: rtl; line-height: 1.8;">
<h2>۵. ساخت آزمایش غیر‌فعال و صف بازبینی</h2>
<p>رکورد آزمایشی تنها زمانی ساخته می‌شود که پاسخ span دقیق quote باشد و سؤال تمام کنترل‌های ماشینی را بگذراند. source_evidence رکورد جدید خودِ quote واقعی است؛ evidence قدیمی فقط با نام legacy_seed_evidence برای ممیزی نگهداری می‌شود.</p>
</div>


In [10]:
# Code Cell 10
experiment = []
review_queue = list(source_review_queue)
question_counts = Counter()

for item in generation:
    payload = item.get("payload") or {}

    if item["errors"]:
        continue

    question_counts[canon(payload["question"])] += 1

child_chunk_ids = {chunk["chunk_id"] for chunk in children}
medium_chunk_ids = {chunk["chunk_id"] for chunk in medium}

for item in generation:
    payload = item.get("payload") or {}
    errors = list(item["errors"])

    if (
        not errors
        and question_counts[canon(payload["question"])] > 1
    ):
        errors.append("duplicate_question")

    if errors:
        review_queue.append(
            {
                "quote_id": item["quote_id"],
                "source_seed_id": item["source_seed_id"],
                "quote": item["quote"],
                "question": payload.get("question"),
                "answer_span": payload.get("answer_span"),
                "review_reasons": sorted(set(errors)),
                "human_decision": None,
                "human_notes": None,
            }
        )
        continue

    seed = seeds[item["source_seed_id"]]
    candidate_id = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            (
                f"{build_fingerprint}:{item['quote_hash']}:"
                f"{canon(payload['question'])}"
            ),
        )
    )

    if item["chunk_id"] in child_chunk_ids:
        retrieval_candidate = "parent_child_children"
    elif item["chunk_id"] in medium_chunk_ids:
        retrieval_candidate = "structure_aware_medium"
    else:
        raise AssertionError(
            f"Unknown frozen chunk label: {item['chunk_id']}"
        )

    experiment.append(
        {
            "candidate_id": candidate_id,
            "dataset_version": (
                "golden_dataset_quote_grounded_experiment"
            ),
            "source_seed_id": item["source_seed_id"],
            "source_lineage": seed["lineage"],
            "question": payload["question"].strip(),
            "expected_answer": payload["answer_span"].strip(),
            "question_intent": payload["intent"],
            "evaluation_split": split_by_seed[item["source_seed_id"]],
            "source_evidence": item["quote"],
            "evidence_quote": item["quote"],
            "legacy_seed_evidence": seed["approved_evidence"],
            "taxonomy_topics": seed["taxonomy_topics"],
            "competency_ids": seed["competency_ids"],
            "pilot_case_id": provenance[
                item["source_seed_id"]
            ]["pilot_case_id"],
            "provenance_source": provenance[
                item["source_seed_id"]
            ]["source"],
            "document_ids": provenance[
                item["source_seed_id"]
            ]["document_ids"],
            "retrieval_labels": {
                "candidate": retrieval_candidate,
                "positive_chunk_ids": [item["chunk_id"]],
                "positive_parent_chunk_ids": (
                    [item["parent_chunk_id"]]
                    if item["parent_chunk_id"]
                    else []
                ),
                "label_basis": item["label_basis"],
            },
            "generation": {
                "model": chat_model,
                "prompt_version": QUESTION_PROMPT_VERSION,
                "build_fingerprint": build_fingerprint,
            },
            "generation_status": "EXPERIMENT_MACHINE_VALIDATED",
        }
    )

assert len(
    {record["candidate_id"] for record in experiment}
) == len(experiment)
assert len(
    {canon(record["question"]) for record in experiment}
) == len(experiment)

audit_size = max(1, math.ceil(len(experiment) * 0.10))
audit_records = sorted(
    experiment,
    key=lambda record: hashlib.sha256(
        ("audit:" + record["candidate_id"]).encode("utf-8")
    ).hexdigest(),
)[:audit_size]

for record in audit_records:
    review_queue.append(
        {
            "candidate_id": record["candidate_id"],
            "source_seed_id": record["source_seed_id"],
            "quote": record["evidence_quote"],
            "question": record["question"],
            "answer_span": record["expected_answer"],
            "review_reasons": ["random_quality_audit"],
            "human_decision": None,
            "human_notes": None,
        }
    )

experiment_path = golden_dir / f"{EXPERIMENT_NAME}.jsonl"
review_path = golden_dir / f"{EXPERIMENT_NAME}_review_queue.jsonl"
write_jsonl_atomic(experiment_path, experiment)
write_jsonl_atomic(review_path, review_queue)

print("Final quote candidates:", len(all_quote_ledger))
print("Machine-valid experiment records:", len(experiment))
print("Source-review records:", len(source_review_queue))
print(
    "Generation-review records:",
    sum(bool(item["errors"]) for item in generation),
)
print("Random audit records:", len(audit_records))
print("Total review queue records:", len(review_queue))
print("Experiment:", experiment_path)
print("\nNON-ACTIVE EXPERIMENT ASSEMBLY: PASS")


Final quote candidates: 51
Machine-valid experiment records: 51
Source-review records: 2
Generation-review records: 0
Random audit records: 6
Total review queue records: 8
Experiment: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_experiment.jsonl

NON-ACTIVE EXPERIMENT ASSEMBLY: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۶. diagnostic بازیابی</h2>
<p>این سنجش فقط یک diagnostic داخلی برای بررسی سازگاری سؤال با chunk برچسب‌خورده است. نتیجهٔ آن معادل benchmark نهایی NB13 نیست و rank برای حذف یا پذیرش رکورد استفاده نمی‌شود.</p>
</div>


In [11]:
# Code Cell 11
assert importlib.util.find_spec("sklearn") is not None
from sklearn.feature_extraction.text import TfidfVectorizer

child_ids = [chunk["chunk_id"] for chunk in children]
child_index_by_id = {
    chunk_id: index
    for index, chunk_id in enumerate(child_ids)
}
child_texts = [canon(chunk["content"]) for chunk in children]

vectorizer = TfidfVectorizer(
    tokenizer=words,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    norm="l2",
)
passage_matrix = vectorizer.fit_transform(child_texts)
diagnostic_rows = []

for record in experiment:
    labels = record["retrieval_labels"]

    if labels["candidate"] != "parent_child_children":
        diagnostic_rows.append(
            {
                "candidate_id": record["candidate_id"],
                "source_seed_id": record["source_seed_id"],
                "first_positive_rank": None,
                "excluded_reason": "no_parent_child_label",
            }
        )
        continue

    positive_chunk_id = labels["positive_chunk_ids"][0]
    assert positive_chunk_id in child_index_by_id

    query_vector = vectorizer.transform(
        [canon(record["question"])]
    )
    scores = (passage_matrix @ query_vector.T).toarray().ravel()
    ranking = np.argsort(-scores, kind="stable")
    positive_index = child_index_by_id[positive_chunk_id]
    rank = int(np.where(ranking == positive_index)[0][0]) + 1

    diagnostic_rows.append(
        {
            "candidate_id": record["candidate_id"],
            "source_seed_id": record["source_seed_id"],
            "first_positive_rank": rank,
            "hit_at_1": rank <= 1,
            "hit_at_3": rank <= 3,
            "hit_at_5": rank <= 5,
            "hit_at_10": rank <= 10,
            "reciprocal_rank": 1 / rank,
        }
    )

eligible_diagnostics = [
    row
    for row in diagnostic_rows
    if row["first_positive_rank"] is not None
]
assert eligible_diagnostics, "No parent-child diagnostics are available."

metrics = {
    "eligible_queries": len(eligible_diagnostics),
    "recall_at_1": float(
        np.mean([row["hit_at_1"] for row in eligible_diagnostics])
    ),
    "recall_at_3": float(
        np.mean([row["hit_at_3"] for row in eligible_diagnostics])
    ),
    "recall_at_5": float(
        np.mean([row["hit_at_5"] for row in eligible_diagnostics])
    ),
    "recall_at_10": float(
        np.mean([row["hit_at_10"] for row in eligible_diagnostics])
    ),
    "mrr": float(
        np.mean(
            [row["reciprocal_rank"] for row in eligible_diagnostics]
        )
    ),
}

print("=== LEXICAL RETRIEVAL DIAGNOSTIC ===")

for metric_name, metric_value in metrics.items():
    print(metric_name, ":", metric_value)

print("Rank was not used as an acceptance criterion.")
print("This diagnostic is not the final NB13 benchmark.")
print("\nRETRIEVAL DIAGNOSTIC: PASS")


=== LEXICAL RETRIEVAL DIAGNOSTIC ===
eligible_queries : 48
recall_at_1 : 0.6666666666666666
recall_at_3 : 0.7916666666666666
recall_at_5 : 0.8333333333333334
recall_at_10 : 0.8958333333333334
mrr : 0.7357520986606301
Rank was not used as an acceptance criterion.
This diagnostic is not the final NB13 benchmark.

RETRIEVAL DIAGNOSTIC: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۷. کنترل نهایی artifactها و handoff</h2>
<p>در پایان، تمام رکوردهای آزمایشی دوباره از نظر span پاسخ، اتصال به chunk frozen، یکتایی و عدم ساخت نسخهٔ فعال کنترل می‌شوند.</p>
</div>


In [12]:
# Code Cell 12
current_v1_sha256 = sha256_file(v1_path)
assert current_v1_sha256 == dataset_sha256, (
    "Golden Dataset v1 changed during execution."
)

all_frozen_chunk_ids = child_chunk_ids | medium_chunk_ids

for record in experiment:
    assert record["source_evidence"] == record["evidence_quote"]
    assert canon(record["expected_answer"]) in canon(
        record["source_evidence"]
    )
    assert record["question"].endswith("؟")
    assert record["retrieval_labels"]["positive_chunk_ids"]
    assert set(
        record["retrieval_labels"]["positive_chunk_ids"]
    ) <= all_frozen_chunk_ids

assert len(read_jsonl(experiment_path)) == len(experiment)
assert len(read_jsonl(review_path)) == len(review_queue)

active_version_pattern = re.compile(r"^golden_dataset_v\d+\.jsonl$")
assert not active_version_pattern.match(experiment_path.name)
assert experiment_path.name != SOURCE_DATASET_FILENAME

report = {
    "schema_version": (
        "golden_dataset_quote_grounded_experiment_report_v2"
    ),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "build_fingerprint": build_fingerprint,
    "source": {
        "dataset": str(v1_path.relative_to(project_root)),
        "sha256": dataset_sha256,
        "records": len(v1),
        "legacy_questions_used": False,
        "source_dataset_modified": False,
    },
    "experiment": {
        "path": str(experiment_path.relative_to(project_root)),
        "sha256": sha256_file(experiment_path),
        "records": len(experiment),
        "quote_candidates": len(all_quote_ledger),
        "represented_seed_ids": sorted(
            {record["source_seed_id"] for record in experiment}
        ),
        "active_dataset_created": False,
    },
    "quality": {
        "initial_quote_count": len(quote_ledger),
        "window_repair_quote_count": len(resolved_quote_windows),
        "manual_source_review_seed_ids": sorted(
            {item["source_seed_id"] for item in source_review_queue}
        ),
        "generation_review_count": sum(
            bool(item["errors"]) for item in generation
        ),
        "random_audit_count": len(audit_records),
        "review_queue_records": len(review_queue),
        "exact_answer_span_required": True,
        "source_evidence_is_exact_quote": True,
    },
    "cache": {
        "namespace": cache_namespace,
        "successful_records_reused": cache_reuse_count,
        "live_requests": live_request_count,
        "notebook_filename_affects_cache": False,
    },
    "retrieval_diagnostic": {
        "model": "persian_tfidf_lexical",
        "candidate": "parent_child_children",
        "metrics": metrics,
        "scope": "INTERNAL_LABEL_CONSISTENCY_DIAGNOSTIC",
        "selection_bias_guard": "RANK_NOT_USED_FOR_ACCEPTANCE",
    },
    "status": "EXPERIMENT_COMPLETE_NOT_A_GOLDEN_VERSION",
    "production_approved": False,
}

report_path = golden_dir / f"{EXPERIMENT_NAME}_quality_report.json"
diagnostic_path = (
    golden_dir / f"{EXPERIMENT_NAME}_retrieval_diagnostic.json"
)
manifest_output_path = (
    golden_dir / f"{EXPERIMENT_NAME}_build_manifest.json"
)

write_json_atomic(report_path, report)
write_json_atomic(
    diagnostic_path,
    {
        "metrics": metrics,
        "rows": diagnostic_rows,
        "scope": "INTERNAL_LABEL_CONSISTENCY_DIAGNOSTIC",
    },
)
write_json_atomic(
    manifest_output_path,
    {
        **build_identity,
        "build_fingerprint": build_fingerprint,
        "experiment_path": str(
            experiment_path.relative_to(project_root)
        ),
        "experiment_sha256": sha256_file(experiment_path),
        "creates_active_dataset": False,
    },
)

required_outputs = (
    experiment_path,
    review_path,
    report_path,
    diagnostic_path,
    manifest_output_path,
)
assert all(path.is_file() for path in required_outputs)

print("=== NB12G CLEAN FINAL HANDOFF ===")
print("Status:", report["status"])
print("Active v1 unchanged:", current_v1_sha256 == dataset_sha256)
print("Active dataset created:", False)
print("Experiment records:", len(experiment))
print("Review queue records:", len(review_queue))
print(
    "Manual source-review seeds:",
    report["quality"]["manual_source_review_seed_ids"],
)
print("Notebook filename affects cache:", False)
print("Experiment:", experiment_path)
print("Review queue:", review_path)
print("Quality report:", report_path)
print("\nNB12G CLEAN FINAL HANDOFF: PASS")


=== NB12G CLEAN FINAL HANDOFF ===
Status: EXPERIMENT_COMPLETE_NOT_A_GOLDEN_VERSION
Active v1 unchanged: True
Active dataset created: False
Experiment records: 51
Review queue records: 8
Manual source-review seeds: [7, 22]
Notebook filename affects cache: False
Experiment: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_experiment.jsonl
Review queue: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_experiment_review_queue.jsonl
Quality report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_experiment_quality_report.json

NB12G CLEAN FINAL HANDOFF: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>مرز مرحلهٔ بعد</h2>
<p>این notebook هیچ Golden Dataset v2 نمی‌سازد و NB13 را اجرا نمی‌کند. خروجی فعلی فقط یک آزمایش quote-grounded است. seedهای باقی‌مانده در صف review و نمونه‌های audit باید بررسی شوند. پس از بازخورد مؤلف، ساخت v2 در یک مرحلهٔ مستقل و مقایسهٔ صریح v1، experiment و v2 انجام خواهد شد.</p>
</div>


In [13]:
# Code Cell 13
# NB12G — Deterministic semantic-quality audit

assert "experiment" in globals(), "Run Code Cell 10 first."
assert "experiment_path" in globals(), "Run Code Cell 10 first."
assert len(experiment) > 0, "Experiment is empty."

SEMANTIC_AUDIT_VERSION = "nb12g_semantic_audit_v1"

GENERIC_QUESTION_PATTERNS = (
    "چه نکته‌ای باید بررسی شود",
    "چه نکته ای باید بررسی شود",
    "چه مفهومی باید درک شود",
    "درباره چه چیزی",
    "طبق متن",
    "بر اساس متن",
    "در این متن",
    "در این بخش",
    "با توجه به منبع",
)

DEICTIC_PATTERNS = (
    "این موضوع",
    "این مورد",
    "این کار",
    "این وضعیت",
    "این مسئله",
    "آن موضوع",
    "آن مورد",
    "آن‌ها",
    "آنها",
    "موارد گفته‌شده",
    "موارد گفته شده",
)

WEAK_QUESTION_ENDINGS = (
    "چیست؟",
    "کدام است؟",
    "چه بود؟",
    "چه می‌شود؟",
    "چه میشود؟",
)

INCOMPLETE_ANSWER_ENDINGS = (
    "و",
    "یا",
    "که",
    "از",
    "به",
    "در",
    "با",
    "برای",
    "اما",
    "اگر",
    "تا",
)

def normalized_content_tokens(value):
    return [
        token
        for token in words(value)
        if len(token) > 2 and token not in STOP
    ]

def repeated_adjacent_token(value):
    tokens = words(value)

    return any(
        tokens[index] == tokens[index + 1]
        for index in range(len(tokens) - 1)
    )

def question_answer_overlap(question, answer):
    question_tokens = set(normalized_content_tokens(question))
    answer_tokens = set(normalized_content_tokens(answer))

    if not answer_tokens:
        return 0.0

    return len(question_tokens & answer_tokens) / len(answer_tokens)

def inspect_semantic_record(record):
    question = str(record.get("question", "")).strip()
    answer = str(record.get("expected_answer", "")).strip()
    evidence = str(record.get("source_evidence", "")).strip()

    blocking_errors = []
    review_flags = []

    # Fundamental structural checks
    if not question:
        blocking_errors.append("empty_question")

    if not answer:
        blocking_errors.append("empty_answer")

    if not evidence:
        blocking_errors.append("empty_source_evidence")

    if question and not question.endswith("؟"):
        blocking_errors.append("missing_persian_question_mark")

    if answer and canon(answer) not in canon(evidence):
        blocking_errors.append("answer_not_exact_evidence_span")

    # Question-form checks
    generic_matches = [
        pattern
        for pattern in GENERIC_QUESTION_PATTERNS
        if canon(pattern) in canon(question)
    ]

    if generic_matches:
        blocking_errors.append("generic_question_template")

    deictic_matches = [
        pattern
        for pattern in DEICTIC_PATTERNS
        if canon(pattern) in canon(question)
    ]

    if deictic_matches:
        review_flags.append("question_contains_deictic_reference")

    if repeated_adjacent_token(question):
        blocking_errors.append("repeated_adjacent_question_token")

    if repeated_adjacent_token(answer):
        review_flags.append("repeated_adjacent_answer_token")

    question_token_count = len(words(question))
    answer_token_count = len(words(answer))
    evidence_token_count = len(words(evidence))

    if question_token_count < 4:
        blocking_errors.append("question_too_short")

    if question_token_count > 40:
        review_flags.append("question_too_long")

    if answer_token_count < 1:
        blocking_errors.append("answer_too_short")

    if answer_token_count > 28:
        review_flags.append("answer_too_long")

    if evidence_token_count < 6:
        blocking_errors.append("evidence_too_short")

    # Detect incomplete answer fragments
    answer_tokens = words(answer)

    if answer_tokens and answer_tokens[-1] in INCOMPLETE_ANSWER_ENDINGS:
        blocking_errors.append("answer_ends_with_incomplete_connector")

    if answer.startswith(("،", ".", "؛", ":", "و ")):
        review_flags.append("answer_has_suspicious_start")

    # Detect question/answer tautology
    overlap_ratio = question_answer_overlap(question, answer)

    if (
        answer_token_count >= 2
        and overlap_ratio >= 0.90
    ):
        blocking_errors.append("question_answer_tautology")

    elif overlap_ratio >= 0.70:
        review_flags.append("high_question_answer_overlap")

    # Weak questions are not automatically rejected, but require review
    if question.endswith(WEAK_QUESTION_ENDINGS):
        question_signal = set(normalized_content_tokens(question))
        evidence_signal = set(normalized_content_tokens(evidence))

        if len(question_signal & evidence_signal) <= 1:
            review_flags.append("weak_low_information_question")

    # A usable question needs a lexical connection to its evidence
    question_signal = set(normalized_content_tokens(question))
    evidence_signal = set(normalized_content_tokens(evidence))

    lexical_overlap = question_signal & evidence_signal

    if not lexical_overlap:
        blocking_errors.append("question_has_no_evidence_signal")

    # Detect suspiciously copied questions
    if canon(question.rstrip("؟")) in canon(evidence):
        review_flags.append("question_copied_from_evidence")

    blocking_errors = sorted(set(blocking_errors))
    review_flags = sorted(set(review_flags))

    if blocking_errors:
        audit_status = "BLOCK"
    elif review_flags:
        audit_status = "REVIEW"
    else:
        audit_status = "PASS"

    return {
        "candidate_id": record["candidate_id"],
        "source_seed_id": record["source_seed_id"],
        "question": question,
        "expected_answer": answer,
        "source_evidence": evidence,
        "question_token_count": question_token_count,
        "answer_token_count": answer_token_count,
        "evidence_token_count": evidence_token_count,
        "question_answer_overlap": overlap_ratio,
        "question_evidence_shared_tokens": sorted(lexical_overlap),
        "generic_pattern_matches": generic_matches,
        "deictic_pattern_matches": deictic_matches,
        "blocking_errors": blocking_errors,
        "review_flags": review_flags,
        "audit_status": audit_status,
    }

semantic_audit_rows = [
    inspect_semantic_record(record)
    for record in experiment
]

assert len(semantic_audit_rows) == len(experiment)
assert len({
    row["candidate_id"]
    for row in semantic_audit_rows
}) == len(semantic_audit_rows)

semantic_audit_status_counts = Counter(
    row["audit_status"]
    for row in semantic_audit_rows
)

semantic_blocking_reason_counts = Counter(
    reason
    for row in semantic_audit_rows
    for reason in row["blocking_errors"]
)

semantic_review_reason_counts = Counter(
    reason
    for row in semantic_audit_rows
    for reason in row["review_flags"]
)

semantic_pass_rows = [
    row
    for row in semantic_audit_rows
    if row["audit_status"] == "PASS"
]

semantic_review_rows = [
    row
    for row in semantic_audit_rows
    if row["audit_status"] == "REVIEW"
]

semantic_blocked_rows = [
    row
    for row in semantic_audit_rows
    if row["audit_status"] == "BLOCK"
]

semantic_repair_rows = [
    row
    for row in semantic_audit_rows
    if row["audit_status"] in {"BLOCK", "REVIEW"}
]

semantic_audit_report = {
    "audit_version": SEMANTIC_AUDIT_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "active_dataset_created": False,
    "source_experiment_path": str(experiment_path),
    "source_experiment_sha256": sha256_file(experiment_path),
    "record_count": len(semantic_audit_rows),
    "status_counts": dict(semantic_audit_status_counts),
    "blocking_reason_counts": dict(
        semantic_blocking_reason_counts
    ),
    "review_reason_counts": dict(
        semantic_review_reason_counts
    ),
    "rows": semantic_audit_rows,
}

semantic_audit_report_path = (
    golden_dir
    / f"{EXPERIMENT_NAME}_semantic_audit.json"
)

write_json_atomic(
    semantic_audit_report_path,
    semantic_audit_report,
)

print("=== DETERMINISTIC SEMANTIC-QUALITY AUDIT ===")
print("Records audited:", len(semantic_audit_rows))
print("PASS:", len(semantic_pass_rows))
print("REVIEW:", len(semantic_review_rows))
print("BLOCK:", len(semantic_blocked_rows))
print("Repair candidates:", len(semantic_repair_rows))

print("\nBlocking reasons:")
if semantic_blocking_reason_counts:
    for reason, count in semantic_blocking_reason_counts.most_common():
        print(f"- {reason}: {count}")
else:
    print("- None")

print("\nReview reasons:")
if semantic_review_reason_counts:
    for reason, count in semantic_review_reason_counts.most_common():
        print(f"- {reason}: {count}")
else:
    print("- None")

print("\nFirst 15 non-PASS records:")
for row in semantic_repair_rows[:15]:
    print(
        f"- seed={row['source_seed_id']} | "
        f"status={row['audit_status']} | "
        f"blocking={row['blocking_errors']} | "
        f"review={row['review_flags']}"
    )
    print("  Question:", row["question"])
    print("  Answer:", row["expected_answer"])

print("\nReport:", semantic_audit_report_path)
print("\nDETERMINISTIC SEMANTIC-QUALITY AUDIT: PASS")

=== DETERMINISTIC SEMANTIC-QUALITY AUDIT ===
Records audited: 51
PASS: 39
REVIEW: 8
BLOCK: 4
Repair candidates: 12

Blocking reasons:
- question_answer_tautology: 4

Review reasons:
- high_question_answer_overlap: 5
- question_copied_from_evidence: 3
- question_contains_deictic_reference: 2

First 15 non-PASS records:
- seed=1 | status=REVIEW | blocking=[] | review=['high_question_answer_overlap']
  Question: چرا استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد؟
  Answer: استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد
- seed=3 | status=REVIEW | blocking=[] | review=['question_contains_deictic_reference']
  Question: آن‌ها باید چه چیزی را تجسم کنند؟
  Answer: دو حالت تجسم کنند
- seed=4 | status=BLOCK | blocking=['question_answer_tautology'] | review=[]
  Question: پول توجیبی به چند بخش تقسیم می‌شود؟
  Answer: سه بخش
- seed=5 | status=BLOCK | blocking=['question_answer_tautology'] | review=[]
  Question: درآمد خالص ماهانه و توان بازپرداخت اقساط چیست؟
  Answer: درآمد خالص 

In [14]:
# Code Cell 14
# NB12G — Calibrated semantic audit and repair queue

assert "semantic_audit_rows" in globals(), (
    "Run Code Cell 13 first."
)
assert len(semantic_audit_rows) == len(experiment)

CALIBRATED_AUDIT_VERSION = "nb12g_semantic_audit_v2"

def calibrated_tautology_check(question, answer):
    question_tokens = set(normalized_content_tokens(question))
    answer_tokens = set(normalized_content_tokens(answer))

    # Short factual answers such as «سه بخش» or «بیمه‌گر»
    # must not be classified as tautological only because one
    # important word also appears in the question.
    if len(answer_tokens) < 4:
        return False

    overlap_ratio = (
        len(question_tokens & answer_tokens) / len(answer_tokens)
        if answer_tokens
        else 0.0
    )

    answer_to_question_length_ratio = (
        len(words(answer)) / len(words(question))
        if words(question)
        else 0.0
    )

    answer_without_punctuation = canon(
        re.sub(r"[؟?!.,،؛:]+$", "", answer)
    )
    question_without_punctuation = canon(
        re.sub(r"[؟?!.,،؛:]+$", "", question)
    )

    exact_copy = (
        answer_without_punctuation
        == question_without_punctuation
    )

    near_copy = (
        overlap_ratio >= 0.85
        and answer_to_question_length_ratio >= 0.55
    )

    return exact_copy or near_copy

def calibrated_inspect_semantic_record(record):
    original = inspect_semantic_record(record)

    question = original["question"]
    answer = original["expected_answer"]

    blocking_errors = [
        error
        for error in original["blocking_errors"]
        if error != "question_answer_tautology"
    ]

    review_flags = [
        flag
        for flag in original["review_flags"]
        if flag != "high_question_answer_overlap"
    ]

    calibrated_overlap = question_answer_overlap(
        question,
        answer,
    )

    answer_content_token_count = len(
        set(normalized_content_tokens(answer))
    )

    if calibrated_tautology_check(question, answer):
        blocking_errors.append("question_answer_tautology")

    elif (
        answer_content_token_count >= 4
        and calibrated_overlap >= 0.70
    ):
        review_flags.append("high_question_answer_overlap")

    # A copied answer fragment that does not form a useful standalone
    # answer must remain in the repair queue.
    if "question_copied_from_evidence" in review_flags:
        if len(words(answer)) <= 5:
            review_flags.append("possibly_incomplete_answer_fragment")

    blocking_errors = sorted(set(blocking_errors))
    review_flags = sorted(set(review_flags))

    if blocking_errors:
        audit_status = "BLOCK"
        repair_priority = 1
    elif review_flags:
        audit_status = "REVIEW"
        repair_priority = 2
    else:
        audit_status = "PASS"
        repair_priority = 3

    return {
        **original,
        "blocking_errors": blocking_errors,
        "review_flags": review_flags,
        "audit_status": audit_status,
        "repair_priority": repair_priority,
        "answer_content_token_count": (
            answer_content_token_count
        ),
        "calibrated_question_answer_overlap": (
            calibrated_overlap
        ),
    }

calibrated_semantic_audit_rows = [
    calibrated_inspect_semantic_record(record)
    for record in experiment
]

assert len(calibrated_semantic_audit_rows) == len(experiment)

calibrated_status_counts = Counter(
    row["audit_status"]
    for row in calibrated_semantic_audit_rows
)

calibrated_blocking_reason_counts = Counter(
    reason
    for row in calibrated_semantic_audit_rows
    for reason in row["blocking_errors"]
)

calibrated_review_reason_counts = Counter(
    reason
    for row in calibrated_semantic_audit_rows
    for reason in row["review_flags"]
)

calibrated_pass_rows = [
    row
    for row in calibrated_semantic_audit_rows
    if row["audit_status"] == "PASS"
]

calibrated_review_rows = [
    row
    for row in calibrated_semantic_audit_rows
    if row["audit_status"] == "REVIEW"
]

calibrated_blocked_rows = [
    row
    for row in calibrated_semantic_audit_rows
    if row["audit_status"] == "BLOCK"
]

semantic_repair_queue = sorted(
    [
        row
        for row in calibrated_semantic_audit_rows
        if row["audit_status"] in {"BLOCK", "REVIEW"}
    ],
    key=lambda row: (
        row["repair_priority"],
        row["source_seed_id"],
        row["candidate_id"],
    ),
)

calibrated_audit_report = {
    "audit_version": CALIBRATED_AUDIT_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "active_dataset_created": False,
    "source_experiment_path": str(experiment_path),
    "source_experiment_sha256": sha256_file(
        experiment_path
    ),
    "record_count": len(
        calibrated_semantic_audit_rows
    ),
    "status_counts": dict(
        calibrated_status_counts
    ),
    "blocking_reason_counts": dict(
        calibrated_blocking_reason_counts
    ),
    "review_reason_counts": dict(
        calibrated_review_reason_counts
    ),
    "repair_queue_count": len(
        semantic_repair_queue
    ),
    "rows": calibrated_semantic_audit_rows,
}

calibrated_audit_report_path = (
    golden_dir
    / f"{EXPERIMENT_NAME}_semantic_audit_v2.json"
)

write_json_atomic(
    calibrated_audit_report_path,
    calibrated_audit_report,
)

print("=== CALIBRATED SEMANTIC-QUALITY AUDIT ===")
print(
    "Records audited:",
    len(calibrated_semantic_audit_rows),
)
print("PASS:", len(calibrated_pass_rows))
print("REVIEW:", len(calibrated_review_rows))
print("BLOCK:", len(calibrated_blocked_rows))
print(
    "Repair queue:",
    len(semantic_repair_queue),
)

print("\nBlocking reasons:")
if calibrated_blocking_reason_counts:
    for reason, count in (
        calibrated_blocking_reason_counts.most_common()
    ):
        print(f"- {reason}: {count}")
else:
    print("- None")

print("\nReview reasons:")
if calibrated_review_reason_counts:
    for reason, count in (
        calibrated_review_reason_counts.most_common()
    ):
        print(f"- {reason}: {count}")
else:
    print("- None")

print("\nCalibrated repair queue:")
for position, row in enumerate(
    semantic_repair_queue,
    1,
):
    print(
        f"\n{position}. "
        f"seed={row['source_seed_id']} | "
        f"status={row['audit_status']}"
    )
    print(
        "   Blocking:",
        row["blocking_errors"],
    )
    print(
        "   Review:",
        row["review_flags"],
    )
    print(
        "   Question:",
        row["question"],
    )
    print(
        "   Answer:",
        row["expected_answer"],
    )

print("\nReport:", calibrated_audit_report_path)
print(
    "\nCALIBRATED SEMANTIC-QUALITY AUDIT: PASS"
)

=== CALIBRATED SEMANTIC-QUALITY AUDIT ===
Records audited: 51
PASS: 41
REVIEW: 5
BLOCK: 5
Repair queue: 10

Blocking reasons:
- question_answer_tautology: 5

Review reasons:
- question_copied_from_evidence: 3
- question_contains_deictic_reference: 2
- high_question_answer_overlap: 2

Calibrated repair queue:

1. seed=1 | status=BLOCK
   Blocking: ['question_answer_tautology']
   Review: []
   Question: چرا استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد؟
   Answer: استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد

2. seed=5 | status=BLOCK
   Blocking: ['question_answer_tautology']
   Review: []
   Question: درآمد خالص ماهانه و توان بازپرداخت اقساط چیست؟
   Answer: درآمد خالص ماهانه و توان بازپرداخت اقساط

3. seed=10 | status=BLOCK
   Blocking: ['question_answer_tautology']
   Review: ['question_copied_from_evidence']
   Question: کجا به ما آموزش دادن که بهتر تصمیم گیری مالی کنیم؟
   Answer: کجا به ما آموزش دادن که بهتر تصمیم گیری مالی کنیم

4. seed=10 | status=BLOCK
   B

In [15]:
# Code Cell 15
# NB12G — Controlled semantic repair of non-PASS records

assert "semantic_repair_queue" in globals(), (
    "Run Code Cell 14 first."
)
assert "chat" in globals(), (
    "Run Code Cell 08 first."
)
assert "chat_model" in globals(), (
    "Run Code Cell 08 first."
)
assert len(semantic_repair_queue) == 10

SEMANTIC_REPAIR_PROMPT_VERSION = (
    "nb12g_semantic_repair_v1"
)

semantic_repair_cache_directory = (
    golden_dir
    / "cache"
    / EXPERIMENT_NAME
    / "semantic_repairs"
    / build_fingerprint
    / SEMANTIC_REPAIR_PROMPT_VERSION
)

semantic_repair_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)

experiment_by_candidate_id = {
    record["candidate_id"]: record
    for record in experiment
}

assert len(experiment_by_candidate_id) == len(experiment)

def semantic_repair_prompt(audit_row):
    blocking_text = json.dumps(
        audit_row["blocking_errors"],
        ensure_ascii=False,
    )
    review_text = json.dumps(
        audit_row["review_flags"],
        ensure_ascii=False,
    )

    return f"""
فقط با استفاده از QUOTE، سؤال و پاسخ فعلی را اصلاح کن.

هدف:
- ساخت یک سؤال فارسی طبیعی، روشن و مستقل
- پاسخ‌پذیری کامل سؤال فقط از QUOTE
- پاسخ کوتاه و عیناً موجود در QUOTE
- پرهیز از تکرار پاسخ در خود سؤال
- پرهیز از ضمیرهای مبهم مانند «آن‌ها»، «این موضوع» و «این مورد»
- پرهیز از عبارت‌های «طبق متن»، «بر اساس متن»، «در این بخش» و «منبع»
- پرهیز از سؤال ناقص، محاوره‌ای نامفهوم یا سؤال دارای پاسخ درون خودش
- سؤال باید برای فردی که فقط سؤال را می‌بیند قابل فهم باشد

اگر QUOTE برای ساخت سؤال روشن مناسب نیست:
usable را false قرار بده.

خروجی فقط یک JSON معتبر باشد:
{{
  "usable": true,
  "intent": "DIRECT_FACT|EXPLANATION|CONSEQUENCE|COMPARISON",
  "question": "...؟",
  "answer_span": "..."
}}

خطاهای مسدودکننده فعلی:
{blocking_text}

موارد نیازمند بررسی:
{review_text}

سؤال فعلی:
{audit_row["question"]}

پاسخ فعلی:
{audit_row["expected_answer"]}

QUOTE:
{audit_row["source_evidence"]}
""".strip()

def inspect_repaired_payload(payload, audit_row):
    quote_item = {
        "quote": audit_row["source_evidence"],
    }

    deterministic_validation_errors = validate_payload(
        payload,
        quote_item,
    )

    if deterministic_validation_errors:
        return {
            "status": "BLOCK",
            "blocking_errors": (
                deterministic_validation_errors
            ),
            "review_flags": [],
        }

    temporary_record = {
        "candidate_id": audit_row["candidate_id"],
        "source_seed_id": audit_row["source_seed_id"],
        "question": payload["question"],
        "expected_answer": payload["answer_span"],
        "source_evidence": audit_row["source_evidence"],
    }

    calibrated_result = (
        calibrated_inspect_semantic_record(
            temporary_record
        )
    )

    return {
        "status": calibrated_result["audit_status"],
        "blocking_errors": (
            calibrated_result["blocking_errors"]
        ),
        "review_flags": (
            calibrated_result["review_flags"]
        ),
    }

def semantic_repair_cache_key(audit_row):
    cache_material = {
        "prompt_version": (
            SEMANTIC_REPAIR_PROMPT_VERSION
        ),
        "model": chat_model,
        "candidate_id": audit_row["candidate_id"],
        "source_evidence": canon(
            audit_row["source_evidence"]
        ),
        "original_question": canon(
            audit_row["question"]
        ),
        "original_answer": canon(
            audit_row["expected_answer"]
        ),
        "blocking_errors": audit_row[
            "blocking_errors"
        ],
        "review_flags": audit_row[
            "review_flags"
        ],
    }

    return hashlib.sha256(
        json.dumps(
            cache_material,
            ensure_ascii=False,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()

def load_valid_semantic_repair(
    audit_row,
    cache_path,
):
    if not cache_path.is_file():
        return None

    cached_record = read_json(cache_path)
    payload = cached_record.get("payload")

    validation = inspect_repaired_payload(
        payload,
        audit_row,
    )

    if validation["status"] == "PASS":
        return {
            **cached_record,
            "validation": validation,
            "cache_reused": True,
        }

    return None

semantic_repair_results = []
semantic_repair_cache_reuse_count = 0
semantic_repair_live_request_count = 0

print("=== CONTROLLED SEMANTIC REPAIR ===")
print(
    "Repair candidates:",
    len(semantic_repair_queue),
)

for position, audit_row in enumerate(
    semantic_repair_queue,
    1,
):
    cache_key = semantic_repair_cache_key(
        audit_row
    )

    cache_path = (
        semantic_repair_cache_directory
        / f"{cache_key}.json"
    )

    accepted_record = load_valid_semantic_repair(
        audit_row,
        cache_path,
    )

    if accepted_record is not None:
        semantic_repair_cache_reuse_count += 1

    else:
        accepted_record = None
        previous_validation = None

        for attempt in range(1, 4):
            prompt = semantic_repair_prompt(
                audit_row
            )

            if previous_validation is not None:
                prompt += (
                    "\n\nپاسخ قبلی هنوز معتبر نبود."
                    "\nخطاهای قبلی:"
                    "\n"
                    + json.dumps(
                        previous_validation,
                        ensure_ascii=False,
                    )
                    + "\nیک سؤال متفاوت و کاملاً سالم بساز."
                )

            try:
                (
                    raw_response,
                    payload,
                    finish_reason,
                ) = call_question(prompt)

                request_error = None

            except Exception as error:
                raw_response = ""
                payload = None
                finish_reason = None
                request_error = (
                    f"{type(error).__name__}: {error}"
                )

            semantic_repair_live_request_count += 1

            validation = inspect_repaired_payload(
                payload,
                audit_row,
            )

            attempt_record = {
                "prompt_version": (
                    SEMANTIC_REPAIR_PROMPT_VERSION
                ),
                "model": chat_model,
                "candidate_id": (
                    audit_row["candidate_id"]
                ),
                "source_seed_id": (
                    audit_row["source_seed_id"]
                ),
                "attempt": attempt,
                "payload": payload,
                "raw_response": raw_response,
                "finish_reason": finish_reason,
                "request_error": request_error,
                "validation": validation,
                "created_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
            }

            # Save every attempt separately for diagnosis.
            attempt_path = (
                semantic_repair_cache_directory
                / f"{cache_key}_attempt_{attempt}.json"
            )

            write_json_atomic(
                attempt_path,
                attempt_record,
            )

            previous_validation = validation

            if (
                request_error is None
                and validation["status"] == "PASS"
            ):
                accepted_record = {
                    **attempt_record,
                    "cache_reused": False,
                }

                write_json_atomic(
                    cache_path,
                    accepted_record,
                )
                break

    if accepted_record is None:
        final_result = {
            "candidate_id": (
                audit_row["candidate_id"]
            ),
            "source_seed_id": (
                audit_row["source_seed_id"]
            ),
            "original_question": (
                audit_row["question"]
            ),
            "original_answer": (
                audit_row["expected_answer"]
            ),
            "source_evidence": (
                audit_row["source_evidence"]
            ),
            "repair_status": "REVIEW",
            "repaired_payload": None,
            "validation": previous_validation,
            "cache_reused": False,
        }

    else:
        final_result = {
            "candidate_id": (
                audit_row["candidate_id"]
            ),
            "source_seed_id": (
                audit_row["source_seed_id"]
            ),
            "original_question": (
                audit_row["question"]
            ),
            "original_answer": (
                audit_row["expected_answer"]
            ),
            "source_evidence": (
                audit_row["source_evidence"]
            ),
            "repair_status": "PASS",
            "repaired_payload": (
                accepted_record["payload"]
            ),
            "validation": (
                accepted_record["validation"]
            ),
            "cache_reused": (
                accepted_record["cache_reused"]
            ),
        }

    semantic_repair_results.append(final_result)

    print(
        f"Repair {position}/"
        f"{len(semantic_repair_queue)} | "
        f"seed={audit_row['source_seed_id']} | "
        f"{final_result['repair_status']}"
    )

semantic_repair_pass_rows = [
    row
    for row in semantic_repair_results
    if row["repair_status"] == "PASS"
]

semantic_repair_review_rows = [
    row
    for row in semantic_repair_results
    if row["repair_status"] != "PASS"
]

assert len(semantic_repair_results) == len(
    semantic_repair_queue
)

semantic_repair_report = {
    "prompt_version": (
        SEMANTIC_REPAIR_PROMPT_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "source_experiment_path": str(
        experiment_path
    ),
    "source_experiment_sha256": sha256_file(
        experiment_path
    ),
    "repair_candidate_count": len(
        semantic_repair_results
    ),
    "repair_pass_count": len(
        semantic_repair_pass_rows
    ),
    "repair_review_count": len(
        semantic_repair_review_rows
    ),
    "cache_reuse_count": (
        semantic_repair_cache_reuse_count
    ),
    "live_request_count": (
        semantic_repair_live_request_count
    ),
    "results": semantic_repair_results,
}

semantic_repair_report_path = (
    golden_dir
    / f"{EXPERIMENT_NAME}_semantic_repair.json"
)

write_json_atomic(
    semantic_repair_report_path,
    semantic_repair_report,
)

print("\nRepair PASS:", len(
    semantic_repair_pass_rows
))
print("Repair REVIEW:", len(
    semantic_repair_review_rows
))
print(
    "Cached repairs reused:",
    semantic_repair_cache_reuse_count,
)
print(
    "Live model requests:",
    semantic_repair_live_request_count,
)

print("\nAccepted repairs:")
for row in semantic_repair_pass_rows:
    payload = row["repaired_payload"]

    print(
        f"\n- seed={row['source_seed_id']}"
    )
    print(
        "  Old question:",
        row["original_question"],
    )
    print(
        "  New question:",
        payload["question"],
    )
    print(
        "  New answer:",
        payload["answer_span"],
    )

if semantic_repair_review_rows:
    print("\nUnresolved repairs:")

    for row in semantic_repair_review_rows:
        print(
            f"- seed={row['source_seed_id']} | "
            f"validation={row['validation']}"
        )

print("\nReport:", semantic_repair_report_path)
print("\nCONTROLLED SEMANTIC REPAIR: COMPLETE")

=== CONTROLLED SEMANTIC REPAIR ===
Repair candidates: 10
Repair 1/10 | seed=1 | PASS
Repair 2/10 | seed=5 | REVIEW
Repair 3/10 | seed=10 | REVIEW
Repair 4/10 | seed=10 | REVIEW
Repair 5/10 | seed=13 | PASS
Repair 6/10 | seed=3 | PASS
Repair 7/10 | seed=9 | PASS
Repair 8/10 | seed=10 | PASS
Repair 9/10 | seed=16 | PASS
Repair 10/10 | seed=23 | REVIEW

Repair PASS: 6
Repair REVIEW: 4
Cached repairs reused: 2
Live model requests: 20

Accepted repairs:

- seed=1
  Old question: چرا استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد؟
  New question: چرا استفاده از فیل به عنوان ستون ممکن است مناسب به نظر نرسد؟
  New answer: فیل یک کل به‌هم‌ پیوسته است و در واقعیت، پای فیل جدای گوش و خرطوم و عاجش نیست

- seed=13
  Old question: آیا بعد از همه هزینه‌ها سود دیگری هم باقی مانده یا کسب‌وکار علاوه بر این، بدهکار شده است؟
  New question: چه مبلغی از حقوق برای خودم برداشت کرده‌ام؟
  New answer: 36 میلیون تومان

- seed=3
  Old question: آن‌ها باید چه چیزی را تجسم کنند؟
  New question: چه چیزی با

In [16]:
# Code Cell 16
# NB12G — Semantic repair evidence packet

assert "semantic_repair_queue" in globals(), (
    "Run Code Cell 14 first."
)
assert "semantic_repair_results" in globals(), (
    "Run Code Cell 15 first."
)

repair_result_by_candidate_id = {
    row["candidate_id"]: row
    for row in semantic_repair_results
}

assert len(repair_result_by_candidate_id) == len(
    semantic_repair_results
)

semantic_repair_evidence_packet = []

for position, audit_row in enumerate(
    semantic_repair_queue,
    1,
):
    repair_result = repair_result_by_candidate_id[
        audit_row["candidate_id"]
    ]

    proposed_payload = (
        repair_result.get("repaired_payload")
        or {}
    )

    packet_record = {
        "position": position,
        "candidate_id": audit_row["candidate_id"],
        "source_seed_id": audit_row["source_seed_id"],
        "audit_status": audit_row["audit_status"],
        "blocking_errors": audit_row[
            "blocking_errors"
        ],
        "review_flags": audit_row["review_flags"],
        "original_question": audit_row["question"],
        "original_answer": audit_row[
            "expected_answer"
        ],
        "source_evidence": audit_row[
            "source_evidence"
        ],
        "automatic_repair_status": repair_result[
            "repair_status"
        ],
        "proposed_question": proposed_payload.get(
            "question"
        ),
        "proposed_answer": proposed_payload.get(
            "answer_span"
        ),
        "human_decision": None,
        "human_question": None,
        "human_answer_span": None,
        "human_notes": None,
    }

    semantic_repair_evidence_packet.append(
        packet_record
    )

assert len(semantic_repair_evidence_packet) == 10

semantic_repair_evidence_packet_path = (
    golden_dir
    / f"{EXPERIMENT_NAME}_semantic_repair_evidence_packet.json"
)

write_json_atomic(
    semantic_repair_evidence_packet_path,
    {
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "active_dataset_created": False,
        "record_count": len(
            semantic_repair_evidence_packet
        ),
        "records": semantic_repair_evidence_packet,
    },
)

print("=== SEMANTIC REPAIR EVIDENCE PACKET ===")
print(
    "Records:",
    len(semantic_repair_evidence_packet),
)

for row in semantic_repair_evidence_packet:
    print("\n" + "=" * 80)
    print(
        f"POSITION {row['position']} | "
        f"SEED {row['source_seed_id']}"
    )

    print("\nOriginal question:")
    print(row["original_question"])

    print("\nOriginal answer:")
    print(row["original_answer"])

    print("\nSource evidence:")
    print(row["source_evidence"])

    print("\nAutomatic proposal:")
    print(
        "Question:",
        row["proposed_question"],
    )
    print(
        "Answer:",
        row["proposed_answer"],
    )

    print(
        "\nAudit reasons:",
        row["blocking_errors"]
        + row["review_flags"],
    )

print(
    "\nPacket:",
    semantic_repair_evidence_packet_path,
)
print(
    "\nSEMANTIC REPAIR EVIDENCE PACKET: PASS"
)

=== SEMANTIC REPAIR EVIDENCE PACKET ===
Records: 10

POSITION 1 | SEED 1

Original question:
چرا استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد؟

Original answer:
استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد

Source evidence:
تلقی مردم از فیل، به عنوان ناودان و ستون و تخت و بادبزن، آنجا تبدیل به مشکل می شود که بخواهند فیل را بر اساس همان تصویر خود، به کار بگیرند. استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد اما فیل یک کل به‌هم‌ پیوسته است و در واقعیت، پای فیل جدای گوش و خرطوم و عاجش نیست. چنین تصوری نه تنها کارآمد نیست که آسیب جدی نیز در پی خواهد داشت.

Automatic proposal:
Question: چرا استفاده از فیل به عنوان ستون ممکن است مناسب به نظر نرسد؟
Answer: فیل یک کل به‌هم‌ پیوسته است و در واقعیت، پای فیل جدای گوش و خرطوم و عاجش نیست

Audit reasons: ['question_answer_tautology']

POSITION 2 | SEED 5

Original question:
درآمد خالص ماهانه و توان بازپرداخت اقساط چیست؟

Original answer:
درآمد خالص ماهانه و توان بازپرداخت اقساط

Source evidence:
درآمد خالص ماهانه و ت

In [17]:
# Code Cell 17
# NB12G — Evidence-based manual adjudication

assert "semantic_repair_evidence_packet" in globals(), (
    "Run Code Cell 16 first."
)
assert len(semantic_repair_evidence_packet) == 10

MANUAL_ADJUDICATION_VERSION = (
    "nb12g_evidence_adjudication_v1"
)

packet_by_position = {
    row["position"]: row
    for row in semantic_repair_evidence_packet
}

manual_adjudication_spec = {
    1: {
        "decision": "REPAIR",
        "intent": "EXPLANATION",
        "question": (
            "چرا نمی‌توان یکی از اجزای فیل را "
            "کاملاً جدا از بقیه در نظر گرفت؟"
        ),
        "answer_span": (
            "فیل یک کل به‌هم‌ پیوسته است و در واقعیت، "
            "پای فیل جدای گوش و خرطوم و عاجش نیست."
        ),
        "reason": (
            "The evidence supports a self-contained "
            "question about the interdependence of parts."
        ),
    },

    2: {
        "decision": "REJECT",
        "reason": (
            "The evidence is only a noun phrase and "
            "does not contain an answerable proposition."
        ),
    },

    3: {
        "decision": "REJECT",
        "reason": (
            "The evidence is itself an incomplete "
            "spoken question and contains no answer."
        ),
    },

    4: {
        "decision": "REJECT",
        "reason": (
            "The evidence is an unsupported spoken "
            "claim fragment and is semantically unclear."
        ),
    },

    5: {
        "decision": "REPAIR",
        "intent": "DIRECT_FACT",
        "question": (
            "برای ارزیابی وضعیت مالی کسب‌وکار "
            "در پایان سال چه چیزی باید بررسی شود؟"
        ),
        "answer_span": (
            "آیا بعد از همه هزینه‌ها سود دیگری هم "
            "باقی مانده یا کسب‌وکار علاوه بر این، "
            "بدهکار شده است؟"
        ),
        "reason": (
            "The evidence explicitly states the "
            "end-of-year financial check."
        ),
    },
    6: {
        "decision": "REPAIR",
        "intent": "COMPARISON",
        "question": (
            "برای مقایسه لذت فوری با پس‌انداز، "
            "چه دو حالتی را می‌توان تجسم کرد؟"
        ),
        "answer_span": (
            "یکی زمانی که لذت خوردن بستنی "
            "(یا هر وسوسۀ بیرونی دیگر) را تجربه "
            "می‌کنند و دیگر، وقتی پول‌های‌شان "
            "را جمع کرده‌اند"
        ),
        "reason": (
            "The ambiguous pronoun was removed and "
            "a shorter exact evidence span was selected."
        ),
    },
    7: {
        "decision": "REPAIR",
        "intent": "DIRECT_FACT",
        "question": (
            "برای شناخت مقدمات فعالیت اقتصادی، "
            "آگاهی از چه منابعی لازم است؟"
        ),
        "answer_span": (
            "منابع مالی، انسانی، طبیعی و مواد اولیه"
        ),
        "reason": (
            "The ambiguous subject was removed while "
            "preserving the exact resource list."
        ),
    },

    8: {
        "decision": "REJECT",
        "reason": (
            "The evidence is an incomplete spoken "
            "question and contains no factual answer."
        ),
    },

    9: {
        "decision": "REPAIR",
        "intent": "DIRECT_FACT",
        "question": (
            "در مدیریت خرج مبتنی بر قناعت، "
            "پول باید برای چه چیزی هزینه شود؟"
        ),
        "answer_span": (
            "برای چیزی که واقعاً می‌خواهی و در "
            "چهارچوب ارزش‌ها و نیازهایت است"
        ),
        "reason": (
            "The repaired question asks directly about "
            "the spending rule stated in the evidence."
        ),
    },

    10: {
        "decision": "REJECT",
        "reason": (
            "The evidence contains only a question "
            "and provides no answer."
        ),
    },
}

assert set(manual_adjudication_spec) == set(
    packet_by_position
)

manual_adjudication_results = []

for position in sorted(manual_adjudication_spec):
    packet = packet_by_position[position]
    specification = manual_adjudication_spec[position]

    result = {
        "position": position,
        "candidate_id": packet["candidate_id"],
        "source_seed_id": packet["source_seed_id"],
        "source_evidence": packet["source_evidence"],
        "original_question": packet[
            "original_question"
        ],
        "original_answer": packet[
            "original_answer"
        ],
        "decision": specification["decision"],
        "reason": specification["reason"],
        "replacement": None,
        "validation": None,
    }

    if specification["decision"] == "REPAIR":
        replacement = {
            "usable": True,
            "intent": specification["intent"],
            "question": specification["question"],
            "answer_span": specification[
                "answer_span"
            ],
        }

        quote_item = {
            "quote": packet["source_evidence"],
        }

        deterministic_errors = validate_payload(
            replacement,
            quote_item,
        )

        temporary_record = {
            "candidate_id": packet["candidate_id"],
            "source_seed_id": packet[
                "source_seed_id"
            ],
            "question": replacement["question"],
            "expected_answer": replacement[
                "answer_span"
            ],
            "source_evidence": packet[
                "source_evidence"
            ],
        }

        semantic_validation = (
            calibrated_inspect_semantic_record(
                temporary_record
            )
        )

        validation = {
            "deterministic_errors": (
                deterministic_errors
            ),
            "semantic_status": (
                semantic_validation["audit_status"]
            ),
            "semantic_blocking_errors": (
                semantic_validation[
                    "blocking_errors"
                ]
            ),
            "semantic_review_flags": (
                semantic_validation[
                    "review_flags"
                ]
            ),
        }

        assert not deterministic_errors, (
            f"Position {position} failed deterministic "
            f"validation: {deterministic_errors}"
        )

        assert (
            semantic_validation["audit_status"]
            == "PASS"
        ), (
            f"Position {position} failed semantic "
            f"validation: {semantic_validation}"
        )

        result["replacement"] = replacement
        result["validation"] = validation

    manual_adjudication_results.append(result)

manual_repaired_results = [
    row
    for row in manual_adjudication_results
    if row["decision"] == "REPAIR"
]

manual_rejected_results = [
    row
    for row in manual_adjudication_results
    if row["decision"] == "REJECT"
]

assert len(manual_repaired_results) == 5
assert len(manual_rejected_results) == 5

manual_adjudication_report = {
    "adjudication_version": (
        MANUAL_ADJUDICATION_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "source_experiment_path": str(
        experiment_path
    ),
    "source_experiment_sha256": sha256_file(
        experiment_path
    ),
    "record_count": len(
        manual_adjudication_results
    ),
    "repair_count": len(
        manual_repaired_results
    ),
    "reject_count": len(
        manual_rejected_results
    ),
    "results": manual_adjudication_results,
}

manual_adjudication_report_path = (
    golden_dir
    / f"{EXPERIMENT_NAME}_manual_adjudication.json"
)

write_json_atomic(
    manual_adjudication_report_path,
    manual_adjudication_report,
)

print("=== EVIDENCE-BASED MANUAL ADJUDICATION ===")
print(
    "Records adjudicated:",
    len(manual_adjudication_results),
)
print(
    "Validated repairs:",
    len(manual_repaired_results),
)
print(
    "Rejected evidence fragments:",
    len(manual_rejected_results),
)

print("\nValidated repairs:")
for row in manual_repaired_results:
    replacement = row["replacement"]

    print(
        f"\n- seed={row['source_seed_id']}"
    )
    print(
        "  Question:",
        replacement["question"],
    )
    print(
        "  Answer:",
        replacement["answer_span"],
    )

print("\nRejected records:")
for row in manual_rejected_results:
    print(
        f"- seed={row['source_seed_id']} | "
        f"{row['reason']}"
    )

print(
    "\nReport:",
    manual_adjudication_report_path,
)
print(
    "\nEVIDENCE-BASED MANUAL ADJUDICATION: PASS"
)

=== EVIDENCE-BASED MANUAL ADJUDICATION ===
Records adjudicated: 10
Validated repairs: 5
Rejected evidence fragments: 5

Validated repairs:

- seed=1
  Question: چرا نمی‌توان یکی از اجزای فیل را کاملاً جدا از بقیه در نظر گرفت؟
  Answer: فیل یک کل به‌هم‌ پیوسته است و در واقعیت، پای فیل جدای گوش و خرطوم و عاجش نیست.

- seed=13
  Question: برای ارزیابی وضعیت مالی کسب‌وکار در پایان سال چه چیزی باید بررسی شود؟
  Answer: آیا بعد از همه هزینه‌ها سود دیگری هم باقی مانده یا کسب‌وکار علاوه بر این، بدهکار شده است؟

- seed=3
  Question: برای مقایسه لذت فوری با پس‌انداز، چه دو حالتی را می‌توان تجسم کرد؟
  Answer: یکی زمانی که لذت خوردن بستنی (یا هر وسوسۀ بیرونی دیگر) را تجربه می‌کنند و دیگر، وقتی پول‌های‌شان را جمع کرده‌اند

- seed=9
  Question: برای شناخت مقدمات فعالیت اقتصادی، آگاهی از چه منابعی لازم است؟
  Answer: منابع مالی، انسانی، طبیعی و مواد اولیه

- seed=16
  Question: در مدیریت خرج مبتنی بر قناعت، پول باید برای چه چیزی هزینه شود؟
  Answer: برای چیزی که واقعاً می‌خواهی و در چهارچوب ارزش‌ها 

In [18]:
# Code Cell 18
# NB12G — Assemble curated non-active experiment

assert "calibrated_semantic_audit_rows" in globals(), (
    "Run Code Cell 14 first."
)
assert "manual_adjudication_results" in globals(), (
    "Run Code Cell 17 first."
)
assert len(experiment) == 51

CURATED_EXPERIMENT_NAME = (
    "golden_dataset_quote_grounded_curated_experiment"
)
CURATION_VERSION = "nb12g_curated_experiment_v1"

original_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in experiment
}

calibrated_row_by_candidate_id = {
    row["candidate_id"]: row
    for row in calibrated_semantic_audit_rows
}

non_pass_candidate_ids = {
    row["candidate_id"]
    for row in calibrated_semantic_audit_rows
    if row["audit_status"] != "PASS"
}

adjudication_by_candidate_id = {
    row["candidate_id"]: row
    for row in manual_adjudication_results
}

assert len(original_record_by_candidate_id) == 51
assert len(non_pass_candidate_ids) == 10
assert set(adjudication_by_candidate_id) == (
    non_pass_candidate_ids
)

curated_experiment = []

# Preserve the 41 records that passed the calibrated audit.
for original_record in experiment:
    candidate_id = original_record["candidate_id"]

    if candidate_id in non_pass_candidate_ids:
        continue

    audit_row = calibrated_row_by_candidate_id[
        candidate_id
    ]

    assert audit_row["audit_status"] == "PASS"

    curated_record = dict(original_record)

    curated_record["dataset_version"] = (
        CURATED_EXPERIMENT_NAME
    )

    curated_record["curation"] = {
        "curation_version": CURATION_VERSION,
        "decision": "RETAIN",
        "source_candidate_id": candidate_id,
        "semantic_audit_status": "PASS",
        "active_dataset_record": False,
    }

    curated_experiment.append(curated_record)

# Add only the five evidence-grounded repaired records.
for adjudication in manual_adjudication_results:
    if adjudication["decision"] != "REPAIR":
        continue

    original_candidate_id = adjudication[
        "candidate_id"
    ]
    original_record = original_record_by_candidate_id[
        original_candidate_id
    ]
    replacement = adjudication["replacement"]

    replacement_candidate_id = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            (
                f"{CURATION_VERSION}:"
                f"{original_candidate_id}:"
                f"{canon(replacement['question'])}:"
                f"{canon(replacement['answer_span'])}"
            ),
        )
    )

    repaired_record = dict(original_record)

    repaired_record["candidate_id"] = (
        replacement_candidate_id
    )
    repaired_record["dataset_version"] = (
        CURATED_EXPERIMENT_NAME
    )
    repaired_record["question"] = (
        replacement["question"]
    )
    repaired_record["expected_answer"] = (
        replacement["answer_span"]
    )
    repaired_record["question_archetype"] = (
        replacement["intent"]
    )

    # Keep compatibility with the existing field.
    repaired_record["intent"] = replacement[
        "intent"
    ]

    repaired_record["curation"] = {
        "curation_version": CURATION_VERSION,
        "decision": "REPAIR",
        "source_candidate_id": (
            original_candidate_id
        ),
        "supersedes_candidate_id": (
            original_candidate_id
        ),
        "reason": adjudication["reason"],
        "semantic_audit_status": "PASS",
        "active_dataset_record": False,
    }

    curated_experiment.append(repaired_record)

curated_experiment.sort(
    key=lambda record: (
        record["source_seed_id"],
        record["candidate_id"],
    )
)

assert len(curated_experiment) == 46, (
    f"Expected 46 curated records; "
    f"found {len(curated_experiment)}."
)

assert len({
    record["candidate_id"]
    for record in curated_experiment
}) == len(curated_experiment)

assert len({
    canon(record["question"])
    for record in curated_experiment
}) == len(curated_experiment)

# Re-run the calibrated audit against every curated record.
curated_semantic_audit_rows = [
    calibrated_inspect_semantic_record(record)
    for record in curated_experiment
]

curated_non_pass_rows = [
    row
    for row in curated_semantic_audit_rows
    if row["audit_status"] != "PASS"
]

assert not curated_non_pass_rows, (
    "One or more curated records failed semantic audit: "
    + json.dumps(
        curated_non_pass_rows[:5],
        ensure_ascii=False,
    )
)

# Verify exact answer grounding.
for record in curated_experiment:
    assert (
        canon(record["expected_answer"])
        in canon(record["source_evidence"])
    ), (
        "Answer is not an exact evidence span: "
        f"{record['candidate_id']}"
    )

# Rejected records remain traceable in a separate queue.
curated_rejected_records = []

for adjudication in manual_adjudication_results:
    if adjudication["decision"] != "REJECT":
        continue

    original_record = original_record_by_candidate_id[
        adjudication["candidate_id"]
    ]

    curated_rejected_records.append({
        "candidate_id": adjudication[
            "candidate_id"
        ],
        "source_seed_id": adjudication[
            "source_seed_id"
        ],
        "question": original_record["question"],
        "expected_answer": original_record[
            "expected_answer"
        ],
        "source_evidence": original_record[
            "source_evidence"
        ],
        "decision": "REJECT",
        "reason": adjudication["reason"],
        "active_dataset_record": False,
    })

assert len(curated_rejected_records) == 5

curated_experiment_path = (
    golden_dir
    / f"{CURATED_EXPERIMENT_NAME}.jsonl"
)

curated_rejected_path = (
    golden_dir
    / f"{CURATED_EXPERIMENT_NAME}_rejected.jsonl"
)

curated_quality_report_path = (
    golden_dir
    / f"{CURATED_EXPERIMENT_NAME}_quality_report.json"
)

write_jsonl_atomic(
    curated_experiment_path,
    curated_experiment,
)

write_jsonl_atomic(
    curated_rejected_path,
    curated_rejected_records,
)

curated_seed_counts = Counter(
    record["source_seed_id"]
    for record in curated_experiment
)

curated_quality_report = {
    "curation_version": CURATION_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        "CURATED_EXPERIMENT_NOT_ACTIVE_GOLDEN"
    ),
    "active_dataset_created": False,
    "active_v1_modified": False,
    "source_experiment_path": str(
        experiment_path
    ),
    "source_experiment_sha256": sha256_file(
        experiment_path
    ),
    "curated_experiment_path": str(
        curated_experiment_path
    ),
    "curated_experiment_sha256": sha256_file(
        curated_experiment_path
    ),
    "input_record_count": len(experiment),
    "retained_record_count": 41,
    "repaired_record_count": 5,
    "rejected_record_count": 5,
    "final_record_count": len(
        curated_experiment
    ),
    "covered_seed_count": len(
        curated_seed_counts
    ),
    "covered_seed_ids": sorted(
        curated_seed_counts
    ),
    "records_by_seed": {
        str(seed_id): count
        for seed_id, count in sorted(
            curated_seed_counts.items()
        )
    },
    "semantic_audit": {
        "pass": len(
            curated_semantic_audit_rows
        ),
        "review": 0,
        "block": 0,
    },
    "unresolved_source_review_seeds": [
        7,
        22,
    ],
}

write_json_atomic(
    curated_quality_report_path,
    curated_quality_report,
)

print("=== CURATED NON-ACTIVE EXPERIMENT ===")
print("Input records:", len(experiment))
print("Retained records:", 41)
print("Repaired records:", 5)
print("Rejected records:", 5)
print(
    "Final curated records:",
    len(curated_experiment),
)
print(
    "Covered seeds:",
    len(curated_seed_counts),
)
print(
    "Semantic audit PASS:",
    len(curated_semantic_audit_rows),
)
print("Semantic audit REVIEW: 0")
print("Semantic audit BLOCK: 0")
print(
    "Unresolved source-review seeds:",
    [7, 22],
)

print(
    "\nCurated experiment:",
    curated_experiment_path,
)
print(
    "Rejected records:",
    curated_rejected_path,
)
print(
    "Quality report:",
    curated_quality_report_path,
)
print(
    "\nCURATED NON-ACTIVE EXPERIMENT: PASS"
)

=== CURATED NON-ACTIVE EXPERIMENT ===
Input records: 51
Retained records: 41
Repaired records: 5
Rejected records: 5
Final curated records: 46
Covered seeds: 25
Semantic audit PASS: 46
Semantic audit REVIEW: 0
Semantic audit BLOCK: 0
Unresolved source-review seeds: [7, 22]

Curated experiment: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_curated_experiment.jsonl
Rejected records: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_curated_experiment_rejected.jsonl
Quality report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_curated_experiment_quality_report.json

CURATED NON-ACTIVE EXPERIMENT: PASS


In [19]:
# Code Cell 19
# NB12G — Comparable lexical retrieval evaluation

assert "curated_experiment" in globals(), (
    "Run Code Cell 18 first."
)
assert "vectorizer" in globals(), (
    "Run Code Cell 11 first."
)
assert "passage_matrix" in globals(), (
    "Run Code Cell 11 first."
)
assert len(experiment) == 51
assert len(curated_experiment) == 46

RETRIEVAL_COMPARISON_VERSION = (
    "nb12g_curated_lexical_comparison_v1"
)

def evaluate_lexical_records(records):
    rows = []

    for record in records:
        labels = record["retrieval_labels"]

        if (
            labels["candidate"]
            != "parent_child_children"
        ):
            rows.append({
                "candidate_id": record[
                    "candidate_id"
                ],
                "source_seed_id": record[
                    "source_seed_id"
                ],
                "first_positive_rank": None,
                "excluded_reason": (
                    "no_parent_child_label"
                ),
            })
            continue

        positive_chunk_ids = labels.get(
            "positive_chunk_ids",
            [],
        )

        valid_positive_ids = [
            chunk_id
            for chunk_id in positive_chunk_ids
            if chunk_id in child_index_by_id
        ]

        assert valid_positive_ids, (
            "No valid positive child chunk for "
            f"{record['candidate_id']}"
        )

        query_vector = vectorizer.transform([
            canon(record["question"])
        ])

        scores = (
            passage_matrix @ query_vector.T
        ).toarray().ravel()

        ranking = np.argsort(
            -scores,
            kind="stable",
        )

        inverse_ranking = np.empty_like(ranking)
        inverse_ranking[ranking] = np.arange(
            len(ranking)
        )

        positive_ranks = [
            int(
                inverse_ranking[
                    child_index_by_id[chunk_id]
                ]
            ) + 1
            for chunk_id in valid_positive_ids
        ]

        first_positive_rank = min(
            positive_ranks
        )

        rows.append({
            "candidate_id": record[
                "candidate_id"
            ],
            "source_seed_id": record[
                "source_seed_id"
            ],
            "first_positive_rank": (
                first_positive_rank
            ),
            "hit_at_1": (
                first_positive_rank <= 1
            ),
            "hit_at_3": (
                first_positive_rank <= 3
            ),
            "hit_at_5": (
                first_positive_rank <= 5
            ),
            "hit_at_10": (
                first_positive_rank <= 10
            ),
            "reciprocal_rank": (
                1 / first_positive_rank
            ),
            "excluded_reason": None,
        })

    eligible_rows = [
        row
        for row in rows
        if row["first_positive_rank"] is not None
    ]

    assert eligible_rows, (
        "No eligible retrieval records."
    )

    metrics = {
        "total_records": len(records),
        "eligible_queries": len(
            eligible_rows
        ),
        "excluded_queries": (
            len(records) - len(eligible_rows)
        ),
        "recall_at_1": float(np.mean([
            row["hit_at_1"]
            for row in eligible_rows
        ])),
        "recall_at_3": float(np.mean([
            row["hit_at_3"]
            for row in eligible_rows
        ])),
        "recall_at_5": float(np.mean([
            row["hit_at_5"]
            for row in eligible_rows
        ])),
        "recall_at_10": float(np.mean([
            row["hit_at_10"]
            for row in eligible_rows
        ])),
        "mrr": float(np.mean([
            row["reciprocal_rank"]
            for row in eligible_rows
        ])),
        "mean_first_positive_rank": float(
            np.mean([
                row["first_positive_rank"]
                for row in eligible_rows
            ])
        ),
    }

    return {
        "metrics": metrics,
        "rows": rows,
    }

original_full_evaluation = (
    evaluate_lexical_records(experiment)
)

curated_full_evaluation = (
    evaluate_lexical_records(
        curated_experiment
    )
)

# Create a fair paired comparison:
# every curated record is compared with the exact original
# record from which it came.
paired_original_records = []
paired_curated_records = []

for curated_record in curated_experiment:
    source_candidate_id = curated_record[
        "curation"
    ]["source_candidate_id"]

    assert (
        source_candidate_id
        in original_record_by_candidate_id
    )

    paired_original_records.append(
        original_record_by_candidate_id[
            source_candidate_id
        ]
    )

    paired_curated_records.append(
        curated_record
    )

assert len(paired_original_records) == 46
assert len(paired_curated_records) == 46

paired_original_evaluation = (
    evaluate_lexical_records(
        paired_original_records
    )
)

paired_curated_evaluation = (
    evaluate_lexical_records(
        paired_curated_records
    )
)

# Isolate the five repaired records.
repaired_curated_records = [
    record
    for record in curated_experiment
    if record["curation"]["decision"]
    == "REPAIR"
]

repaired_original_records = [
    original_record_by_candidate_id[
        record["curation"][
            "source_candidate_id"
        ]
    ]
    for record in repaired_curated_records
]

assert len(repaired_curated_records) == 5
assert len(repaired_original_records) == 5

repaired_original_evaluation = (
    evaluate_lexical_records(
        repaired_original_records
    )
)

repaired_curated_evaluation = (
    evaluate_lexical_records(
        repaired_curated_records
    )
)

def metric_delta(after, before):
    comparable_metrics = (
        "recall_at_1",
        "recall_at_3",
        "recall_at_5",
        "recall_at_10",
        "mrr",
        "mean_first_positive_rank",
    )

    return {
        metric_name: (
            after[metric_name]
            - before[metric_name]
        )
        for metric_name in comparable_metrics
    }

paired_delta = metric_delta(
    paired_curated_evaluation["metrics"],
    paired_original_evaluation["metrics"],
)

repaired_delta = metric_delta(
    repaired_curated_evaluation["metrics"],
    repaired_original_evaluation["metrics"],
)

retrieval_comparison_report = {
    "comparison_version": (
        RETRIEVAL_COMPARISON_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "evaluation_contract": {
        "retrieval_method": (
            "persian_tfidf_word_unigram_bigram"
        ),
        "same_vectorizer": True,
        "same_passage_corpus": True,
        "same_positive_chunk_labels": True,
        "rank_used_as_acceptance_criterion": (
            False
        ),
        "final_nb13_benchmark": False,
    },
    "original_full": (
        original_full_evaluation
    ),
    "curated_full": (
        curated_full_evaluation
    ),
    "paired_original": (
        paired_original_evaluation
    ),
    "paired_curated": (
        paired_curated_evaluation
    ),
    "paired_delta": paired_delta,
    "repaired_original": (
        repaired_original_evaluation
    ),
    "repaired_curated": (
        repaired_curated_evaluation
    ),
    "repaired_delta": repaired_delta,
}

retrieval_comparison_report_path = (
    golden_dir
    / (
        f"{CURATED_EXPERIMENT_NAME}"
        "_lexical_comparison.json"
    )
)

write_json_atomic(
    retrieval_comparison_report_path,
    retrieval_comparison_report,
)

def print_metric_block(title, evaluation):
    metrics = evaluation["metrics"]

    print(f"\n{title}")
    print(
        "- Total records:",
        metrics["total_records"],
    )
    print(
        "- Eligible queries:",
        metrics["eligible_queries"],
    )
    print(
        "- Recall@1:",
        f"{metrics['recall_at_1']:.4f}",
    )
    print(
        "- Recall@3:",
        f"{metrics['recall_at_3']:.4f}",
    )
    print(
        "- Recall@5:",
        f"{metrics['recall_at_5']:.4f}",
    )
    print(
        "- Recall@10:",
        f"{metrics['recall_at_10']:.4f}",
    )
    print(
        "- MRR:",
        f"{metrics['mrr']:.4f}",
    )
    print(
        "- Mean first-positive rank:",
        f"{metrics['mean_first_positive_rank']:.2f}",
    )

def print_delta_block(title, delta):
    print(f"\n{title}")

    for metric_name, value in delta.items():
        print(
            f"- {metric_name}: {value:+.4f}"
        )

print(
    "=== CURATED LEXICAL RETRIEVAL COMPARISON ==="
)

print_metric_block(
    "Original full experiment",
    original_full_evaluation,
)

print_metric_block(
    "Curated full experiment",
    curated_full_evaluation,
)

print_metric_block(
    "Paired original records",
    paired_original_evaluation,
)

print_metric_block(
    "Paired curated records",
    paired_curated_evaluation,
)

print_delta_block(
    "Paired delta: curated minus original",
    paired_delta,
)

print_metric_block(
    "Five records before repair",
    repaired_original_evaluation,
)

print_metric_block(
    "Five records after repair",
    repaired_curated_evaluation,
)

print_delta_block(
    "Repair-only delta",
    repaired_delta,
)

print(
    "\nImportant:"
)
print(
    "- The paired comparison isolates wording changes."
)
print(
    "- The full comparison also reflects record rejection."
)
print(
    "- This is a lexical diagnostic, not the final NB13 benchmark."
)
print(
    "- Retrieval rank was not used to accept or reject records."
)

print(
    "\nReport:",
    retrieval_comparison_report_path,
)
print(
    "\nCURATED LEXICAL RETRIEVAL COMPARISON: PASS"
)

=== CURATED LEXICAL RETRIEVAL COMPARISON ===

Original full experiment
- Total records: 51
- Eligible queries: 48
- Recall@1: 0.6667
- Recall@3: 0.7917
- Recall@5: 0.8333
- Recall@10: 0.8958
- MRR: 0.7358
- Mean first-positive rank: 7.40

Curated full experiment
- Total records: 46
- Eligible queries: 43
- Recall@1: 0.5581
- Recall@3: 0.7674
- Recall@5: 0.8140
- Recall@10: 0.9070
- MRR: 0.6708
- Mean first-positive rank: 8.12

Paired original records
- Total records: 46
- Eligible queries: 43
- Recall@1: 0.6279
- Recall@3: 0.7674
- Recall@5: 0.8140
- Recall@10: 0.8837
- MRR: 0.7050
- Mean first-positive rank: 8.14

Paired curated records
- Total records: 46
- Eligible queries: 43
- Recall@1: 0.5581
- Recall@3: 0.7674
- Recall@5: 0.8140
- Recall@10: 0.9070
- MRR: 0.6708
- Mean first-positive rank: 8.12

Paired delta: curated minus original
- recall_at_1: -0.0698
- recall_at_3: +0.0000
- recall_at_5: +0.0000
- recall_at_10: +0.0233
- mrr: -0.0342
- mean_first_positive_rank: -0.0233

Five

In [20]:
# Code Cell 20
# NB12G — Retrieval leakage and repaired-rank analysis

assert "paired_original_evaluation" in globals(), (
    "Run Code Cell 19 first."
)
assert "paired_curated_evaluation" in globals(), (
    "Run Code Cell 19 first."
)
assert len(repaired_curated_records) == 5

LEAKAGE_AUDIT_VERSION = (
    "nb12g_retrieval_leakage_audit_v1"
)

def question_evidence_overlap(question, evidence):
    question_tokens = set(
        normalized_content_tokens(question)
    )
    evidence_tokens = set(
        normalized_content_tokens(evidence)
    )

    if not question_tokens:
        return 0.0

    return (
        len(question_tokens & evidence_tokens)
        / len(question_tokens)
    )

original_rank_by_candidate_id = {
    row["candidate_id"]: row[
        "first_positive_rank"
    ]
    for row in paired_original_evaluation["rows"]
}

curated_rank_by_candidate_id = {
    row["candidate_id"]: row[
        "first_positive_rank"
    ]
    for row in paired_curated_evaluation["rows"]
}

retrieval_leakage_rows = []

for curated_record in curated_experiment:
    source_candidate_id = curated_record[
        "curation"
    ]["source_candidate_id"]

    original_record = (
        original_record_by_candidate_id[
            source_candidate_id
        ]
    )

    original_rank = (
        original_rank_by_candidate_id[
            source_candidate_id
        ]
    )

    curated_rank = (
        curated_rank_by_candidate_id[
            curated_record["candidate_id"]
        ]
    )

    original_qa_overlap = question_answer_overlap(
        original_record["question"],
        original_record["expected_answer"],
    )

    curated_qa_overlap = question_answer_overlap(
        curated_record["question"],
        curated_record["expected_answer"],
    )

    original_qe_overlap = (
        question_evidence_overlap(
            original_record["question"],
            original_record["source_evidence"],
        )
    )

    curated_qe_overlap = (
        question_evidence_overlap(
            curated_record["question"],
            curated_record["source_evidence"],
        )
    )

    retrieval_leakage_rows.append({
        "source_candidate_id": source_candidate_id,
        "curated_candidate_id": curated_record[
            "candidate_id"
        ],
        "source_seed_id": curated_record[
            "source_seed_id"
        ],
        "curation_decision": curated_record[
            "curation"
        ]["decision"],
        "original_question": original_record[
            "question"
        ],
        "curated_question": curated_record[
            "question"
        ],
        "original_answer": original_record[
            "expected_answer"
        ],
        "curated_answer": curated_record[
            "expected_answer"
        ],
        "original_rank": original_rank,
        "curated_rank": curated_rank,
        "original_question_answer_overlap": (
            original_qa_overlap
        ),
        "curated_question_answer_overlap": (
            curated_qa_overlap
        ),
        "question_answer_overlap_delta": (
            curated_qa_overlap
            - original_qa_overlap
        ),
        "original_question_evidence_overlap": (
            original_qe_overlap
        ),
        "curated_question_evidence_overlap": (
            curated_qe_overlap
        ),
        "question_evidence_overlap_delta": (
            curated_qe_overlap
            - original_qe_overlap
        ),
        "rank_delta": (
            curated_rank - original_rank
            if (
                original_rank is not None
                and curated_rank is not None
            )
            else None
        ),
    })

assert len(retrieval_leakage_rows) == 46

repaired_leakage_rows = [
    row
    for row in retrieval_leakage_rows
    if row["curation_decision"] == "REPAIR"
]

retained_leakage_rows = [
    row
    for row in retrieval_leakage_rows
    if row["curation_decision"] == "RETAIN"
]

assert len(repaired_leakage_rows) == 5
assert len(retained_leakage_rows) == 41

comparable_repaired_rows = [
    row
    for row in repaired_leakage_rows
    if (
        row["original_rank"] is not None
        and row["curated_rank"] is not None
    )
]

mean_original_qa_overlap = float(np.mean([
    row["original_question_answer_overlap"]
    for row in comparable_repaired_rows
]))

mean_curated_qa_overlap = float(np.mean([
    row["curated_question_answer_overlap"]
    for row in comparable_repaired_rows
]))

mean_original_qe_overlap = float(np.mean([
    row["original_question_evidence_overlap"]
    for row in comparable_repaired_rows
]))

mean_curated_qe_overlap = float(np.mean([
    row["curated_question_evidence_overlap"]
    for row in comparable_repaired_rows
]))

top1_losses = [
    row
    for row in comparable_repaired_rows
    if (
        row["original_rank"] == 1
        and row["curated_rank"] > 1
    )
]

top10_gains = [
    row
    for row in comparable_repaired_rows
    if (
        row["original_rank"] > 10
        and row["curated_rank"] <= 10
    )
]

retrieval_leakage_report = {
    "audit_version": LEAKAGE_AUDIT_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "record_count": len(
        retrieval_leakage_rows
    ),
    "repaired_record_count": len(
        repaired_leakage_rows
    ),
    "comparable_repaired_record_count": len(
        comparable_repaired_rows
    ),
    "repaired_summary": {
        "mean_original_question_answer_overlap": (
            mean_original_qa_overlap
        ),
        "mean_curated_question_answer_overlap": (
            mean_curated_qa_overlap
        ),
        "question_answer_overlap_delta": (
            mean_curated_qa_overlap
            - mean_original_qa_overlap
        ),
        "mean_original_question_evidence_overlap": (
            mean_original_qe_overlap
        ),
        "mean_curated_question_evidence_overlap": (
            mean_curated_qe_overlap
        ),
        "question_evidence_overlap_delta": (
            mean_curated_qe_overlap
            - mean_original_qe_overlap
        ),
        "top1_losses": len(top1_losses),
        "top10_gains": len(top10_gains),
    },
    "rows": retrieval_leakage_rows,
}

retrieval_leakage_report_path = (
    golden_dir
    / (
        f"{CURATED_EXPERIMENT_NAME}"
        "_retrieval_leakage_audit.json"
    )
)

write_json_atomic(
    retrieval_leakage_report_path,
    retrieval_leakage_report,
)

print("=== RETRIEVAL LEAKAGE AUDIT ===")
print(
    "Curated records:",
    len(retrieval_leakage_rows),
)
print(
    "Repaired records:",
    len(repaired_leakage_rows),
)
print(
    "Comparable repaired records:",
    len(comparable_repaired_rows),
)

print("\nMean question-answer overlap:")
print(
    "- Before repair:",
    f"{mean_original_qa_overlap:.4f}",
)
print(
    "- After repair:",
    f"{mean_curated_qa_overlap:.4f}",
)
print(
    "- Delta:",
    (
        f"{mean_curated_qa_overlap - mean_original_qa_overlap:+.4f}"
    ),
)

print("\nMean question-evidence overlap:")
print(
    "- Before repair:",
    f"{mean_original_qe_overlap:.4f}",
)
print(
    "- After repair:",
    f"{mean_curated_qe_overlap:.4f}",
)
print(
    "- Delta:",
    (
        f"{mean_curated_qe_overlap - mean_original_qe_overlap:+.4f}"
    ),
)

print("\nRepaired-record details:")
for row in repaired_leakage_rows:
    print(
        f"\n- seed={row['source_seed_id']} | "
        f"rank {row['original_rank']} "
        f"-> {row['curated_rank']}"
    )
    print(
        "  QA overlap:",
        (
            f"{row['original_question_answer_overlap']:.4f}"
            " -> "
            f"{row['curated_question_answer_overlap']:.4f}"
        ),
    )
    print(
        "  Old:",
        row["original_question"],
    )
    print(
        "  New:",
        row["curated_question"],
    )

print("\nTop-1 losses:", len(top1_losses))
print("Top-10 gains:", len(top10_gains))
print(
    "\nReport:",
    retrieval_leakage_report_path,
)
print("\nRETRIEVAL LEAKAGE AUDIT: PASS")

=== RETRIEVAL LEAKAGE AUDIT ===
Curated records: 46
Repaired records: 5
Comparable repaired records: 4

Mean question-answer overlap:
- Before repair: 0.6271
- After repair: 0.0782
- Delta: -0.5489

Mean question-evidence overlap:
- Before repair: 0.6813
- After repair: 0.2801
- Delta: -0.4011

Repaired-record details:

- seed=1 | rank 1 -> 2
  QA overlap: 0.8750 -> 0.1111
  Old: چرا استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد؟
  New: چرا نمی‌توان یکی از اجزای فیل را کاملاً جدا از بقیه در نظر گرفت؟

- seed=3 | rank 14 -> 2
  QA overlap: 0.3333 -> 0.0588
  Old: آن‌ها باید چه چیزی را تجسم کنند؟
  New: برای مقایسه لذت فوری با پس‌انداز، چه دو حالتی را می‌توان تجسم کرد؟

- seed=9 | rank 1 -> 10
  QA overlap: 0.3000 -> 0.0000
  Old: آنها باید از چه منابعی آگاه باشند؟
  New: برای شناخت مقدمات فعالیت اقتصادی، آگاهی از چه منابعی لازم است؟

- seed=13 | rank 1 -> 2
  QA overlap: 1.0000 -> 0.1429
  Old: آیا بعد از همه هزینه‌ها سود دیگری هم باقی مانده یا کسب‌وکار علاوه بر این، بدهکار شد

In [21]:
# Code Cell 21
# NB12G — Curated seed-coverage gap audit

assert "curated_experiment" in globals(), (
    "Run Code Cell 18 first."
)
assert "seeds" in globals()
assert "source_chunks" in globals()
assert "source_review_queue" in globals()

COVERAGE_GAP_AUDIT_VERSION = (
    "nb12g_curated_seed_coverage_gap_v1"
)

expected_seed_ids = sorted(seeds)

curated_records_by_seed = defaultdict(list)

for record in curated_experiment:
    curated_records_by_seed[
        record["source_seed_id"]
    ].append(record)

curated_seed_counts = {
    seed_id: len(
        curated_records_by_seed.get(seed_id, [])
    )
    for seed_id in expected_seed_ids
}

missing_curated_seed_ids = [
    seed_id
    for seed_id, count in curated_seed_counts.items()
    if count == 0
]

single_record_seed_ids = [
    seed_id
    for seed_id, count in curated_seed_counts.items()
    if count == 1
]

multi_record_seed_ids = [
    seed_id
    for seed_id, count in curated_seed_counts.items()
    if count >= 2
]

source_review_seed_ids = sorted({
    row["source_seed_id"]
    for row in source_review_queue
})

rejected_records_by_seed = defaultdict(list)

for record in curated_rejected_records:
    rejected_records_by_seed[
        record["source_seed_id"]
    ].append(record)

coverage_gap_rows = []

for seed_id in expected_seed_ids:
    seed_record = seeds[seed_id]
    selected_source_record = source_chunks.get(
        seed_id,
        {},
    )

    selected_chunks = selected_source_record.get(
        "selected",
        [],
    )

    rejected_for_seed = (
        rejected_records_by_seed.get(
            seed_id,
            [],
        )
    )

    if curated_seed_counts[seed_id] == 0:
        gap_status = "MISSING"
    elif curated_seed_counts[seed_id] == 1:
        gap_status = "SINGLE_RECORD"
    else:
        gap_status = "MULTI_RECORD"

    if seed_id in source_review_seed_ids:
        gap_reason = "SOURCE_REVIEW_REQUIRED"
    elif rejected_for_seed:
        gap_reason = "EVIDENCE_FRAGMENT_REJECTED"
    elif not selected_chunks:
        gap_reason = "NO_SELECTED_SOURCE_CHUNK"
    elif curated_seed_counts[seed_id] == 1:
        gap_reason = "ONLY_ONE_VALID_RECORD"
    else:
        gap_reason = None

    coverage_gap_rows.append({
        "source_seed_id": seed_id,
        "gap_status": gap_status,
        "gap_reason": gap_reason,
        "curated_record_count": (
            curated_seed_counts[seed_id]
        ),
        "selected_source_chunk_count": len(
            selected_chunks
        ),
        "source_review_required": (
            seed_id in source_review_seed_ids
        ),
        "rejected_record_count": len(
            rejected_for_seed
        ),
        "approved_evidence": seed_record[
            "approved_evidence"
        ],
        "pilot_case_id": provenance[
            seed_id
        ]["pilot_case_id"],
        "proTrayvenance_source": provenance[
            seed_id
        ]["source"],
        "registered_document_ids": provenance[
            seed_id
        ]["document_ids"],
        "selected_chunk_ids": [
            chunk["chunk_id"]
            for chunk in selected_chunks
        ],
    })

# Correct the output key name without changing source data.
for row in coverage_gap_rows:
    row["provenance_source"] = row.pop(
        "proTrayvenance_source"
    )

coverage_gap_report = {
    "audit_version": (
        COVERAGE_GAP_AUDIT_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "expected_seed_count": len(
        expected_seed_ids
    ),
    "covered_seed_count": (
        len(expected_seed_ids)
        - len(missing_curated_seed_ids)
    ),
    "missing_seed_count": len(
        missing_curated_seed_ids
    ),
    "missing_seed_ids": (
        missing_curated_seed_ids
    ),
    "single_record_seed_count": len(
        single_record_seed_ids
    ),
    "single_record_seed_ids": (
        single_record_seed_ids
    ),
    "multi_record_seed_count": len(
        multi_record_seed_ids
    ),
    "source_review_seed_ids": (
        source_review_seed_ids
    ),
    "rows": coverage_gap_rows,
}

coverage_gap_report_path = (
    golden_dir
    / (
        f"{CURATED_EXPERIMENT_NAME}"
        "_seed_coverage_gap_audit.json"
    )
)

write_json_atomic(
    coverage_gap_report_path,
    coverage_gap_report,
)

print("=== CURATED SEED-COVERAGE GAP AUDIT ===")
print("Expected seeds:", len(expected_seed_ids))
print(
    "Covered seeds:",
    len(expected_seed_ids)
    - len(missing_curated_seed_ids),
)
print(
    "Missing seeds:",
    missing_curated_seed_ids,
)
print(
    "Single-record seeds:",
    single_record_seed_ids,
)
print(
    "Multi-record seeds:",
    len(multi_record_seed_ids),
)
print(
    "Source-review seeds:",
    source_review_seed_ids,
)

print("\nMissing-seed details:")

for seed_id in missing_curated_seed_ids:
    row = next(
        item
        for item in coverage_gap_rows
        if item["source_seed_id"] == seed_id
    )

    print("\n" + "=" * 80)
    print("Seed:", seed_id)
    print("Reason:", row["gap_reason"])
    print(
        "Selected source chunks:",
        row["selected_source_chunk_count"],
    )
    print(
        "Rejected records:",
        row["rejected_record_count"],
    )
    print(
        "Registered documents:",
        row["registered_document_ids"],
    )
    print("\nApproved evidence:")
    print(row["approved_evidence"])

    selected_chunks = source_chunks.get(
        seed_id,
        {},
    ).get("selected", [])

    if selected_chunks:
        print("\nSelected chunk previews:")

        for chunk in selected_chunks[:3]:
            preview = re.sub(
                r"\s+",
                " ",
                chunk["content"],
            ).strip()[:700]

            print(
                f"- {chunk['chunk_id']}"
            )
            print(" ", preview)

print(
    "\nReport:",
    coverage_gap_report_path,
)
print(
    "\nCURATED SEED-COVERAGE GAP AUDIT: PASS"
)

=== CURATED SEED-COVERAGE GAP AUDIT ===
Expected seeds: 28
Covered seeds: 25
Missing seeds: [7, 10, 22]
Single-record seeds: [1, 5, 11, 14, 15, 18, 19, 20, 24, 26, 27, 28]
Multi-record seeds: 13
Source-review seeds: [7, 22]

Missing-seed details:

Seed: 7
Reason: SOURCE_REVIEW_REQUIRED
Selected source chunks: 1
Rejected records: 0
Registered documents: ['book:adult-financial-literacy-course:fa-source:p000623', 'book:adult-financial-literacy-course:fa-source:p000624', 'book:adult-financial-literacy-course:fa-source:p000625', 'book:adult-financial-literacy-course:fa-source:p000626', 'book:adult-financial-literacy-course:fa-source:p000627', 'book:adult-financial-literacy-course:fa-source:p000628', 'book:adult-financial-literacy-course:fa-source:p000629', 'book:adult-financial-literacy-course:fa-source:p000630', 'book:adult-financial-literacy-course:fa-source:p000631', 'book:adult-financial-literacy-course:fa-source:p000632', 'book:adult-financial-literacy-course:fa-source:p000633', 'book:

In [22]:
# Code Cell 22
# NB12G — Deep direct-provenance source-window discovery

assert "missing_curated_seed_ids" in globals(), (
    "Run Code Cell 21 first."
)
assert missing_curated_seed_ids == [7, 10, 22]
assert "parents" in globals()
assert "parent_by_id" in globals()

DEEP_SOURCE_DISCOVERY_VERSION = (
    "nb12g_deep_source_window_discovery_v1"
)

TARGET_MISSING_SEEDS = [7, 10, 22]
MAX_DISCOVERY_WINDOWS_PER_SEED = 12
MAX_PRINTED_WINDOWS_PER_SEED = 5

def chunk_identifiers(chunk):
    identifiers = set(
        chunk.get("member_document_ids", [])
    )

    if chunk.get("source_id"):
        identifiers.add(chunk["source_id"])

    locator_start = chunk.get(
        "locator_start",
        {},
    )
    locator_end = chunk.get(
        "locator_end",
        {},
    )

    if isinstance(locator_start, dict):
        document_id = locator_start.get(
            "document_id"
        )
        if document_id:
            identifiers.add(document_id)

    if isinstance(locator_end, dict):
        document_id = locator_end.get(
            "document_id"
        )
        if document_id:
            identifiers.add(document_id)

    return identifiers

def deep_sentence_windows(
    text,
    maximum_sentences=8,
    maximum_tokens=220,
):
    source_sentences = sentences(text)
    windows = []

    for start_index in range(
        len(source_sentences)
    ):
        for width in range(
            1,
            maximum_sentences + 1,
        ):
            end_index = start_index + width

            if end_index > len(source_sentences):
                break

            window = " ".join(
                source_sentences[
                    start_index:end_index
                ]
            ).strip()

            token_count = len(words(window))

            if 6 <= token_count <= maximum_tokens:
                windows.append({
                    "sentence_start": start_index,
                    "sentence_end": end_index - 1,
                    "sentence_count": width,
                    "token_count": token_count,
                    "text": window,
                })

    return windows

all_chunk_records = []
seen_chunk_ids = set()

for chunk_type, records in (
    ("child", children),
    ("parent", parents),
    ("medium", medium),
):
    for chunk in records:
        chunk_id = chunk["chunk_id"]

        if chunk_id in seen_chunk_ids:
            continue

        seen_chunk_ids.add(chunk_id)

        all_chunk_records.append({
            "chunk_type": chunk_type,
            "chunk": chunk,
        })

deep_source_discovery = {}

for seed_id in TARGET_MISSING_SEEDS:
    registered_document_ids = set(
        provenance[seed_id]["document_ids"]
    )

    direct_chunk_records = []

    for item in all_chunk_records:
        identifiers = chunk_identifiers(
            item["chunk"]
        )

        overlap = sorted(
            registered_document_ids
            & identifiers
        )

        if overlap:
            direct_chunk_records.append({
                **item,
                "document_overlap": overlap,
                "linkage": (
                    "DIRECT_DOCUMENT_OVERLAP"
                ),
            })

    # Include parents reached through directly linked children.
    linked_parent_ids = {
        item["chunk"].get("parent_chunk_id")
        for item in direct_chunk_records
        if (
            item["chunk_type"] == "child"
            and item["chunk"].get(
                "parent_chunk_id"
            )
        )
    }

    already_included_ids = {
        item["chunk"]["chunk_id"]
        for item in direct_chunk_records
    }

    for parent_id in sorted(linked_parent_ids):
        if (
            parent_id in parent_by_id
            and parent_id not in already_included_ids
        ):
            direct_chunk_records.append({
                "chunk_type": "linked_parent",
                "chunk": parent_by_id[parent_id],
                "document_overlap": [],
                "linkage": (
                    "PARENT_OF_DIRECT_CHILD"
                ),
            })

    assert direct_chunk_records, (
        f"No direct source chunks found "
        f"for seed {seed_id}."
    )

    candidate_windows = []

    for item in direct_chunk_records:
        chunk = item["chunk"]

        for window in deep_sentence_windows(
            chunk["content"]
        ):
            score = coverage(
                seeds[seed_id][
                    "approved_evidence"
                ],
                window["text"],
            )

            shared_tokens = sorted(
                content_words(
                    seeds[seed_id][
                        "approved_evidence"
                    ]
                )
                & content_words(window["text"])
            )

            candidate_windows.append({
                "source_seed_id": seed_id,
                "chunk_id": chunk["chunk_id"],
                "chunk_type": item[
                    "chunk_type"
                ],
                "linkage": item["linkage"],
                "document_overlap": item[
                    "document_overlap"
                ],
                "parent_chunk_id": chunk.get(
                    "parent_chunk_id"
                ),
                "coverage": float(score),
                "shared_token_count": len(
                    shared_tokens
                ),
                "shared_tokens": shared_tokens,
                "sentence_start": window[
                    "sentence_start"
                ],
                "sentence_end": window[
                    "sentence_end"
                ],
                "sentence_count": window[
                    "sentence_count"
                ],
                "token_count": window[
                    "token_count"
                ],
                "text": window["text"],
            })

    candidate_windows.sort(
        key=lambda row: (
            -row["coverage"],
            -row["shared_token_count"],
            abs(row["token_count"] - 80),
            row["chunk_type"] != "child",
            row["chunk_id"],
            row["sentence_start"],
        )
    )

    unique_windows = []
    seen_window_texts = set()

    for candidate in candidate_windows:
        text_key = canon(candidate["text"])

        if text_key in seen_window_texts:
            continue

        seen_window_texts.add(text_key)
        unique_windows.append(candidate)

        if (
            len(unique_windows)
            == MAX_DISCOVERY_WINDOWS_PER_SEED
        ):
            break

    raw_review_evidence = []

    provenance_raw = provenance[
        seed_id
    ].get("raw", {})

    for evidence_item in provenance_raw.get(
        "source_evidence",
        [],
    ):
        if not isinstance(evidence_item, dict):
            continue

        evidence_text = str(
            evidence_item.get("text", "")
        ).strip()

        if not evidence_text:
            continue

        raw_review_evidence.append({
            "role": evidence_item.get("role"),
            "document_id": evidence_item.get(
                "document_id"
            ),
            "source_path": evidence_item.get(
                "source_path"
            ),
            "text": evidence_text,
            "coverage": coverage(
                seeds[seed_id][
                    "approved_evidence"
                ],
                evidence_text,
            ),
        })

    raw_review_evidence.sort(
        key=lambda row: -row["coverage"]
    )

    deep_source_discovery[seed_id] = {
        "source_seed_id": seed_id,
        "approved_evidence": seeds[
            seed_id
        ]["approved_evidence"],
        "registered_document_ids": sorted(
            registered_document_ids
        ),
        "direct_chunk_count": len(
            direct_chunk_records
        ),
        "candidate_window_count": len(
            candidate_windows
        ),
        "top_windows": unique_windows,
        "raw_review_evidence": (
            raw_review_evidence
        ),
    }

deep_source_discovery_report = {
    "discovery_version": (
        DEEP_SOURCE_DISCOVERY_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "target_seed_ids": (
        TARGET_MISSING_SEEDS
    ),
    "results": {
        str(seed_id): result
        for seed_id, result
        in deep_source_discovery.items()
    },
}

deep_source_discovery_report_path = (
    golden_dir
    / (
        f"{CURATED_EXPERIMENT_NAME}"
        "_deep_source_window_discovery.json"
    )
)

write_json_atomic(
    deep_source_discovery_report_path,
    deep_source_discovery_report,
)

print(
    "=== DEEP DIRECT-PROVENANCE "
    "SOURCE-WINDOW DISCOVERY ==="
)

for seed_id in TARGET_MISSING_SEEDS:
    result = deep_source_discovery[seed_id]

    print("\n" + "=" * 80)
    print("SEED:", seed_id)
    print(
        "Direct/linked chunks:",
        result["direct_chunk_count"],
    )
    print(
        "Candidate windows:",
        result["candidate_window_count"],
    )

    print("\nApproved evidence:")
    print(result["approved_evidence"])

    print("\nTop source windows:")

    for position, window in enumerate(
        result["top_windows"][
            :MAX_PRINTED_WINDOWS_PER_SEED
        ],
        1,
    ):
        print(
            f"\n{position}. "
            f"coverage={window['coverage']:.4f} | "
            f"type={window['chunk_type']} | "
            f"tokens={window['token_count']}"
        )
        print(
            "chunk_id:",
            window["chunk_id"],
        )
        print(
            "linkage:",
            window["linkage"],
        )
        print(
            "document_overlap:",
            window["document_overlap"],
        )
        print("text:")
        print(window["text"])

    if result["raw_review_evidence"]:
        print(
            "\nHighest-coverage human-review "
            "evidence units:"
        )

        for evidence_item in (
            result["raw_review_evidence"][:3]
        ):
            print(
                f"\n- coverage="
                f"{evidence_item['coverage']:.4f}"
            )
            print(
                "  document_id:",
                evidence_item["document_id"],
            )
            print(
                "  role:",
                evidence_item["role"],
            )
            print("  text:")
            print(evidence_item["text"])

print(
    "\nReport:",
    deep_source_discovery_report_path,
)
print(
    "\nDEEP DIRECT-PROVENANCE "
    "SOURCE-WINDOW DISCOVERY: PASS"
)

=== DEEP DIRECT-PROVENANCE SOURCE-WINDOW DISCOVERY ===

SEED: 7
Direct/linked chunks: 4
Candidate windows: 848

Approved evidence:
این پرسشنامه چند جنبه اصلی را بررسی می‌کند: ماهیت و جذابیت خودِ کار، رفتار و عملکرد سرپرست، حقوق و مزایا، ویژگی‌ها و رفتار همکاران، فرصت‌های رشد و ترفیع، و شرایط فیزیکی و امکانات محیط کار مانند فضای کاری، نور، تهویه، تجهیزات، ساختمان و کیفیت غذا.

Top source windows:

1. coverage=0.2059 | type=child | tokens=77
chunk_id: chunk:parent_child:child:98ec9f0e52da36ef060a8c8e
linkage: DIRECT_DOCUMENT_OVERLAP
document_overlap: ['book:adult-financial-literacy-course:fa-source:p000653', 'book:adult-financial-literacy-course:fa-source:p000654', 'book:adult-financial-literacy-course:fa-source:p000655', 'book:adult-financial-literacy-course:fa-source:p000656', 'book:adult-financial-literacy-course:fa-source:p000657', 'book:adult-financial-literacy-course:fa-source:p000658', 'book:adult-financial-literacy-course:fa-source:p000659', 'book:adult-financial-literacy-course:

In [23]:
# Code Cell 23
# NB12G — Evidence-scope recovery for seeds 7 and 22

assert "curated_experiment" in globals(), (
    "Run Code Cell 18 first."
)
assert "deep_source_discovery" in globals(), (
    "Run Code Cell 22 first."
)

SOURCE_RECOVERY_VERSION = (
    "nb12g_evidence_scope_recovery_v1"
)

COVERAGE_REPAIRED_EXPERIMENT_NAME = (
    "golden_dataset_quote_grounded_coverage_repaired_experiment"
)

child_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in children
}

source_recovery_spec = {
    7: {
        "decision": "RECOVER",
        "scope_decision": (
            "NARROW_TO_DIRECT_PHYSICAL_WORKPLACE_EVIDENCE"
        ),
        "chunk_id": (
            "chunk:parent_child:child:"
            "98ec9f0e52da36ef060a8c8e"
        ),
        "question_intent": "DIRECT_FACT",
        "question": (
            "در ارزیابی شرایط فیزیکی محیط کار، "
            "درباره تجهیزات اداری چه چیزی "
            "سنجیده می‌شود؟"
        ),
        "expected_answer": (
            "تجهیزات اداری (میز، صندلی و "
            "لوازم و ...) در حد مناسبی است."
        ),
        "source_evidence": (
            "در شغل من امکان پیشرفت، رشد و ترفیع "
            "نسبتاً مطلوب است. فضای کار و اتاق "
            "کاری من مناسب است. محیط کاری من، از "
            "نظر سر و صدا مناسب است. تجهیزات "
            "اداری (میز، صندلی و لوازم و ...) "
            "در حد مناسبی است. محیط کاری من، از "
            "نور مناسبی برخوردار است. محیط کاری "
            "من تهویه مناسبی (سرمایش و گرمایش) "
            "دارد. ساختمان مورد استفاده (آسانسور، "
            "راهروها، فضای رستوران و ...) از کیفیت "
            "مناسبی برخوردار است. کیفیت غذا در حد "
            "مناسبی قرار دارد."
        ),
        "reason": (
            "The registered source directly supports "
            "physical workplace-condition items, but "
            "not the entire legacy summary."
        ),
    },

    22: {
        "decision": "RECOVER",
        "scope_decision": (
            "NARROW_TO_DIRECT_SELF_ASSESSMENT_INSTRUCTION"
        ),
        "chunk_id": (
            "chunk:parent_child:child:"
            "4666885172174434fcefeb8b"
        ),
        "question_intent": "DIRECT_FACT",
        "question": (
            "در خودارزیابی موقعیت فرد در پلکان "
            "توانمندی مالی، پس از ارزیابی جایگاه "
            "فعلی چه کاری خواسته شده است؟"
        ),
        "expected_answer": (
            "چند جمله‌ای درباره وضعیت خود بنویسید."
        ),
        "source_evidence": (
            "خودارزیابی 1: موقعیت من در پلکان "
            "توانمندی مالی حتماً احساس می‌کنید "
            "همه این پله‌ها را در زمان‌های متفاوتی "
            "تجربه کرده‌اید، الان جایگاه فعلی خود "
            "را چگونه ارزیابی می‌کنید. چند جمله‌ای "
            "درباره وضعیت خود بنویسید. (5 پله به "
            "صورت پلکانی از شماره 1 پایین پلکان و "
            "5 بالای پلکان طراحی می‌شود)"
        ),
        "reason": (
            "The registered source supports the "
            "self-assessment instruction, but does "
            "not contain the five stage definitions."
        ),
    },
}

source_recovery_records = []

for seed_id, specification in (
    source_recovery_spec.items()
):
    assert specification["decision"] == "RECOVER"

    chunk_id = specification["chunk_id"]

    assert chunk_id in child_by_id, (
        f"Recovery chunk missing for seed {seed_id}: "
        f"{chunk_id}"
    )

    source_chunk = child_by_id[chunk_id]

    assert (
        canon(specification["source_evidence"])
        in canon(source_chunk["content"])
    ), (
        f"Source evidence is not an exact chunk span "
        f"for seed {seed_id}."
    )

    assert (
        canon(specification["expected_answer"])
        in canon(specification["source_evidence"])
    ), (
        f"Answer is not an exact evidence span "
        f"for seed {seed_id}."
    )

    temporary_record = {
        "candidate_id": f"temporary-seed-{seed_id}",
        "source_seed_id": seed_id,
        "question": specification["question"],
        "expected_answer": specification[
            "expected_answer"
        ],
        "source_evidence": specification[
            "source_evidence"
        ],
    }

    semantic_validation = (
        calibrated_inspect_semantic_record(
            temporary_record
        )
    )

    assert (
        semantic_validation["audit_status"]
        == "PASS"
    ), (
        f"Recovered seed {seed_id} failed semantic "
        f"validation: {semantic_validation}"
    )

    candidate_id = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            (
                f"{SOURCE_RECOVERY_VERSION}:"
                f"{seed_id}:"
                f"{chunk_id}:"
                f"{canon(specification['question'])}:"
                f"{canon(specification['expected_answer'])}"
            ),
        )
    )

    source_recovery_records.append({
        "candidate_id": candidate_id,
        "dataset_version": (
            COVERAGE_REPAIRED_EXPERIMENT_NAME
        ),
        "source_seed_id": seed_id,
        "source_lineage": seeds[
            seed_id
        ]["lineage"],
        "question": specification["question"],
        "expected_answer": specification[
            "expected_answer"
        ],
        "question_intent": specification[
            "question_intent"
        ],
        "evaluation_split": split_by_seed[
            seed_id
        ],
        "source_evidence": specification[
            "source_evidence"
        ],
        "evidence_quote": specification[
            "source_evidence"
        ],
        "legacy_seed_evidence": seeds[
            seed_id
        ]["approved_evidence"],
        "taxonomy_topics": seeds[
            seed_id
        ]["taxonomy_topics"],
        "competency_ids": seeds[
            seed_id
        ]["competency_ids"],
        "pilot_case_id": provenance[
            seed_id
        ]["pilot_case_id"],
        "provenance_source": provenance[
            seed_id
        ]["source"],
        "document_ids": provenance[
            seed_id
        ]["document_ids"],
        "retrieval_labels": {
            "candidate": (
                "parent_child_children"
            ),
            "positive_chunk_ids": [
                chunk_id
            ],
            "positive_parent_chunk_ids": (
                [source_chunk["parent_chunk_id"]]
                if source_chunk.get(
                    "parent_chunk_id"
                )
                else []
            ),
            "label_basis": (
                "DIRECT_PROVENANCE_SCOPE_RECOVERY"
            ),
        },
        "generation": {
            "model": None,
            "prompt_version": None,
            "method": (
                "EVIDENCE_BASED_CURATED_RECOVERY"
            ),
        },
        "generation_status": (
            "EXPERIMENT_EVIDENCE_CURATED"
        ),
        "curation": {
            "curation_version": (
                SOURCE_RECOVERY_VERSION
            ),
            "decision": "RECOVER",
            "scope_decision": specification[
                "scope_decision"
            ],
            "reason": specification["reason"],
            "active_dataset_record": False,
        },
    })

assert len(source_recovery_records) == 2

coverage_repaired_experiment = [
    dict(record)
    for record in curated_experiment
]

for record in coverage_repaired_experiment:
    record["dataset_version"] = (
        COVERAGE_REPAIRED_EXPERIMENT_NAME
    )

coverage_repaired_experiment.extend(
    source_recovery_records
)

coverage_repaired_experiment.sort(
    key=lambda record: (
        record["source_seed_id"],
        record["candidate_id"],
    )
)

assert len(coverage_repaired_experiment) == 48

assert len({
    record["candidate_id"]
    for record in coverage_repaired_experiment
}) == 48

assert len({
    canon(record["question"])
    for record in coverage_repaired_experiment
}) == 48

final_covered_seed_ids = sorted({
    record["source_seed_id"]
    for record in coverage_repaired_experiment
})

final_missing_seed_ids = sorted(
    set(expected_seed_ids)
    - set(final_covered_seed_ids)
)

assert final_missing_seed_ids == [10]

coverage_repaired_audit_rows = [
    calibrated_inspect_semantic_record(record)
    for record in coverage_repaired_experiment
]

coverage_repaired_non_pass_rows = [
    row
    for row in coverage_repaired_audit_rows
    if row["audit_status"] != "PASS"
]

assert not coverage_repaired_non_pass_rows, (
    "Recovered experiment contains non-PASS rows: "
    + json.dumps(
        coverage_repaired_non_pass_rows[:5],
        ensure_ascii=False,
    )
)

seed_10_review_record = {
    "source_seed_id": 10,
    "pilot_case_id": provenance[
        10
    ]["pilot_case_id"],
    "decision": (
        "DEFER_FOR_SOURCE_INTERPRETATION_REVIEW"
    ),
    "reason": (
        "Registered transcript evidence is primarily "
        "rhetorical and interrogative. The approved "
        "summary requires an interpretation that is "
        "not available as an exact answer span."
    ),
    "registered_document_ids": provenance[
        10
    ]["document_ids"],
    "approved_evidence": seeds[
        10
    ]["approved_evidence"],
    "active_dataset_record": False,
}

coverage_repaired_experiment_path = (
    golden_dir
    / (
        f"{COVERAGE_REPAIRED_EXPERIMENT_NAME}"
        ".jsonl"
    )
)

coverage_repaired_review_path = (
    golden_dir
    / (
        f"{COVERAGE_REPAIRED_EXPERIMENT_NAME}"
        "_review_queue.jsonl"
    )
)

coverage_repaired_report_path = (
    golden_dir
    / (
        f"{COVERAGE_REPAIRED_EXPERIMENT_NAME}"
        "_quality_report.json"
    )
)

write_jsonl_atomic(
    coverage_repaired_experiment_path,
    coverage_repaired_experiment,
)

write_jsonl_atomic(
    coverage_repaired_review_path,
    [seed_10_review_record],
)

coverage_repaired_report = {
    "recovery_version": (
        SOURCE_RECOVERY_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        "COVERAGE_REPAIRED_EXPERIMENT_NOT_ACTIVE"
    ),
    "active_dataset_created": False,
    "active_v1_modified": False,
    "source_curated_experiment": str(
        curated_experiment_path
    ),
    "experiment_path": str(
        coverage_repaired_experiment_path
    ),
    "experiment_sha256": sha256_file(
        coverage_repaired_experiment_path
    ),
    "record_count": len(
        coverage_repaired_experiment
    ),
    "covered_seed_count": len(
        final_covered_seed_ids
    ),
    "covered_seed_ids": final_covered_seed_ids,
    "missing_seed_ids": final_missing_seed_ids,
    "recovered_seed_ids": [7, 22],
    "deferred_seed_ids": [10],
    "semantic_audit": {
        "pass": len(
            coverage_repaired_audit_rows
        ),
        "review": 0,
        "block": 0,
    },
}

write_json_atomic(
    coverage_repaired_report_path,
    coverage_repaired_report,
)

print("=== EVIDENCE-SCOPE RECOVERY ===")
print(
    "Input curated records:",
    len(curated_experiment),
)
print(
    "Recovered records:",
    len(source_recovery_records),
)
print(
    "Final experiment records:",
    len(coverage_repaired_experiment),
)
print(
    "Covered seeds:",
    len(final_covered_seed_ids),
)
print(
    "Recovered seeds:",
    [7, 22],
)
print(
    "Deferred seeds:",
    final_missing_seed_ids,
)
print(
    "Semantic audit PASS:",
    len(coverage_repaired_audit_rows),
)
print("Semantic audit REVIEW: 0")
print("Semantic audit BLOCK: 0")

print(
    "\nExperiment:",
    coverage_repaired_experiment_path,
)
print(
    "Review queue:",
    coverage_repaired_review_path,
)
print(
    "Quality report:",
    coverage_repaired_report_path,
)
print(
    "\nEVIDENCE-SCOPE RECOVERY: PASS"
)

=== EVIDENCE-SCOPE RECOVERY ===
Input curated records: 46
Recovered records: 2
Final experiment records: 48
Covered seeds: 27
Recovered seeds: [7, 22]
Deferred seeds: [10]
Semantic audit PASS: 48
Semantic audit REVIEW: 0
Semantic audit BLOCK: 0

Experiment: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_coverage_repaired_experiment.jsonl
Review queue: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_coverage_repaired_experiment_review_queue.jsonl
Quality report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_coverage_repaired_experiment_quality_report.json

EVIDENCE-SCOPE RECOVERY: PASS


In [24]:
# Code Cell 24
# NB12G — Seed 10 neighboring transcript-context discovery

assert "coverage_repaired_experiment" in globals(), (
    "Run Code Cell 23 first."
)
assert final_missing_seed_ids == [10]

SEED_10_CONTEXT_DISCOVERY_VERSION = (
    "nb12g_seed10_neighbor_context_v1"
)

TARGET_SEED_ID = 10

registered_seed_10_document_ids = provenance[
    TARGET_SEED_ID
]["document_ids"]

assert registered_seed_10_document_ids

seed_10_source_prefixes = sorted({
    document_id.rsplit(
        ":utterance:",
        1,
    )[0]
    for document_id in registered_seed_10_document_ids
    if ":utterance:" in document_id
})

assert len(seed_10_source_prefixes) == 1

seed_10_source_prefix = (
    seed_10_source_prefixes[0]
)

def utterance_index_from_id(document_id):
    match = re.search(
        r":utterance:(\d+)$",
        str(document_id),
    )

    return (
        int(match.group(1))
        if match
        else None
    )

registered_utterance_indexes = [
    utterance_index_from_id(document_id)
    for document_id
    in registered_seed_10_document_ids
]

registered_utterance_indexes = [
    index
    for index in registered_utterance_indexes
    if index is not None
]

assert registered_utterance_indexes

registered_start_index = min(
    registered_utterance_indexes
)
registered_end_index = max(
    registered_utterance_indexes
)

seed_10_neighbor_candidates = []

for chunk_type, records in (
    ("child", children),
    ("parent", parents),
    ("medium", medium),
):
    for chunk in records:
        identifiers = chunk_identifiers(chunk)

        source_related_ids = sorted({
            document_id
            for document_id in identifiers
            if (
                document_id
                == seed_10_source_prefix
                or document_id.startswith(
                    seed_10_source_prefix
                    + ":utterance:"
                )
            )
        })

        source_id_matches = (
            chunk.get("source_id")
            == seed_10_source_prefix
        )

        if (
            not source_related_ids
            and not source_id_matches
        ):
            continue

        utterance_indexes = [
            utterance_index_from_id(
                document_id
            )
            for document_id in source_related_ids
        ]

        utterance_indexes = [
            index
            for index in utterance_indexes
            if index is not None
        ]

        if utterance_indexes:
            chunk_start_index = min(
                utterance_indexes
            )
            chunk_end_index = max(
                utterance_indexes
            )
        else:
            chunk_start_index = None
            chunk_end_index = None

        if (
            chunk_start_index is not None
            and chunk_end_index is not None
        ):
            if chunk_end_index < registered_start_index:
                distance = (
                    registered_start_index
                    - chunk_end_index
                )
                relative_position = "BEFORE"

            elif chunk_start_index > registered_end_index:
                distance = (
                    chunk_start_index
                    - registered_end_index
                )
                relative_position = "AFTER"

            else:
                distance = 0
                relative_position = "OVERLAP"

        else:
            distance = 999999
            relative_position = "SOURCE_LEVEL"

        seed_10_neighbor_candidates.append({
            "chunk_id": chunk["chunk_id"],
            "chunk_type": chunk_type,
            "source_id": chunk.get("source_id"),
            "parent_chunk_id": chunk.get(
                "parent_chunk_id"
            ),
            "utterance_start": chunk_start_index,
            "utterance_end": chunk_end_index,
            "relative_position": (
                relative_position
            ),
            "distance_from_registered_range": (
                distance
            ),
            "approved_evidence_coverage": coverage(
                seeds[TARGET_SEED_ID][
                    "approved_evidence"
                ],
                chunk["content"],
            ),
            "token_count": len(
                words(chunk["content"])
            ),
            "content": chunk["content"],
        })

assert seed_10_neighbor_candidates, (
    "No neighboring transcript chunks found."
)

seed_10_neighbor_candidates.sort(
    key=lambda row: (
        row[
            "distance_from_registered_range"
        ],
        row["chunk_type"] != "child",
        (
            row["utterance_start"]
            if row["utterance_start"] is not None
            else 999999
        ),
        row["chunk_id"],
    )
)

unique_seed_10_neighbors = []
seen_neighbor_contents = set()

for candidate in seed_10_neighbor_candidates:
    content_key = canon(candidate["content"])

    if content_key in seen_neighbor_contents:
        continue

    seen_neighbor_contents.add(content_key)
    unique_seed_10_neighbors.append(candidate)

    if len(unique_seed_10_neighbors) == 15:
        break

seed_10_context_report = {
    "discovery_version": (
        SEED_10_CONTEXT_DISCOVERY_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "source_seed_id": TARGET_SEED_ID,
    "source_prefix": seed_10_source_prefix,
    "registered_utterance_start": (
        registered_start_index
    ),
    "registered_utterance_end": (
        registered_end_index
    ),
    "registered_document_ids": (
        registered_seed_10_document_ids
    ),
    "approved_evidence": seeds[
        TARGET_SEED_ID
    ]["approved_evidence"],
    "candidate_count": len(
        seed_10_neighbor_candidates
    ),
    "displayed_unique_candidate_count": len(
        unique_seed_10_neighbors
    ),
    "candidates": unique_seed_10_neighbors,
}

seed_10_context_report_path = (
    golden_dir
    / (
        f"{COVERAGE_REPAIRED_EXPERIMENT_NAME}"
        "_seed10_neighbor_context.json"
    )
)

write_json_atomic(
    seed_10_context_report_path,
    seed_10_context_report,
)

print(
    "=== SEED 10 NEIGHBORING "
    "TRANSCRIPT CONTEXT ==="
)
print(
    "Source:",
    seed_10_source_prefix,
)
print(
    "Registered utterance range:",
    (
        registered_start_index,
        registered_end_index,
    ),
)
print(
    "Related chunks found:",
    len(seed_10_neighbor_candidates),
)
print(
    "Unique chunks displayed:",
    len(unique_seed_10_neighbors),
)

print("\nApproved evidence:")
print(
    seeds[TARGET_SEED_ID][
        "approved_evidence"
    ]
)

for position, candidate in enumerate(
    unique_seed_10_neighbors,
    1,
):
    print("\n" + "=" * 80)
    print(
        f"{position}. "
        f"type={candidate['chunk_type']} | "
        f"position={candidate['relative_position']} | "
        f"distance="
        f"{candidate['distance_from_registered_range']} | "
        f"utterances="
        f"{candidate['utterance_start']}"
        f"..{candidate['utterance_end']} | "
        f"coverage="
        f"{candidate['approved_evidence_coverage']:.4f}"
    )
    print(
        "chunk_id:",
        candidate["chunk_id"],
    )
    print("content:")
    print(candidate["content"])

print(
    "\nReport:",
    seed_10_context_report_path,
)
print(
    "\nSEED 10 NEIGHBORING "
    "TRANSCRIPT CONTEXT: PASS"
)

=== SEED 10 NEIGHBORING TRANSCRIPT CONTEXT ===
Source: program:dar-jostojo-ye-khoshbakhti:episode-028
Registered utterance range: (66, 79)
Related chunks found: 14
Unique chunks displayed: 12

Approved evidence:
چون آموزش سواد مالی کمک می‌کند افراد تصمیم‌های مالی بهتری بگیرند، از خود و خانواده‌شان بهتر مراقبت کنند، ارزش پولشان را بهتر حفظ کنند و در شرایط اقتصادی نامطلوب کمتر آسیب ببینند.

1. type=child | position=OVERLAP | distance=0 | utterances=43..87 | coverage=0.5200
chunk_id: chunk:parent_child:child:3b43e9da2fd8c54d69470ab8
content:
یا گونی های ماسته بچینید و موقعی موشک بارون مثلا در زیر زمین قرار بگیرید که اطرافش گونی های ماسته چیده شده
یعنی یه آموزش های کلان و کاربردی به ما می دادن که تو اون جنگ کمتر زخمی بشیم
کمتر آسیب ببینیم
خب تو جنگ اقتصادی چه آموزش هایی به ما دادن؟
چی به ما گفتن که ما بهتر عمل بکنیم
کمتر تو این شرایط اقتصادی نامطلوب صدمه ببینیم
آسیب ببینیم
خب نرخ جمعیت ما که روبه کاهشه
پس چطور می‌شه نرخ زنان سرپرست خانوار افزایش پیدا بکنه
چطور می‌شه نرخ کارتن‌خواب ها افزای

In [25]:
# Code Cell 25
# NB12G — Direct-evidence recovery for seed 10

assert "coverage_repaired_experiment" in globals(), (
    "Run Code Cell 23 first."
)
assert "unique_seed_10_neighbors" in globals(), (
    "Run Code Cell 24 first."
)
assert final_missing_seed_ids == [10]

FINAL_EXPERIMENT_NAME = (
    "golden_dataset_quote_grounded_complete_experiment"
)

SEED_10_RECOVERY_VERSION = (
    "nb12g_seed10_direct_evidence_recovery_v1"
)

seed_10_chunk_id = (
    "chunk:parent_child:child:"
    "3b43e9da2fd8c54d69470ab8"
)

assert seed_10_chunk_id in child_by_id

seed_10_chunk = child_by_id[
    seed_10_chunk_id
]

seed_10_source_evidence = (
    "خب آموزش سواد مالی وقتی از بالا بهش "
    "نگاه می‌کنیم یکی از ضرورتهای مهمه"
)

seed_10_question = (
    "از دیدگاه کلان، آموزش سواد مالی "
    "چه جایگاهی دارد؟"
)

seed_10_expected_answer = (
    "یکی از ضرورتهای مهمه"
)

assert (
    canon(seed_10_source_evidence)
    in canon(seed_10_chunk["content"])
), "Seed 10 evidence is not present in the frozen chunk."

assert (
    canon(seed_10_expected_answer)
    in canon(seed_10_source_evidence)
), "Seed 10 answer is not an exact evidence span."

seed_10_temporary_record = {
    "candidate_id": "temporary-seed-10",
    "source_seed_id": 10,
    "question": seed_10_question,
    "expected_answer": (
        seed_10_expected_answer
    ),
    "source_evidence": (
        seed_10_source_evidence
    ),
}

seed_10_semantic_validation = (
    calibrated_inspect_semantic_record(
        seed_10_temporary_record
    )
)

assert (
    seed_10_semantic_validation[
        "audit_status"
    ]
    == "PASS"
), (
    "Seed 10 recovery failed semantic validation: "
    + json.dumps(
        seed_10_semantic_validation,
        ensure_ascii=False,
    )
)

seed_10_candidate_id = str(
    uuid.uuid5(
        uuid.NAMESPACE_URL,
        (
            f"{SEED_10_RECOVERY_VERSION}:"
            f"{seed_10_chunk_id}:"
            f"{canon(seed_10_question)}:"
            f"{canon(seed_10_expected_answer)}"
        ),
    )
)

seed_10_recovery_record = {
    "candidate_id": seed_10_candidate_id,
    "dataset_version": FINAL_EXPERIMENT_NAME,
    "source_seed_id": 10,
    "source_lineage": seeds[10]["lineage"],
    "question": seed_10_question,
    "expected_answer": (
        seed_10_expected_answer
    ),
    "question_intent": "DIRECT_FACT",
    "evaluation_split": split_by_seed[10],
    "source_evidence": (
        seed_10_source_evidence
    ),
    "evidence_quote": (
        seed_10_source_evidence
    ),
    "legacy_seed_evidence": seeds[
        10
    ]["approved_evidence"],
    "taxonomy_topics": seeds[
        10
    ]["taxonomy_topics"],
    "competency_ids": seeds[
        10
    ]["competency_ids"],
    "pilot_case_id": provenance[
        10
    ]["pilot_case_id"],
    "provenance_source": provenance[
        10
    ]["source"],
    "document_ids": provenance[
        10
    ]["document_ids"],
    "retrieval_labels": {
        "candidate": (
            "parent_child_children"
        ),
        "positive_chunk_ids": [
            seed_10_chunk_id
        ],
        "positive_parent_chunk_ids": (
            [seed_10_chunk["parent_chunk_id"]]
            if seed_10_chunk.get(
                "parent_chunk_id"
            )
            else []
        ),
        "label_basis": (
            "DIRECT_PROVENANCE_SCOPE_RECOVERY"
        ),
    },
    "generation": {
        "model": None,
        "prompt_version": None,
        "method": (
            "EVIDENCE_BASED_CURATED_RECOVERY"
        ),
    },
    "generation_status": (
        "EXPERIMENT_EVIDENCE_CURATED"
    ),
    "curation": {
        "curation_version": (
            SEED_10_RECOVERY_VERSION
        ),
        "decision": "RECOVER",
        "scope_decision": (
            "NARROW_TO_EXPLICIT_DIRECT_CLAIM"
        ),
        "reason": (
            "The rhetorical transcript fragments "
            "were rejected. A separate explicit "
            "statement in the same directly linked "
            "chunk supports this narrower question."
        ),
        "active_dataset_record": False,
    },
}

complete_experiment = [
    dict(record)
    for record in coverage_repaired_experiment
]

for record in complete_experiment:
    record["dataset_version"] = (
        FINAL_EXPERIMENT_NAME
    )

complete_experiment.append(
    seed_10_recovery_record
)

complete_experiment.sort(
    key=lambda record: (
        record["source_seed_id"],
        record["candidate_id"],
    )
)

assert len(complete_experiment) == 49

assert len({
    record["candidate_id"]
    for record in complete_experiment
}) == 49

assert len({
    canon(record["question"])
    for record in complete_experiment
}) == 49

complete_covered_seed_ids = sorted({
    record["source_seed_id"]
    for record in complete_experiment
})

complete_missing_seed_ids = sorted(
    set(expected_seed_ids)
    - set(complete_covered_seed_ids)
)

assert complete_missing_seed_ids == []
assert complete_covered_seed_ids == list(
    range(1, 29)
)

complete_semantic_audit_rows = [
    calibrated_inspect_semantic_record(record)
    for record in complete_experiment
]

complete_non_pass_rows = [
    row
    for row in complete_semantic_audit_rows
    if row["audit_status"] != "PASS"
]

assert not complete_non_pass_rows, (
    "Complete experiment contains non-PASS rows: "
    + json.dumps(
        complete_non_pass_rows[:5],
        ensure_ascii=False,
    )
)

for record in complete_experiment:
    assert (
        canon(record["expected_answer"])
        in canon(record["source_evidence"])
    ), (
        "Answer is not grounded in evidence: "
        f"{record['candidate_id']}"
    )

complete_experiment_path = (
    golden_dir
    / f"{FINAL_EXPERIMENT_NAME}.jsonl"
)

complete_quality_report_path = (
    golden_dir
    / (
        f"{FINAL_EXPERIMENT_NAME}"
        "_quality_report.json"
    )
)

complete_review_queue_path = (
    golden_dir
    / (
        f"{FINAL_EXPERIMENT_NAME}"
        "_review_queue.jsonl"
    )
)

write_jsonl_atomic(
    complete_experiment_path,
    complete_experiment,
)

# All 28 seeds are covered in this experiment.
# The queue remains empty but is written explicitly.
write_jsonl_atomic(
    complete_review_queue_path,
    [],
)

complete_seed_counts = Counter(
    record["source_seed_id"]
    for record in complete_experiment
)

complete_quality_report = {
    "recovery_version": (
        SEED_10_RECOVERY_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        "COMPLETE_EXPERIMENT_NOT_ACTIVE_GOLDEN"
    ),
    "active_dataset_created": False,
    "active_v1_modified": False,
    "experiment_path": str(
        complete_experiment_path
    ),
    "experiment_sha256": sha256_file(
        complete_experiment_path
    ),
    "record_count": len(
        complete_experiment
    ),
    "covered_seed_count": len(
        complete_covered_seed_ids
    ),
    "covered_seed_ids": (
        complete_covered_seed_ids
    ),
    "missing_seed_ids": (
        complete_missing_seed_ids
    ),
    "records_by_seed": {
        str(seed_id): count
        for seed_id, count in sorted(
            complete_seed_counts.items()
        )
    },
    "semantic_audit": {
        "pass": len(
            complete_semantic_audit_rows
        ),
        "review": 0,
        "block": 0,
    },
    "scope_recovered_seed_ids": [
        7,
        10,
        22,
    ],
    "active_golden_version": None,
}

write_json_atomic(
    complete_quality_report_path,
    complete_quality_report,
)

print(
    "=== COMPLETE QUOTE-GROUNDED EXPERIMENT ==="
)
print(
    "Input records:",
    len(coverage_repaired_experiment),
)
print("Recovered Seed 10 records: 1")
print(
    "Final records:",
    len(complete_experiment),
)
print(
    "Covered seeds:",
    len(complete_covered_seed_ids),
)
print(
    "Missing seeds:",
    complete_missing_seed_ids,
)
print(
    "Semantic audit PASS:",
    len(complete_semantic_audit_rows),
)
print("Semantic audit REVIEW: 0")
print("Semantic audit BLOCK: 0")
print("Active Golden Dataset created: False")
print("Golden Dataset v1 modified: False")

print(
    "\nExperiment:",
    complete_experiment_path,
)
print(
    "Review queue:",
    complete_review_queue_path,
)
print(
    "Quality report:",
    complete_quality_report_path,
)
print(
    "\nCOMPLETE QUOTE-GROUNDED "
    "EXPERIMENT: PASS"
)

=== COMPLETE QUOTE-GROUNDED EXPERIMENT ===
Input records: 48
Recovered Seed 10 records: 1
Final records: 49
Covered seeds: 28
Missing seeds: []
Semantic audit PASS: 49
Semantic audit REVIEW: 0
Semantic audit BLOCK: 0
Active Golden Dataset created: False
Golden Dataset v1 modified: False

Experiment: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_complete_experiment.jsonl
Review queue: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_complete_experiment_review_queue.jsonl
Quality report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_complete_experiment_quality_report.json

COMPLETE QUOTE-GROUNDED EXPERIMENT: PASS


In [26]:
# Code Cell 26
# NB12G — Complete-experiment lexical retrieval benchmark

assert "complete_experiment" in globals(), (
    "Run Code Cell 25 first."
)
assert "evaluate_lexical_records" in globals(), (
    "Run Code Cell 19 first."
)
assert len(complete_experiment) == 49

COMPLETE_RETRIEVAL_VERSION = (
    "nb12g_complete_lexical_retrieval_v1"
)

complete_retrieval_evaluation = (
    evaluate_lexical_records(
        complete_experiment
    )
)

complete_retrieval_rows = (
    complete_retrieval_evaluation["rows"]
)

complete_retrieval_metrics = (
    complete_retrieval_evaluation["metrics"]
)

complete_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in complete_experiment
}

complete_eligible_rows = [
    row
    for row in complete_retrieval_rows
    if row["first_positive_rank"] is not None
]

complete_excluded_rows = [
    row
    for row in complete_retrieval_rows
    if row["first_positive_rank"] is None
]

rows_by_seed = defaultdict(list)

for row in complete_eligible_rows:
    rows_by_seed[
        row["source_seed_id"]
    ].append(row)

seed_level_rows = []

for seed_id in sorted(rows_by_seed):
    seed_rows = rows_by_seed[seed_id]

    seed_level_rows.append({
        "source_seed_id": seed_id,
        "eligible_query_count": len(seed_rows),
        "recall_at_1": float(np.mean([
            row["hit_at_1"]
            for row in seed_rows
        ])),
        "recall_at_3": float(np.mean([
            row["hit_at_3"]
            for row in seed_rows
        ])),
        "recall_at_5": float(np.mean([
            row["hit_at_5"]
            for row in seed_rows
        ])),
        "recall_at_10": float(np.mean([
            row["hit_at_10"]
            for row in seed_rows
        ])),
        "mrr": float(np.mean([
            row["reciprocal_rank"]
            for row in seed_rows
        ])),
        "mean_first_positive_rank": float(
            np.mean([
                row["first_positive_rank"]
                for row in seed_rows
            ])
        ),
    })

eligible_retrieval_seed_ids = sorted(
    rows_by_seed
)

excluded_retrieval_seed_ids = sorted(
    set(complete_covered_seed_ids)
    - set(eligible_retrieval_seed_ids)
)

seed_macro_metrics = {
    "eligible_seed_count": len(
        seed_level_rows
    ),
    "excluded_seed_ids": (
        excluded_retrieval_seed_ids
    ),
    "macro_recall_at_1": float(np.mean([
        row["recall_at_1"]
        for row in seed_level_rows
    ])),
    "macro_recall_at_3": float(np.mean([
        row["recall_at_3"]
        for row in seed_level_rows
    ])),
    "macro_recall_at_5": float(np.mean([
        row["recall_at_5"]
        for row in seed_level_rows
    ])),
    "macro_recall_at_10": float(np.mean([
        row["recall_at_10"]
        for row in seed_level_rows
    ])),
    "macro_mrr": float(np.mean([
        row["mrr"]
        for row in seed_level_rows
    ])),
    "macro_mean_first_positive_rank": float(
        np.mean([
            row["mean_first_positive_rank"]
            for row in seed_level_rows
        ])
    ),
}

original_complete_delta = {
    metric_name: (
        complete_retrieval_metrics[
            metric_name
        ]
        - original_full_evaluation[
            "metrics"
        ][metric_name]
    )
    for metric_name in (
        "recall_at_1",
        "recall_at_3",
        "recall_at_5",
        "recall_at_10",
        "mrr",
        "mean_first_positive_rank",
    )
}

complete_retrieval_report = {
    "benchmark_version": (
        COMPLETE_RETRIEVAL_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "experiment_path": str(
        complete_experiment_path
    ),
    "experiment_sha256": sha256_file(
        complete_experiment_path
    ),
    "evaluation_contract": {
        "retrieval_method": (
            "persian_tfidf_word_unigram_bigram"
        ),
        "passage_candidate": (
            "parent_child_children"
        ),
        "same_vectorizer_as_prior_diagnostic": (
            True
        ),
        "same_passage_corpus": True,
        "rank_used_as_acceptance_criterion": (
            False
        ),
        "final_nb13_benchmark": False,
    },
    "query_level_metrics": (
        complete_retrieval_metrics
    ),
    "seed_macro_metrics": (
        seed_macro_metrics
    ),
    "original_full_delta": (
        original_complete_delta
    ),
    "seed_level_rows": seed_level_rows,
    "query_rows": complete_retrieval_rows,
}

complete_retrieval_report_path = (
    golden_dir
    / (
        f"{FINAL_EXPERIMENT_NAME}"
        "_lexical_retrieval_benchmark.json"
    )
)

write_json_atomic(
    complete_retrieval_report_path,
    complete_retrieval_report,
)

print(
    "=== COMPLETE-EXPERIMENT "
    "LEXICAL RETRIEVAL ==="
)

print("\nQuery-level metrics:")
print(
    "- Total records:",
    complete_retrieval_metrics[
        "total_records"
    ],
)
print(
    "- Eligible queries:",
    complete_retrieval_metrics[
        "eligible_queries"
    ],
)
print(
    "- Excluded queries:",
    complete_retrieval_metrics[
        "excluded_queries"
    ],
)
print(
    "- Recall@1:",
    f"{complete_retrieval_metrics['recall_at_1']:.4f}",
)
print(
    "- Recall@3:",
    f"{complete_retrieval_metrics['recall_at_3']:.4f}",
)
print(
    "- Recall@5:",
    f"{complete_retrieval_metrics['recall_at_5']:.4f}",
)
print(
    "- Recall@10:",
    f"{complete_retrieval_metrics['recall_at_10']:.4f}",
)
print(
    "- MRR:",
    f"{complete_retrieval_metrics['mrr']:.4f}",
)
print(
    "- Mean first-positive rank:",
    f"{complete_retrieval_metrics['mean_first_positive_rank']:.2f}",
)

print("\nSeed-macro metrics:")
print(
    "- Eligible seeds:",
    seed_macro_metrics[
        "eligible_seed_count"
    ],
)
print(
    "- Excluded seeds:",
    excluded_retrieval_seed_ids,
)
print(
    "- Macro Recall@1:",
    f"{seed_macro_metrics['macro_recall_at_1']:.4f}",
)
print(
    "- Macro Recall@3:",
    f"{seed_macro_metrics['macro_recall_at_3']:.4f}",
)
print(
    "- Macro Recall@5:",
    f"{seed_macro_metrics['macro_recall_at_5']:.4f}",
)
print(
    "- Macro Recall@10:",
    f"{seed_macro_metrics['macro_recall_at_10']:.4f}",
)
print(
    "- Macro MRR:",
    f"{seed_macro_metrics['macro_mrr']:.4f}",
)

print(
    "\nDelta versus original 51-record experiment:"
)
for metric_name, value in (
    original_complete_delta.items()
):
    print(
        f"- {metric_name}: {value:+.4f}"
    )

print("\nWeakest eligible seeds:")
for row in sorted(
    seed_level_rows,
    key=lambda item: (
        item["recall_at_10"],
        item["mrr"],
    ),
)[:10]:
    print(
        f"- seed={row['source_seed_id']} | "
        f"queries={row['eligible_query_count']} | "
        f"Recall@10={row['recall_at_10']:.4f} | "
        f"MRR={row['mrr']:.4f} | "
        f"mean_rank="
        f"{row['mean_first_positive_rank']:.2f}"
    )

print(
    "\nReport:",
    complete_retrieval_report_path,
)
print(
    "\nCOMPLETE-EXPERIMENT "
    "LEXICAL RETRIEVAL: PASS"
)

=== COMPLETE-EXPERIMENT LEXICAL RETRIEVAL ===

Query-level metrics:
- Total records: 49
- Eligible queries: 46
- Excluded queries: 3
- Recall@1: 0.5435
- Recall@3: 0.7609
- Recall@5: 0.8043
- Recall@10: 0.8913
- MRR: 0.6600
- Mean first-positive rank: 9.00

Seed-macro metrics:
- Eligible seeds: 27
- Excluded seeds: [16]
- Macro Recall@1: 0.5617
- Macro Recall@3: 0.7778
- Macro Recall@5: 0.8025
- Macro Recall@10: 0.8827
- Macro MRR: 0.6755

Delta versus original 51-record experiment:
- recall_at_1: -0.1232
- recall_at_3: -0.0308
- recall_at_5: -0.0290
- recall_at_10: -0.0045
- mrr: -0.0757
- mean_first_positive_rank: +1.6042

Weakest eligible seeds:
- seed=10 | queries=1 | Recall@10=0.0000 | MRR=0.0161 | mean_rank=62.00
- seed=14 | queries=1 | Recall@10=0.0000 | MRR=0.0286 | mean_rank=35.00
- seed=12 | queries=3 | Recall@10=0.3333 | MRR=0.0443 | mean_rank=67.67
- seed=6 | queries=2 | Recall@10=0.5000 | MRR=0.5167 | mean_rank=15.50
- seed=11 | queries=1 | Recall@10=1.0000 | MRR=0.1000 | 

In [29]:
# Code Cell 27
# NB12G — Preliminary integrity and leakage audit

assert "complete_experiment" in globals(), (
    "Run Code Cell 25 first."
)
assert len(complete_experiment) == 49

FINAL_INTEGRITY_AUDIT_VERSION = (
    "nb12g_preliminary_integrity_audit_v2"
)

REQUIRED_RECORD_FIELDS = {
    "candidate_id",
    "dataset_version",
    "source_seed_id",
    "source_lineage",
    "question",
    "expected_answer",
    "question_intent",
    "evaluation_split",
    "source_evidence",
    "evidence_quote",
    "legacy_seed_evidence",
    "taxonomy_topics",
    "competency_ids",
    "pilot_case_id",
    "provenance_source",
    "document_ids",
    "retrieval_labels",
    "generation",
    "generation_status",
}

child_chunk_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in children
}

medium_chunk_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in medium
}

all_retrieval_chunk_by_id = {
    **child_chunk_by_id,
    **medium_chunk_by_id,
}

all_parent_chunk_ids = {
    chunk["chunk_id"]
    for chunk in parents
}

complete_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in complete_experiment
}

integrity_rows = []

for record in complete_experiment:
    errors = []
    warnings = []

    candidate_id = record.get("candidate_id")
    seed_id = record.get("source_seed_id")

    missing_fields = sorted(
        REQUIRED_RECORD_FIELDS - set(record)
    )

    if missing_fields:
        errors.append({
            "code": "missing_required_fields",
            "fields": missing_fields,
        })

    try:
        uuid.UUID(str(candidate_id))
    except (
        ValueError,
        TypeError,
        AttributeError,
    ):
        errors.append({
            "code": "invalid_candidate_uuid",
        })

    if record.get("dataset_version") != (
        FINAL_EXPERIMENT_NAME
    ):
        errors.append({
            "code": "invalid_dataset_version",
        })

    if seed_id not in seeds:
        errors.append({
            "code": "unknown_source_seed_id",
        })

    if record.get("evaluation_split") not in {
        "train",
        "dev",
        "test",
    }:
        errors.append({
            "code": "invalid_evaluation_split",
        })

    question = str(
        record.get("question", "")
    ).strip()

    expected_answer = str(
        record.get("expected_answer", "")
    ).strip()

    source_evidence = str(
        record.get("source_evidence", "")
    ).strip()

    evidence_quote = str(
        record.get("evidence_quote", "")
    ).strip()

    if not question.endswith("؟"):
        errors.append({
            "code": "invalid_question_ending",
        })

    if (
        canon(expected_answer)
        not in canon(source_evidence)
    ):
        errors.append({
            "code": (
                "answer_not_in_source_evidence"
            ),
        })

    if (
        canon(source_evidence)
        != canon(evidence_quote)
    ):
        errors.append({
            "code": "evidence_quote_mismatch",
        })

    labels = record.get(
        "retrieval_labels",
        {},
    )

    candidate_name = labels.get("candidate")

    positive_chunk_ids = labels.get(
        "positive_chunk_ids",
        [],
    )

    positive_parent_chunk_ids = labels.get(
        "positive_parent_chunk_ids",
        [],
    )

    if candidate_name not in {
        "parent_child_children",
        "structure_aware_medium",
    }:
        errors.append({
            "code": (
                "invalid_retrieval_candidate"
            ),
        })

    if not positive_chunk_ids:
        errors.append({
            "code": "empty_positive_chunk_ids",
        })

    missing_positive_chunk_ids = [
        chunk_id
        for chunk_id in positive_chunk_ids
        if chunk_id not in all_retrieval_chunk_by_id
    ]

    if missing_positive_chunk_ids:
        errors.append({
            "code": "missing_positive_chunks",
            "chunk_ids": (
                missing_positive_chunk_ids
            ),
        })

    available_positive_chunks = [
        all_retrieval_chunk_by_id[chunk_id]
        for chunk_id in positive_chunk_ids
        if chunk_id in all_retrieval_chunk_by_id
    ]

    evidence_found_in_positive_chunk = any(
        canon(source_evidence)
        in canon(chunk["content"])
        for chunk in available_positive_chunks
    )

    if not evidence_found_in_positive_chunk:
        errors.append({
            "code": (
                "evidence_not_in_positive_chunk"
            ),
        })

    missing_parent_ids = [
        parent_id
        for parent_id in positive_parent_chunk_ids
        if parent_id not in all_parent_chunk_ids
    ]

    if missing_parent_ids:
        errors.append({
            "code": "missing_positive_parents",
            "parent_ids": missing_parent_ids,
        })

    record_document_ids = set(
        record.get("document_ids", [])
    )

    positive_document_ids = set()

    for chunk in available_positive_chunks:
        positive_document_ids.update(
            chunk_identifiers(chunk)
        )

    document_overlap = sorted(
        record_document_ids
        & positive_document_ids
    )

    if not document_overlap:
        warnings.append({
            "code": (
                "no_direct_document_overlap"
            ),
            "positive_chunk_ids": (
                positive_chunk_ids
            ),
        })

    integrity_rows.append({
        "candidate_id": candidate_id,
        "source_seed_id": seed_id,
        "evaluation_split": record.get(
            "evaluation_split"
        ),
        "errors": errors,
        "warnings": warnings,
        "document_overlap": document_overlap,
        "status": (
            "FAIL"
            if errors
            else "PASS_WITH_WARNING"
            if warnings
            else "PASS"
        ),
    })

integrity_failure_rows = [
    row
    for row in integrity_rows
    if row["errors"]
]

integrity_warning_rows = [
    row
    for row in integrity_rows
    if row["warnings"]
]

candidate_ids = [
    record["candidate_id"]
    for record in complete_experiment
]

question_keys = [
    canon(record["question"])
    for record in complete_experiment
]

candidate_ids_unique = (
    len(candidate_ids)
    == len(set(candidate_ids))
)

normalized_questions_unique = (
    len(question_keys)
    == len(set(question_keys))
)

splits_by_seed = defaultdict(set)

for record in complete_experiment:
    splits_by_seed[
        record["source_seed_id"]
    ].add(record["evaluation_split"])

seed_split_failures = {
    seed_id: sorted(split_values)
    for seed_id, split_values
    in splits_by_seed.items()
    if len(split_values) != 1
}

evidence_splits = defaultdict(set)
evidence_candidates = defaultdict(list)

for record in complete_experiment:
    evidence_hash = hashlib.sha256(
        canon(
            record["source_evidence"]
        ).encode("utf-8")
    ).hexdigest()

    evidence_splits[evidence_hash].add(
        record["evaluation_split"]
    )

    evidence_candidates[
        evidence_hash
    ].append(record["candidate_id"])

cross_split_evidence_leakage = [
    {
        "evidence_hash": evidence_hash,
        "splits": sorted(split_values),
        "candidate_ids": evidence_candidates[
            evidence_hash
        ],
    }
    for evidence_hash, split_values
    in evidence_splits.items()
    if len(split_values) > 1
]

document_splits = defaultdict(set)
document_candidates = defaultdict(list)

for record in complete_experiment:
    for document_id in record.get(
        "document_ids",
        [],
    ):
        document_splits[document_id].add(
            record["evaluation_split"]
        )

        document_candidates[
            document_id
        ].append(record["candidate_id"])

cross_split_document_reuse = [
    {
        "document_id": document_id,
        "splits": sorted(split_values),
        "candidate_count": len(
            document_candidates[document_id]
        ),
    }
    for document_id, split_values
    in document_splits.items()
    if len(split_values) > 1
]

preliminary_integrity_status = (
    "FINDINGS_REQUIRING_REPAIR"
    if (
        integrity_failure_rows
        or not candidate_ids_unique
        or not normalized_questions_unique
        or seed_split_failures
        or cross_split_evidence_leakage
    )
    else (
        "PASS_WITH_DOCUMENT_REUSE_REVIEW"
        if cross_split_document_reuse
        else "PASS"
    )
)

final_integrity_report = {
    "audit_version": (
        FINAL_INTEGRITY_AUDIT_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        preliminary_integrity_status
    ),
    "active_dataset_created": False,
    "active_v1_modified": False,
    "experiment_path": str(
        complete_experiment_path
    ),
    "experiment_sha256": sha256_file(
        complete_experiment_path
    ),
    "record_count": len(
        complete_experiment
    ),
    "candidate_ids_unique": (
        candidate_ids_unique
    ),
    "normalized_questions_unique": (
        normalized_questions_unique
    ),
    "covered_seed_count": len(
        splits_by_seed
    ),
    "seed_split_failure_count": len(
        seed_split_failures
    ),
    "seed_split_failures": (
        seed_split_failures
    ),
    "integrity_failure_count": len(
        integrity_failure_rows
    ),
    "integrity_warning_count": len(
        integrity_warning_rows
    ),
    "cross_split_evidence_leakage_count": len(
        cross_split_evidence_leakage
    ),
    "cross_split_evidence_leakage": (
        cross_split_evidence_leakage
    ),
    "cross_split_document_reuse_count": len(
        cross_split_document_reuse
    ),
    "cross_split_document_reuse": (
        cross_split_document_reuse
    ),
    "rows": integrity_rows,
}

final_integrity_report_path = (
    golden_dir
    / (
        f"{FINAL_EXPERIMENT_NAME}"
        "_preliminary_integrity_audit.json"
    )
)

write_json_atomic(
    final_integrity_report_path,
    final_integrity_report,
)

print(
    "=== PRELIMINARY COMPLETE-EXPERIMENT "
    "INTEGRITY AUDIT ==="
)
print(
    "Records:",
    len(complete_experiment),
)
print(
    "Integrity failures:",
    len(integrity_failure_rows),
)
print(
    "Integrity warnings:",
    len(integrity_warning_rows),
)
print(
    "Candidate IDs unique:",
    candidate_ids_unique,
)
print(
    "Normalized questions unique:",
    normalized_questions_unique,
)
print(
    "Covered seeds:",
    len(splits_by_seed),
)
print(
    "Seed split failures:",
    len(seed_split_failures),
)
print(
    "Cross-split exact-evidence leakage:",
    len(cross_split_evidence_leakage),
)
print(
    "Cross-split document reuse:",
    len(cross_split_document_reuse),
)

if integrity_failure_rows:
    print("\nFindings requiring repair:")

    for row in integrity_failure_rows:
        print(
            f"- seed={row['source_seed_id']} | "
            f"candidate={row['candidate_id']} | "
            f"errors={row['errors']}"
        )

if integrity_warning_rows:
    print("\nWarnings:")

    for row in integrity_warning_rows:
        print(
            f"- seed={row['source_seed_id']} | "
            f"candidate={row['candidate_id']} | "
            f"warnings={row['warnings']}"
        )

print(
    "\nStatus:",
    preliminary_integrity_status,
)
print(
    "Report:",
    final_integrity_report_path,
)
print(
    "\nPRELIMINARY COMPLETE-EXPERIMENT "
    "INTEGRITY AUDIT: COMPLETE"
)

=== PRELIMINARY COMPLETE-EXPERIMENT INTEGRITY AUDIT ===
Records: 49
Integrity failures: 3
Integrity warnings: 3
Candidate IDs unique: True
Normalized questions unique: True
Covered seeds: 28
Seed split failures: 0
Cross-split exact-evidence leakage: 0
Cross-split document reuse: 0

Findings requiring repair:
- seed=11 | candidate=47fb0cf4-5c37-57fe-ac56-c1e670d34ad3 | errors=[{'code': 'evidence_not_in_positive_chunk'}]
- seed=26 | candidate=1cd800a6-4fc2-50a8-a1bb-6008d3b26979 | errors=[{'code': 'evidence_not_in_positive_chunk'}]
- seed=28 | candidate=f150e34b-09d1-58a3-8ad6-cddff02825aa | errors=[{'code': 'evidence_not_in_positive_chunk'}]

Warnings:
- seed=16 | candidate=227f8f91-7002-5885-b664-b7d84d4b56b7 | warnings=[{'code': 'no_direct_document_overlap', 'positive_chunk_ids': ['chunk:structure_aware_medium:retrieval:b29c9b340d024d892f01bfb8']}]
- seed=16 | candidate=588c882a-1ce4-5f11-bf9c-54ed6d793d97 | warnings=[{'code': 'no_direct_document_overlap', 'positive_chunk_ids': ['chun

In [30]:
# Code Cell 28
# NB12G — Positive-label evidence mismatch diagnosis

assert "integrity_failure_rows" in globals(), (
    "Run Code Cell 27 first."
)
assert len(integrity_failure_rows) == 3

mismatch_candidate_ids = {
    row["candidate_id"]
    for row in integrity_failure_rows
}

all_searchable_chunks = [
    ("child", chunk)
    for chunk in children
] + [
    ("medium", chunk)
    for chunk in medium
] + [
    ("parent", chunk)
    for chunk in parents
]

evidence_mismatch_diagnostics = []

for candidate_id in sorted(
    mismatch_candidate_ids
):
    record = complete_record_by_candidate_id[
        candidate_id
    ]

    evidence = record["source_evidence"]
    labels = record["retrieval_labels"]

    exact_matches = []
    ranked_matches = []

    for chunk_type, chunk in all_searchable_chunks:
        exact_match = (
            canon(evidence)
            in canon(chunk["content"])
        )

        match_coverage = coverage(
            evidence,
            chunk["content"],
        )

        document_overlap = sorted(
            set(record["document_ids"])
            & chunk_identifiers(chunk)
        )

        if exact_match:
            exact_matches.append({
                "chunk_type": chunk_type,
                "chunk_id": chunk["chunk_id"],
                "parent_chunk_id": chunk.get(
                    "parent_chunk_id"
                ),
                "document_overlap": (
                    document_overlap
                ),
                "coverage": float(
                    match_coverage
                ),
            })

        if document_overlap:
            ranked_matches.append({
                "chunk_type": chunk_type,
                "chunk_id": chunk["chunk_id"],
                "parent_chunk_id": chunk.get(
                    "parent_chunk_id"
                ),
                "document_overlap": (
                    document_overlap
                ),
                "coverage": float(
                    match_coverage
                ),
                "content": chunk["content"],
            })

    ranked_matches.sort(
        key=lambda row: (
            -row["coverage"],
            row["chunk_type"] == "parent",
            row["chunk_id"],
        )
    )

    labeled_chunks = []

    for chunk_id in labels.get(
        "positive_chunk_ids",
        [],
    ):
        chunk = all_retrieval_chunk_by_id.get(
            chunk_id
        )

        labeled_chunks.append({
            "chunk_id": chunk_id,
            "found": chunk is not None,
            "content": (
                chunk["content"]
                if chunk is not None
                else None
            ),
            "coverage": (
                coverage(
                    evidence,
                    chunk["content"],
                )
                if chunk is not None
                else None
            ),
        })

    labeled_parents = []

    for parent_id in labels.get(
        "positive_parent_chunk_ids",
        [],
    ):
        parent = parent_by_id.get(parent_id)

        labeled_parents.append({
            "parent_chunk_id": parent_id,
            "found": parent is not None,
            "contains_exact_evidence": (
                canon(evidence)
                in canon(parent["content"])
                if parent is not None
                else False
            ),
            "coverage": (
                coverage(
                    evidence,
                    parent["content"],
                )
                if parent is not None
                else None
            ),
            "content": (
                parent["content"]
                if parent is not None
                else None
            ),
        })

    evidence_mismatch_diagnostics.append({
        "candidate_id": candidate_id,
        "source_seed_id": record[
            "source_seed_id"
        ],
        "question": record["question"],
        "expected_answer": record[
            "expected_answer"
        ],
        "source_evidence": evidence,
        "retrieval_labels": labels,
        "labeled_chunks": labeled_chunks,
        "labeled_parents": labeled_parents,
        "exact_matches": exact_matches,
        "best_document_linked_matches": (
            ranked_matches[:5]
        ),
    })

evidence_mismatch_report_path = (
    golden_dir
    / (
        f"{FINAL_EXPERIMENT_NAME}"
        "_evidence_label_mismatch_diagnostic.json"
    )
)

write_json_atomic(
    evidence_mismatch_report_path,
    {
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "active_dataset_created": False,
        "record_count": len(
            evidence_mismatch_diagnostics
        ),
        "records": (
            evidence_mismatch_diagnostics
        ),
    },
)

print(
    "=== POSITIVE-LABEL EVIDENCE "
    "MISMATCH DIAGNOSIS ==="
)

for diagnostic in (
    evidence_mismatch_diagnostics
):
    print("\n" + "=" * 80)
    print(
        "Seed:",
        diagnostic["source_seed_id"],
    )
    print(
        "Candidate:",
        diagnostic["candidate_id"],
    )
    print(
        "Question:",
        diagnostic["question"],
    )
    print(
        "Evidence:",
        diagnostic["source_evidence"],
    )

    print("\nCurrent positive labels:")
    print(json.dumps(
        diagnostic["retrieval_labels"],
        ensure_ascii=False,
        indent=2,
    ))

    print("\nLabeled chunks:")
    for chunk in diagnostic[
        "labeled_chunks"
    ]:
        print(
            f"- {chunk['chunk_id']} | "
            f"found={chunk['found']} | "
            f"coverage={chunk['coverage']}"
        )
        if chunk["content"]:
            print(
                re.sub(
                    r"\s+",
                    " ",
                    chunk["content"],
                )[:1000]
            )

    print("\nLabeled parents:")
    if diagnostic["labeled_parents"]:
        for parent in diagnostic[
            "labeled_parents"
        ]:
            print(
                f"- {parent['parent_chunk_id']} | "
                f"exact="
                f"{parent['contains_exact_evidence']} | "
                f"coverage={parent['coverage']}"
            )
    else:
        print("- None")

    print("\nExact matches elsewhere:")
    if diagnostic["exact_matches"]:
        for match in diagnostic[
            "exact_matches"
        ]:
            print(
                f"- type={match['chunk_type']} | "
                f"chunk={match['chunk_id']} | "
                f"parent={match['parent_chunk_id']} | "
                f"coverage={match['coverage']:.4f}"
            )
    else:
        print("- None")

    print(
        "\nBest provenance-linked alternatives:"
    )
    for match in diagnostic[
        "best_document_linked_matches"
    ]:
        print(
            f"- type={match['chunk_type']} | "
            f"chunk={match['chunk_id']} | "
            f"coverage={match['coverage']:.4f}"
        )
        print(
            re.sub(
                r"\s+",
                " ",
                match["content"],
            )[:1000]
        )

print(
    "\nReport:",
    evidence_mismatch_report_path,
)
print(
    "\nPOSITIVE-LABEL EVIDENCE "
    "MISMATCH DIAGNOSIS: PASS"
)

=== POSITIVE-LABEL EVIDENCE MISMATCH DIAGNOSIS ===

Seed: 26
Candidate: 1cd800a6-4fc2-50a8-a1bb-6008d3b26979
Question: چه چیزی مخاطبان را تشویق به خرید کالا یا خدماتی می‌کند؟
Evidence: اما اگر هدف از این هزینه‌کردها، مورد قبول واقع شدن در گروه خاصی است، باید بیشتر دربارۀ آن تأمل کرد. خرید در اثر تبلیغات و فعالیت‌های بازاریابی: گاهی در وب‌سایت‌ها و شبکه‌های اجتماعی کالا یا خدماتی تبلیغ می‌شود که الزاماً نیاز افراد را تأمین نمی‌کند، اما فعالیت‌های بازاریابی، مخاطبان را تشویق به خرید آن می‌کنند.

Current positive labels:
{
  "candidate": "parent_child_children",
  "positive_chunk_ids": [
    "chunk:parent_child:child:5376d8f5647efcde0f25a93a"
  ],
  "positive_parent_chunk_ids": [
    "chunk:parent_child:parent:91a0f1b8b02a3ad80a631901"
  ],
  "label_basis": "DIRECT_PROVENANCE_WINDOW_REPAIR"
}

Labeled chunks:
- chunk:parent_child:child:5376d8f5647efcde0f25a93a | found=True | coverage=1.0
الگوهای خرید دانشجویان انجام کارهایی از جمله، دستیاری اساتید، شرکت در انجمن‌های دانشجویی، حضور در شرکت

In [31]:
# Code Cell 29
# NB12G — Repair non-contiguous evidence spans

assert "evidence_mismatch_diagnostics" in globals(), (
    "Run Code Cell 28 first."
)
assert len(evidence_mismatch_diagnostics) == 3
assert len(complete_experiment) == 49

EVIDENCE_SPAN_REPAIR_VERSION = (
    "nb12g_contiguous_evidence_repair_v1"
)

INTEGRITY_REPAIRED_EXPERIMENT_NAME = (
    "golden_dataset_quote_grounded_integrity_repaired_experiment"
)

mismatched_candidate_ids = {
    diagnostic["candidate_id"]
    for diagnostic in evidence_mismatch_diagnostics
}

def sentence_units_preserving_short_text(text):
    units = re.split(
        r"(?<=[.!؟])\s+|\n+",
        str(text),
    )

    return [
        unit.strip()
        for unit in units
        if unit.strip()
    ]

def smallest_contiguous_answer_window(
    chunk_content,
    expected_answer,
    maximum_units=4,
):
    units = sentence_units_preserving_short_text(
        chunk_content
    )

    candidates = []

    for start_index in range(len(units)):
        for width in range(
            1,
            maximum_units + 1,
        ):
            end_index = start_index + width

            if end_index > len(units):
                break

            window = " ".join(
                units[start_index:end_index]
            ).strip()

            if (
                canon(expected_answer)
                not in canon(window)
            ):
                continue

            if (
                canon(window)
                not in canon(chunk_content)
            ):
                continue

            candidates.append({
                "text": window,
                "start_unit": start_index,
                "end_unit": end_index - 1,
                "unit_count": width,
                "token_count": len(words(window)),
            })

    if not candidates:
        return None

    candidates.sort(
        key=lambda item: (
            item["token_count"],
            item["unit_count"],
            item["start_unit"],
        )
    )

    return candidates[0]

integrity_repaired_experiment = []
evidence_span_repair_rows = []

for original_record in complete_experiment:
    original_candidate_id = original_record[
        "candidate_id"
    ]

    if (
        original_candidate_id
        not in mismatched_candidate_ids
    ):
        retained_record = dict(
            original_record
        )
        retained_record["dataset_version"] = (
            INTEGRITY_REPAIRED_EXPERIMENT_NAME
        )
        integrity_repaired_experiment.append(
            retained_record
        )
        continue

    positive_chunk_ids = original_record[
        "retrieval_labels"
    ]["positive_chunk_ids"]

    assert positive_chunk_ids

    selected_window = None
    selected_chunk_id = None

    for chunk_id in positive_chunk_ids:
        chunk = all_retrieval_chunk_by_id.get(
            chunk_id
        )

        if chunk is None:
            continue

        candidate_window = (
            smallest_contiguous_answer_window(
                chunk["content"],
                original_record[
                    "expected_answer"
                ],
            )
        )

        if candidate_window is not None:
            selected_window = candidate_window
            selected_chunk_id = chunk_id
            break

    assert selected_window is not None, (
        "Could not recover contiguous evidence for "
        f"{original_candidate_id}."
    )

    repaired_evidence = selected_window[
        "text"
    ]

    assert (
        canon(original_record["expected_answer"])
        in canon(repaired_evidence)
    )

    assert (
        canon(repaired_evidence)
        in canon(
            all_retrieval_chunk_by_id[
                selected_chunk_id
            ]["content"]
        )
    )

    replacement_candidate_id = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            (
                f"{EVIDENCE_SPAN_REPAIR_VERSION}:"
                f"{original_candidate_id}:"
                f"{selected_chunk_id}:"
                f"{canon(repaired_evidence)}"
            ),
        )
    )

    repaired_record = dict(original_record)

    repaired_record["candidate_id"] = (
        replacement_candidate_id
    )
    repaired_record["dataset_version"] = (
        INTEGRITY_REPAIRED_EXPERIMENT_NAME
    )
    repaired_record["source_evidence"] = (
        repaired_evidence
    )
    repaired_record["evidence_quote"] = (
        repaired_evidence
    )

    prior_curation = dict(
        original_record.get("curation", {})
    )

    repaired_record["curation"] = {
        **prior_curation,
        "evidence_span_repair_version": (
            EVIDENCE_SPAN_REPAIR_VERSION
        ),
        "evidence_span_decision": (
            "NARROW_TO_CONTIGUOUS_ANSWER_WINDOW"
        ),
        "source_candidate_id": (
            prior_curation.get(
                "source_candidate_id",
                original_candidate_id,
            )
        ),
        "supersedes_candidate_id": (
            original_candidate_id
        ),
        "active_dataset_record": False,
    }

    integrity_repaired_experiment.append(
        repaired_record
    )

    evidence_span_repair_rows.append({
        "source_seed_id": original_record[
            "source_seed_id"
        ],
        "old_candidate_id": (
            original_candidate_id
        ),
        "new_candidate_id": (
            replacement_candidate_id
        ),
        "positive_chunk_id": (
            selected_chunk_id
        ),
        "expected_answer": original_record[
            "expected_answer"
        ],
        "old_source_evidence": (
            original_record[
                "source_evidence"
            ]
        ),
        "new_source_evidence": (
            repaired_evidence
        ),
        "new_evidence_token_count": (
            selected_window[
                "token_count"
            ]
        ),
        "unit_count": selected_window[
            "unit_count"
        ],
    })

integrity_repaired_experiment.sort(
    key=lambda record: (
        record["source_seed_id"],
        record["candidate_id"],
    )
)

assert len(integrity_repaired_experiment) == 49
assert len(evidence_span_repair_rows) == 3

assert len({
    record["candidate_id"]
    for record in integrity_repaired_experiment
}) == 49

assert len({
    canon(record["question"])
    for record in integrity_repaired_experiment
}) == 49

for record in integrity_repaired_experiment:
    assert (
        canon(record["expected_answer"])
        in canon(record["source_evidence"])
    )

    positive_chunks = [
        all_retrieval_chunk_by_id[
            chunk_id
        ]
        for chunk_id in record[
            "retrieval_labels"
        ]["positive_chunk_ids"]
        if chunk_id in all_retrieval_chunk_by_id
    ]

    assert positive_chunks

    assert any(
        canon(record["source_evidence"])
        in canon(chunk["content"])
        for chunk in positive_chunks
    ), (
        "Evidence still does not occur contiguously "
        f"in its positive chunk: "
        f"{record['candidate_id']}"
    )

integrity_repaired_experiment_path = (
    golden_dir
    / (
        f"{INTEGRITY_REPAIRED_EXPERIMENT_NAME}"
        ".jsonl"
    )
)

evidence_span_repair_report_path = (
    golden_dir
    / (
        f"{INTEGRITY_REPAIRED_EXPERIMENT_NAME}"
        "_evidence_span_repair.json"
    )
)

write_jsonl_atomic(
    integrity_repaired_experiment_path,
    integrity_repaired_experiment,
)

write_json_atomic(
    evidence_span_repair_report_path,
    {
        "repair_version": (
            EVIDENCE_SPAN_REPAIR_VERSION
        ),
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "active_dataset_created": False,
        "active_v1_modified": False,
        "input_experiment_path": str(
            complete_experiment_path
        ),
        "input_experiment_sha256": (
            sha256_file(
                complete_experiment_path
            )
        ),
        "output_experiment_path": str(
            integrity_repaired_experiment_path
        ),
        "output_experiment_sha256": (
            sha256_file(
                integrity_repaired_experiment_path
            )
        ),
        "record_count": len(
            integrity_repaired_experiment
        ),
        "repaired_record_count": len(
            evidence_span_repair_rows
        ),
        "repairs": (
            evidence_span_repair_rows
        ),
    },
)

print(
    "=== CONTIGUOUS EVIDENCE-SPAN REPAIR ==="
)
print(
    "Input records:",
    len(complete_experiment),
)
print(
    "Output records:",
    len(integrity_repaired_experiment),
)
print(
    "Repaired evidence spans:",
    len(evidence_span_repair_rows),
)

for row in evidence_span_repair_rows:
    print("\n" + "=" * 80)
    print(
        "Seed:",
        row["source_seed_id"],
    )
    print(
        "Old candidate:",
        row["old_candidate_id"],
    )
    print(
        "New candidate:",
        row["new_candidate_id"],
    )
    print(
        "Answer:",
        row["expected_answer"],
    )
    print("New contiguous evidence:")
    print(row["new_source_evidence"])

print(
    "\nExperiment:",
    integrity_repaired_experiment_path,
)
print(
    "Report:",
    evidence_span_repair_report_path,
)
print(
    "\nCONTIGUOUS EVIDENCE-SPAN "
    "REPAIR: PASS"
)

=== CONTIGUOUS EVIDENCE-SPAN REPAIR ===
Input records: 49
Output records: 49
Repaired evidence spans: 3

Seed: 11
Old candidate: 47fb0cf4-5c37-57fe-ac56-c1e670d34ad3
New candidate: 7d56f6ba-5fc5-5b61-9a23-1a05ca4c735e
Answer: پول روی درخت رشد نمی‌کند؛ پشت آن کار و فعالیت وجود دارد.
New contiguous evidence:
پول روی درخت رشد نمی‌کند؛ پشت آن کار و فعالیت وجود دارد.

Seed: 26
Old candidate: 1cd800a6-4fc2-50a8-a1bb-6008d3b26979
New candidate: 1c4c7377-88cf-5e59-8728-c30614bb5bf5
Answer: فعالیت‌های بازاریابی، مخاطبان را تشویق به خرید آن می‌کنند.
New contiguous evidence:
گاهی در وب‌سایت‌ها و شبکه‌های اجتماعی کالا یا خدماتی تبلیغ می‌شود که الزاماً نیاز افراد را تأمین نمی‌کند، اما فعالیت‌های بازاریابی، مخاطبان را تشویق به خرید آن می‌کنند.

Seed: 28
Old candidate: f150e34b-09d1-58a3-8ad6-cddff02825aa
New candidate: a864f7bb-249a-5986-9cdc-de072d8559c8
Answer: وجود چیزهای ارزشمند اطراف‌مان را یادمان می‌رود و جز با پول نمی‌توانیم خودمان را ثروتمند بدانیم.
New contiguous evidence:
گاهی رسیدن به خوا

In [32]:
# Code Cell 30
# NB12G — Final integrity audit after evidence repair

assert "integrity_repaired_experiment" in globals(), (
    "Run Code Cell 29 first."
)
assert len(integrity_repaired_experiment) == 49

FINAL_REPAIRED_AUDIT_VERSION = (
    "nb12g_final_integrity_audit_v1"
)

ALLOWED_FALLBACK_SEED_IDS = {16}
ALLOWED_FALLBACK_LABEL_BASIS = {
    "EXACT_REVIEWED_EVIDENCE_FALLBACK",
}

final_repaired_integrity_rows = []

for record in integrity_repaired_experiment:
    errors = []
    accepted_exceptions = []

    candidate_id = record.get("candidate_id")
    seed_id = record.get("source_seed_id")

    missing_fields = sorted(
        REQUIRED_RECORD_FIELDS - set(record)
    )

    if missing_fields:
        errors.append({
            "code": "missing_required_fields",
            "fields": missing_fields,
        })

    try:
        uuid.UUID(str(candidate_id))
    except (
        ValueError,
        TypeError,
        AttributeError,
    ):
        errors.append({
            "code": "invalid_candidate_uuid",
        })

    if record.get("dataset_version") != (
        INTEGRITY_REPAIRED_EXPERIMENT_NAME
    ):
        errors.append({
            "code": "invalid_dataset_version",
        })

    if seed_id not in seeds:
        errors.append({
            "code": "unknown_source_seed_id",
        })

    if record.get("evaluation_split") not in {
        "train",
        "dev",
        "test",
    }:
        errors.append({
            "code": "invalid_evaluation_split",
        })

    question = str(
        record.get("question", "")
    ).strip()

    expected_answer = str(
        record.get("expected_answer", "")
    ).strip()

    source_evidence = str(
        record.get("source_evidence", "")
    ).strip()

    evidence_quote = str(
        record.get("evidence_quote", "")
    ).strip()

    if not question.endswith("؟"):
        errors.append({
            "code": "invalid_question_ending",
        })

    if (
        canon(expected_answer)
        not in canon(source_evidence)
    ):
        errors.append({
            "code": (
                "answer_not_in_source_evidence"
            ),
        })

    if (
        canon(source_evidence)
        != canon(evidence_quote)
    ):
        errors.append({
            "code": "evidence_quote_mismatch",
        })

    semantic_result = (
        calibrated_inspect_semantic_record(
            record
        )
    )

    if semantic_result["audit_status"] != "PASS":
        errors.append({
            "code": "semantic_audit_failure",
            "blocking_errors": (
                semantic_result[
                    "blocking_errors"
                ]
            ),
            "review_flags": (
                semantic_result[
                    "review_flags"
                ]
            ),
        })

    labels = record.get(
        "retrieval_labels",
        {},
    )

    candidate_name = labels.get("candidate")
    label_basis = labels.get("label_basis")

    positive_chunk_ids = labels.get(
        "positive_chunk_ids",
        [],
    )

    positive_parent_chunk_ids = labels.get(
        "positive_parent_chunk_ids",
        [],
    )

    if candidate_name not in {
        "parent_child_children",
        "structure_aware_medium",
    }:
        errors.append({
            "code": (
                "invalid_retrieval_candidate"
            ),
        })

    if not positive_chunk_ids:
        errors.append({
            "code": "empty_positive_chunk_ids",
        })

    missing_positive_chunk_ids = [
        chunk_id
        for chunk_id in positive_chunk_ids
        if chunk_id not in all_retrieval_chunk_by_id
    ]

    if missing_positive_chunk_ids:
        errors.append({
            "code": "missing_positive_chunks",
            "chunk_ids": (
                missing_positive_chunk_ids
            ),
        })

    available_positive_chunks = [
        all_retrieval_chunk_by_id[chunk_id]
        for chunk_id in positive_chunk_ids
        if chunk_id in all_retrieval_chunk_by_id
    ]

    evidence_found_in_positive_chunk = any(
        canon(source_evidence)
        in canon(chunk["content"])
        for chunk in available_positive_chunks
    )

    if not evidence_found_in_positive_chunk:
        errors.append({
            "code": (
                "evidence_not_in_positive_chunk"
            ),
        })

    missing_parent_ids = [
        parent_id
        for parent_id in positive_parent_chunk_ids
        if parent_id not in all_parent_chunk_ids
    ]

    if missing_parent_ids:
        errors.append({
            "code": "missing_positive_parents",
            "parent_ids": missing_parent_ids,
        })

    record_document_ids = set(
        record.get("document_ids", [])
    )

    positive_document_ids = set()

    for chunk in available_positive_chunks:
        positive_document_ids.update(
            chunk_identifiers(chunk)
        )

    document_overlap = sorted(
        record_document_ids
        & positive_document_ids
    )

    if not document_overlap:
        fallback_is_allowed = (
            seed_id in ALLOWED_FALLBACK_SEED_IDS
            and label_basis
            in ALLOWED_FALLBACK_LABEL_BASIS
            and evidence_found_in_positive_chunk
        )

        if fallback_is_allowed:
            accepted_exceptions.append({
                "code": (
                    "accepted_exact_reviewed_evidence_fallback"
                ),
                "label_basis": label_basis,
            })
        else:
            errors.append({
                "code": (
                    "unapproved_missing_document_overlap"
                ),
                "label_basis": label_basis,
            })

    final_repaired_integrity_rows.append({
        "candidate_id": candidate_id,
        "source_seed_id": seed_id,
        "evaluation_split": record.get(
            "evaluation_split"
        ),
        "errors": errors,
        "accepted_exceptions": (
            accepted_exceptions
        ),
        "document_overlap": document_overlap,
        "status": (
            "FAIL"
            if errors
            else "PASS_WITH_ACCEPTED_EXCEPTION"
            if accepted_exceptions
            else "PASS"
        ),
    })

final_repaired_failure_rows = [
    row
    for row in final_repaired_integrity_rows
    if row["errors"]
]

final_repaired_exception_rows = [
    row
    for row in final_repaired_integrity_rows
    if row["accepted_exceptions"]
]

repaired_candidate_ids = [
    record["candidate_id"]
    for record in integrity_repaired_experiment
]

repaired_question_keys = [
    canon(record["question"])
    for record in integrity_repaired_experiment
]

assert len(repaired_candidate_ids) == len(
    set(repaired_candidate_ids)
), "Duplicate candidate IDs remain."

assert len(repaired_question_keys) == len(
    set(repaired_question_keys)
), "Duplicate normalized questions remain."

repaired_splits_by_seed = defaultdict(set)

for record in integrity_repaired_experiment:
    repaired_splits_by_seed[
        record["source_seed_id"]
    ].add(record["evaluation_split"])

repaired_seed_split_failures = {
    seed_id: sorted(split_values)
    for seed_id, split_values
    in repaired_splits_by_seed.items()
    if len(split_values) != 1
}

assert not repaired_seed_split_failures, (
    "Seed split inconsistency remains: "
    f"{repaired_seed_split_failures}"
)

repaired_evidence_splits = defaultdict(set)

for record in integrity_repaired_experiment:
    evidence_hash = hashlib.sha256(
        canon(
            record["source_evidence"]
        ).encode("utf-8")
    ).hexdigest()

    repaired_evidence_splits[
        evidence_hash
    ].add(record["evaluation_split"])

repaired_cross_split_evidence_leakage = [
    {
        "evidence_hash": evidence_hash,
        "splits": sorted(split_values),
    }
    for evidence_hash, split_values
    in repaired_evidence_splits.items()
    if len(split_values) > 1
]

assert not repaired_cross_split_evidence_leakage, (
    "Cross-split exact-evidence leakage remains."
)

covered_seed_ids_after_repair = sorted(
    repaired_splits_by_seed
)

assert covered_seed_ids_after_repair == list(
    range(1, 29)
)

unexpected_exception_rows = [
    row
    for row in final_repaired_exception_rows
    if row["source_seed_id"] != 16
]

assert not unexpected_exception_rows, (
    "Unexpected integrity exceptions detected."
)

assert len(final_repaired_exception_rows) == 3
assert {
    row["source_seed_id"]
    for row in final_repaired_exception_rows
} == {16}

assert not final_repaired_failure_rows, (
    "Final integrity failures remain: "
    + json.dumps(
        final_repaired_failure_rows[:10],
        ensure_ascii=False,
    )
)

final_repaired_integrity_report = {
    "audit_version": (
        FINAL_REPAIRED_AUDIT_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        "PASS_WITH_DOCUMENTED_SEED16_FALLBACK"
    ),
    "active_dataset_created": False,
    "active_v1_modified": False,
    "experiment_path": str(
        integrity_repaired_experiment_path
    ),
    "experiment_sha256": sha256_file(
        integrity_repaired_experiment_path
    ),
    "record_count": len(
        integrity_repaired_experiment
    ),
    "covered_seed_count": len(
        covered_seed_ids_after_repair
    ),
    "covered_seed_ids": (
        covered_seed_ids_after_repair
    ),
    "candidate_ids_unique": True,
    "normalized_questions_unique": True,
    "seed_split_consistency": True,
    "cross_split_evidence_leakage_count": 0,
    "integrity_failure_count": 0,
    "accepted_exception_count": len(
        final_repaired_exception_rows
    ),
    "accepted_exception_seed_ids": [16],
    "accepted_exception_contract": {
        "reason": (
            "Seed 16 has no registered document ID, "
            "but its reviewed evidence occurs exactly "
            "inside the frozen medium chunk."
        ),
        "allowed_label_basis": sorted(
            ALLOWED_FALLBACK_LABEL_BASIS
        ),
    },
    "rows": final_repaired_integrity_rows,
}

final_repaired_integrity_report_path = (
    golden_dir
    / (
        f"{INTEGRITY_REPAIRED_EXPERIMENT_NAME}"
        "_final_integrity_audit.json"
    )
)

write_json_atomic(
    final_repaired_integrity_report_path,
    final_repaired_integrity_report,
)

print(
    "=== FINAL REPAIRED-EXPERIMENT "
    "INTEGRITY AUDIT ==="
)
print(
    "Records:",
    len(integrity_repaired_experiment),
)
print(
    "Covered seeds:",
    len(covered_seed_ids_after_repair),
)
print(
    "Integrity failures:",
    len(final_repaired_failure_rows),
)
print(
    "Accepted exceptions:",
    len(final_repaired_exception_rows),
)
print(
    "Accepted-exception seeds:",
    sorted({
        row["source_seed_id"]
        for row in final_repaired_exception_rows
    }),
)
print(
    "Candidate IDs unique:",
    True,
)
print(
    "Normalized questions unique:",
    True,
)
print(
    "Seed split failures:",
    len(repaired_seed_split_failures),
)
print(
    "Cross-split exact-evidence leakage:",
    len(
        repaired_cross_split_evidence_leakage
    ),
)
print(
    "Active Golden Dataset created:",
    False,
)
print(
    "Golden Dataset v1 modified:",
    False,
)
print(
    "\nReport:",
    final_repaired_integrity_report_path,
)
print(
    "\nFINAL REPAIRED-EXPERIMENT "
    "INTEGRITY AUDIT: PASS"
)

=== FINAL REPAIRED-EXPERIMENT INTEGRITY AUDIT ===
Records: 49
Covered seeds: 28
Integrity failures: 0
Accepted exceptions: 3
Accepted-exception seeds: [16]
Candidate IDs unique: True
Normalized questions unique: True
Seed split failures: 0
Cross-split exact-evidence leakage: 0
Active Golden Dataset created: False
Golden Dataset v1 modified: False

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_integrity_repaired_experiment_final_integrity_audit.json

FINAL REPAIRED-EXPERIMENT INTEGRITY AUDIT: PASS


In [33]:
# Code Cell 31
# NB12G — Full semantic-judge evaluation

assert "integrity_repaired_experiment" in globals(), (
    "Run Code Cell 29 first."
)
assert "chat" in globals(), (
    "Run Code Cell 08 first."
)
assert "chat_model" in globals()
assert len(integrity_repaired_experiment) == 49

SEMANTIC_JUDGE_VERSION = (
    "nb12g_full_semantic_judge_v1"
)

semantic_judge_cache_directory = (
    golden_dir
    / "cache"
    / EXPERIMENT_NAME
    / "semantic_judge"
    / SEMANTIC_JUDGE_VERSION
)

semantic_judge_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)

def semantic_judge_prompt(record):
    return f"""
نقش شما داور سخت‌گیر کیفیت Golden Dataset فارسی است.

QUESTION، ANSWER و EVIDENCE را فقط بر اساس معیارهای زیر ارزیابی کن:

1. سؤال باید فارسی طبیعی، روشن و مستقل باشد.
2. سؤال نباید ضمیر یا مرجع مبهم داشته باشد.
3. سؤال نباید پاسخ را در خودش تکرار کند.
4. پاسخ باید مستقیماً به سؤال جواب بدهد.
5. پاسخ نباید عبارت ناقص، عنوان، سؤال یا قطعه بی‌معنا باشد.
6. پاسخ باید کاملاً توسط EVIDENCE پشتیبانی شود.
7. برای فهم سؤال و پاسخ نباید دانش بیرونی لازم باشد.
8. طبیعی‌بودن مهم است؛ صرف عبور از قواعد فنی کافی نیست.
9. اگر پاسخ از نظر لفظی در evidence وجود دارد ولی سؤال را جواب نمی‌دهد، PASS نده.
10. اگر با اصلاح محدود سؤال قابل حل است، REVIEW بده.
11. اگر evidence برای ساخت سؤال معتبر کافی نیست، REJECT بده.

خروجی فقط یک JSON معتبر باشد:

{{
  "decision": "PASS|REVIEW|REJECT",
  "question_natural": true,
  "question_self_contained": true,
  "answer_directly_answers": true,
  "answer_complete": true,
  "evidence_supports_answer": true,
  "no_ambiguous_reference": true,
  "no_question_answer_leakage": true,
  "reasons": ["..."]
}}

QUESTION:
{record["question"]}

ANSWER:
{record["expected_answer"]}

EVIDENCE:
{record["source_evidence"]}
""".strip()

def valid_semantic_judgment(payload):
    boolean_fields = (
        "question_natural",
        "question_self_contained",
        "answer_directly_answers",
        "answer_complete",
        "evidence_supports_answer",
        "no_ambiguous_reference",
        "no_question_answer_leakage",
    )

    return (
        isinstance(payload, dict)
        and payload.get("decision")
        in {"PASS", "REVIEW", "REJECT"}
        and all(
            isinstance(payload.get(field), bool)
            for field in boolean_fields
        )
        and isinstance(
            payload.get("reasons"),
            list,
        )
    )

def semantic_judge_cache_key(record):
    material = {
        "judge_version": (
            SEMANTIC_JUDGE_VERSION
        ),
        "model": chat_model,
        "candidate_id": record[
            "candidate_id"
        ],
        "question": canon(
            record["question"]
        ),
        "answer": canon(
            record["expected_answer"]
        ),
        "evidence": canon(
            record["source_evidence"]
        ),
    }

    return hashlib.sha256(
        json.dumps(
            material,
            ensure_ascii=False,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()

def call_semantic_judge(prompt):
    response = chat.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Return only one valid JSON object. "
                    "Do not include markdown."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0,
        max_tokens=1536,
    )

    raw_response = (
        response.choices[0].message.content
        or ""
    ).strip()

    payload = parse_json_object(
        raw_response
    )

    usage = getattr(
        response,
        "usage",
        None,
    )

    observed_cost = None

    if usage is not None:
        observed_cost = getattr(
            usage,
            "cost",
            None,
        )

        if observed_cost is None:
            usage_data = (
                usage.model_dump()
                if hasattr(usage, "model_dump")
                else {}
            )
            observed_cost = usage_data.get(
                "cost"
            )

    return {
        "payload": payload,
        "raw_response": raw_response,
        "finish_reason": (
            response.choices[0].finish_reason
        ),
        "observed_cost": observed_cost,
    }

semantic_judge_results = []
semantic_judge_cache_reuse_count = 0
semantic_judge_live_request_count = 0

print("=== FULL SEMANTIC-JUDGE EVALUATION ===")
print(
    "Records:",
    len(integrity_repaired_experiment),
)

for position, record in enumerate(
    integrity_repaired_experiment,
    1,
):
    cache_key = semantic_judge_cache_key(
        record
    )

    cache_path = (
        semantic_judge_cache_directory
        / f"{cache_key}.json"
    )

    cached_result = None

    if cache_path.is_file():
        candidate_cache = read_json(
            cache_path
        )

        if valid_semantic_judgment(
            candidate_cache.get("payload")
        ):
            cached_result = candidate_cache

    if cached_result is not None:
        result = {
            **cached_result,
            "cache_reused": True,
        }
        semantic_judge_cache_reuse_count += 1

    else:
        result = None
        last_error = None

        for attempt in range(1, 4):
            try:
                response_record = (
                    call_semantic_judge(
                        semantic_judge_prompt(
                            record
                        )
                    )
                )

                semantic_judge_live_request_count += 1

                payload = response_record[
                    "payload"
                ]

                if not valid_semantic_judgment(
                    payload
                ):
                    last_error = (
                        "invalid_judgment_payload"
                    )
                    continue

                # PASS is valid only if every criterion
                # is explicitly true.
                criterion_fields = (
                    "question_natural",
                    "question_self_contained",
                    "answer_directly_answers",
                    "answer_complete",
                    "evidence_supports_answer",
                    "no_ambiguous_reference",
                    "no_question_answer_leakage",
                )

                all_criteria_pass = all(
                    payload[field]
                    for field in criterion_fields
                )

                normalized_decision = payload[
                    "decision"
                ]

                if (
                    normalized_decision == "PASS"
                    and not all_criteria_pass
                ):
                    normalized_decision = "REVIEW"

                payload["decision"] = (
                    normalized_decision
                )

                result = {
                    "judge_version": (
                        SEMANTIC_JUDGE_VERSION
                    ),
                    "candidate_id": record[
                        "candidate_id"
                    ],
                    "source_seed_id": record[
                        "source_seed_id"
                    ],
                    "attempt": attempt,
                    "model": chat_model,
                    "payload": payload,
                    "raw_response": (
                        response_record[
                            "raw_response"
                        ]
                    ),
                    "finish_reason": (
                        response_record[
                            "finish_reason"
                        ]
                    ),
                    "observed_cost": (
                        response_record[
                            "observed_cost"
                        ]
                    ),
                    "request_error": None,
                    "created_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "cache_reused": False,
                }

                write_json_atomic(
                    cache_path,
                    result,
                )
                break

            except Exception as error:
                semantic_judge_live_request_count += 1
                last_error = (
                    f"{type(error).__name__}: "
                    f"{error}"
                )

        if result is None:
            result = {
                "judge_version": (
                    SEMANTIC_JUDGE_VERSION
                ),
                "candidate_id": record[
                    "candidate_id"
                ],
                "source_seed_id": record[
                    "source_seed_id"
                ],
                "attempt": 3,
                "model": chat_model,
                "payload": None,
                "raw_response": "",
                "finish_reason": None,
                "observed_cost": None,
                "request_error": last_error,
                "created_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "cache_reused": False,
            }

    semantic_judge_results.append(
        result
    )

    decision = (
        result["payload"]["decision"]
        if valid_semantic_judgment(
            result.get("payload")
        )
        else "FAILED"
    )

    print(
        f"Record {position}/"
        f"{len(integrity_repaired_experiment)} | "
        f"seed={record['source_seed_id']} | "
        f"{decision}"
    )

assert len(semantic_judge_results) == 49

semantic_judge_pass_rows = [
    row
    for row in semantic_judge_results
    if (
        valid_semantic_judgment(
            row.get("payload")
        )
        and row["payload"]["decision"]
        == "PASS"
    )
]

semantic_judge_review_rows = [
    row
    for row in semantic_judge_results
    if (
        valid_semantic_judgment(
            row.get("payload")
        )
        and row["payload"]["decision"]
        == "REVIEW"
    )
]

semantic_judge_reject_rows = [
    row
    for row in semantic_judge_results
    if (
        valid_semantic_judgment(
            row.get("payload")
        )
        and row["payload"]["decision"]
        == "REJECT"
    )
]

semantic_judge_failed_rows = [
    row
    for row in semantic_judge_results
    if not valid_semantic_judgment(
        row.get("payload")
    )
]

observed_costs = [
    float(row["observed_cost"])
    for row in semantic_judge_results
    if row.get("observed_cost") is not None
]

semantic_judge_report = {
    "judge_version": (
        SEMANTIC_JUDGE_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "active_v1_modified": False,
    "experiment_path": str(
        integrity_repaired_experiment_path
    ),
    "experiment_sha256": sha256_file(
        integrity_repaired_experiment_path
    ),
    "record_count": len(
        semantic_judge_results
    ),
    "pass_count": len(
        semantic_judge_pass_rows
    ),
    "review_count": len(
        semantic_judge_review_rows
    ),
    "reject_count": len(
        semantic_judge_reject_rows
    ),
    "failed_count": len(
        semantic_judge_failed_rows
    ),
    "cache_reuse_count": (
        semantic_judge_cache_reuse_count
    ),
    "live_request_count": (
        semantic_judge_live_request_count
    ),
    "total_observed_cost": (
        float(sum(observed_costs))
        if observed_costs
        else None
    ),
    "results": semantic_judge_results,
}

semantic_judge_report_path = (
    golden_dir
    / (
        f"{INTEGRITY_REPAIRED_EXPERIMENT_NAME}"
        "_semantic_judge.json"
    )
)

write_json_atomic(
    semantic_judge_report_path,
    semantic_judge_report,
)

print("\nSemantic PASS:", len(
    semantic_judge_pass_rows
))
print("Semantic REVIEW:", len(
    semantic_judge_review_rows
))
print("Semantic REJECT:", len(
    semantic_judge_reject_rows
))
print("Request/parse failures:", len(
    semantic_judge_failed_rows
))
print(
    "Cached results reused:",
    semantic_judge_cache_reuse_count,
)
print(
    "Live requests:",
    semantic_judge_live_request_count,
)
print(
    "Total observed cost:",
    (
        f"{sum(observed_costs):.6f}"
        if observed_costs
        else "N/A"
    ),
)
print(
    "\nReport:",
    semantic_judge_report_path,
)
print(
    "\nFULL SEMANTIC-JUDGE "
    "EVALUATION: COMPLETE"
)

=== FULL SEMANTIC-JUDGE EVALUATION ===
Records: 49
Record 1/49 | seed=1 | PASS
Record 2/49 | seed=2 | PASS
Record 3/49 | seed=2 | PASS
Record 4/49 | seed=3 | PASS
Record 5/49 | seed=3 | PASS
Record 6/49 | seed=3 | PASS
Record 7/49 | seed=4 | PASS
Record 8/49 | seed=4 | PASS
Record 9/49 | seed=4 | PASS
Record 10/49 | seed=5 | PASS
Record 11/49 | seed=6 | REVIEW
Record 12/49 | seed=6 | REVIEW
Record 13/49 | seed=7 | REVIEW
Record 14/49 | seed=8 | PASS
Record 15/49 | seed=8 | PASS
Record 16/49 | seed=8 | PASS
Record 17/49 | seed=9 | PASS
Record 18/49 | seed=9 | PASS
Record 19/49 | seed=10 | PASS
Record 20/49 | seed=11 | PASS
Record 21/49 | seed=12 | PASS
Record 22/49 | seed=12 | PASS
Record 23/49 | seed=12 | PASS
Record 24/49 | seed=13 | REJECT
Record 25/49 | seed=13 | PASS
Record 26/49 | seed=13 | REVIEW
Record 27/49 | seed=14 | REVIEW
Record 28/49 | seed=15 | PASS
Record 29/49 | seed=16 | PASS
Record 30/49 | seed=16 | PASS
Record 31/49 | seed=16 | PASS
Record 32/49 | seed=17 | PASS
Reco

In [34]:
# Code Cell 32
# NB12G — Semantic-judge findings packet

assert "semantic_judge_results" in globals(), (
    "Run Code Cell 31 first."
)
assert len(semantic_judge_results) == 49

SEMANTIC_FINDINGS_PACKET_VERSION = (
    "nb12g_semantic_findings_packet_v1"
)

repaired_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in integrity_repaired_experiment
}

semantic_finding_rows = []

for judge_result in semantic_judge_results:
    payload = judge_result.get("payload")

    if not valid_semantic_judgment(payload):
        decision = "FAILED"
        reasons = [
            judge_result.get(
                "request_error",
                "invalid_judgment_payload",
            )
        ]
    else:
        decision = payload["decision"]
        reasons = payload["reasons"]

    if decision == "PASS":
        continue

    candidate_id = judge_result[
        "candidate_id"
    ]

    record = repaired_record_by_candidate_id[
        candidate_id
    ]

    semantic_finding_rows.append({
        "candidate_id": candidate_id,
        "source_seed_id": record[
            "source_seed_id"
        ],
        "decision": decision,
        "reasons": reasons,
        "criteria": (
            payload
            if isinstance(payload, dict)
            else None
        ),
        "question": record["question"],
        "expected_answer": record[
            "expected_answer"
        ],
        "source_evidence": record[
            "source_evidence"
        ],
        "question_intent": record[
            "question_intent"
        ],
        "retrieval_labels": record[
            "retrieval_labels"
        ],
        "curation": record.get(
            "curation"
        ),
        "manual_decision": None,
        "replacement_question": None,
        "replacement_answer": None,
        "manual_notes": None,
    })

semantic_finding_rows.sort(
    key=lambda row: (
        0 if row["decision"] == "REJECT" else 1,
        row["source_seed_id"],
        row["candidate_id"],
    )
)

assert len(semantic_finding_rows) == 11

semantic_finding_counts = Counter(
    row["decision"]
    for row in semantic_finding_rows
)

semantic_findings_packet = {
    "packet_version": (
        SEMANTIC_FINDINGS_PACKET_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "active_v1_modified": False,
    "experiment_path": str(
        integrity_repaired_experiment_path
    ),
    "experiment_sha256": sha256_file(
        integrity_repaired_experiment_path
    ),
    "finding_count": len(
        semantic_finding_rows
    ),
    "decision_counts": dict(
        semantic_finding_counts
    ),
    "records": semantic_finding_rows,
}

semantic_findings_packet_path = (
    golden_dir
    / (
        f"{INTEGRITY_REPAIRED_EXPERIMENT_NAME}"
        "_semantic_findings_packet.json"
    )
)

write_json_atomic(
    semantic_findings_packet_path,
    semantic_findings_packet,
)

print("=== SEMANTIC-JUDGE FINDINGS PACKET ===")
print(
    "Findings:",
    len(semantic_finding_rows),
)
print(
    "REVIEW:",
    semantic_finding_counts.get(
        "REVIEW",
        0,
    ),
)
print(
    "REJECT:",
    semantic_finding_counts.get(
        "REJECT",
        0,
    ),
)
print(
    "FAILED:",
    semantic_finding_counts.get(
        "FAILED",
        0,
    ),
)

for position, row in enumerate(
    semantic_finding_rows,
    1,
):
    print("\n" + "=" * 80)
    print(
        f"POSITION {position} | "
        f"SEED {row['source_seed_id']} | "
        f"{row['decision']}"
    )
    print(
        "Candidate:",
        row["candidate_id"],
    )

    print("\nQuestion:")
    print(row["question"])

    print("\nAnswer:")
    print(row["expected_answer"])

    print("\nEvidence:")
    print(row["source_evidence"])

    print("\nJudge reasons:")
    for reason in row["reasons"]:
        print("-", reason)

    if row["criteria"]:
        print("\nFailed criteria:")

        for criterion_name in (
            "question_natural",
            "question_self_contained",
            "answer_directly_answers",
            "answer_complete",
            "evidence_supports_answer",
            "no_ambiguous_reference",
            "no_question_answer_leakage",
        ):
            if (
                row["criteria"].get(
                    criterion_name
                )
                is False
            ):
                print(
                    "-",
                    criterion_name,
                )

print(
    "\nPacket:",
    semantic_findings_packet_path,
)
print(
    "\nSEMANTIC-JUDGE FINDINGS "
    "PACKET: PASS"
)

=== SEMANTIC-JUDGE FINDINGS PACKET ===
Findings: 11
REVIEW: 9
REJECT: 2
FAILED: 0

POSITION 1 | SEED 13 | REJECT
Candidate: 506d6fc2-0ae5-5d70-a603-3dcfc703685f

Question:
برای ارزیابی وضعیت مالی کسب‌وکار در پایان سال چه چیزی باید بررسی شود؟

Answer:
آیا بعد از همه هزینه‌ها سود دیگری هم باقی مانده یا کسب‌وکار علاوه بر این، بدهکار شده است؟

Evidence:
بعد در پایان سال مالی بررسی کنید: مثلاً طی سال 36 میلیون تومان حقوق برای خودم برداشت کرده‌ام؛ آیا بعد از همه هزینه‌ها سود دیگری هم باقی مانده یا کسب‌وکار علاوه بر این، بدهکار شده است؟

Judge reasons:
- پاسخ به صورت سوال بیان شده و مستقیماً سؤال را تکرار می‌کند (عدم پاسخ مستقیم).
- پاسخ ناقص و غیر کامل است، چرا که فقط یک سوال مطرح می‌کند و جوابی ارائه نمی‌دهد.
- پاسخ دارای تکرار سؤال (question answer leakage) است.
- معیار 6 (پشتیبانی از پاسخ) برقرار نیست، زیرا پاسخ خود به عنوان یک سوال است و اثباتی از پاسخ نیست.

Failed criteria:
- answer_directly_answers
- answer_complete
- evidence_supports_answer
- no_question_answer_leakage

POSITION 2 |

In [40]:
# Code Cell 33
# NB12G — Apply adjudicated semantic repairs

from copy import deepcopy

assert "integrity_repaired_experiment" in globals(), (
    "Run Code Cell 29 first."
)
assert len(integrity_repaired_experiment) == 49

SEMANTIC_REPAIRED_EXPERIMENT_NAME = (
    "golden_dataset_quote_grounded_semantic_repaired_experiment"
)
SEMANTIC_REPAIR_VERSION = "nb12g_semantic_repair_v3"

manual_semantic_decisions = {
    "506d6fc2-0ae5-5d70-a603-3dcfc703685f": {
        "decision": "REPAIR",
        "question": (
            "در پایان سال مالی، برای ارزیابی وضعیت "
            "کسب‌وکار چه پرسشی باید مطرح شود؟"
        ),
        "answer": (
            "آیا بعد از همه هزینه‌ها سود دیگری هم باقی "
            "مانده یا کسب‌وکار علاوه بر این، بدهکار شده است؟"
        ),
        "note": (
            "The repaired question asks for the exact end-of-year "
            "evaluation question stated in the evidence."
        ),
    },
    "fe4f3c5c-1800-5b08-a927-241011f20de7": {
        "decision": "REMOVE",
        "note": (
            "The source is an ambiguous planning utterance and "
            "does not support a standalone answerable QA record."
        ),
    },
    "1e75fd94-6b10-54ea-a1d5-c3875033c0be": {
        "decision": "REPAIR",
        "question": (
            "گزاره‌های علمی و نمونه‌های عملی در طول "
            "رشد فرد چه تغییری می‌کنند؟"
        ),
        "answer": "توسعه یافته سطح قبلی خود است",
        "note": "Question aligned to the exact supported claim.",
    },
    "232e72a6-9624-5f2b-bbd2-59d11f4df617": {
        "decision": "REPAIR",
        "question": (
            "دولت‌ها برای آموزش سواد مالی چه کرده‌اند؟"
        ),
        "answer": (
            "سیاست‌گذاری کرده‌اند تا به سطح مناسبی از نگرش، "
            "دانش، رفتار و توانایی‌های مالی برسند."
        ),
        "note": "Expanded with the directly supported purpose.",
    },
    "899d2be4-9f5b-57f4-91a6-ce55b78aaf97": {
        "decision": "REPAIR",
        "question": (
            "وضعیت تجهیزات اداری در محیط کار چگونه "
            "ارزیابی شده است؟"
        ),
        "answer": (
            "تجهیزات اداری (میز، صندلی و لوازم و ...) "
            "در حد مناسبی است."
        ),
        "note": "Question aligned to the stated condition.",
    },
    "872668c5-952c-500b-bfd8-141013b45ffb": {
        "decision": "REMOVE",
        "note": (
            "The evidence does not identify the two items being "
            "mixed; a standalone question would require outside context."
        ),
    },
    "e751afbd-ad02-5b9e-8006-671e97814a43": {
        "decision": "REPAIR",
        "question": (
            "کار داوطلبانه چه فرصت‌هایی برای فرد فراهم می‌کند؟"
        ),
        "answer": (
            "من ممکنه مدتی کار کنم تا کار رو یاد بگیرم، رزومه‌م "
            "رو پر کنم، ارتباطاتم رو توسعه بدم و عوامل غیرمالی "
            "کار رو بفهمم"
        ),
        "note": "Question aligned to the evidence's stated benefits.",
    },
    "747f84cf-0990-5a0c-9078-9bd7703bf90b": {
        "decision": "REPAIR",
        "question": (
            "دانستن اصطلاحات قراردادهای بیمه چه کمکی "
            "به بیمه‌گذار می‌کند؟"
        ),
        "answer": (
            "قرارداد های بیمه پر از این اصطلاحات و شما باید "
            "بدونی چی کار می‌کنی چون حتما مردم تجربه شرکت های "
            "بیمه رو دارن اونها از همه ظرفیت قرارداد استفاده "
            "می‌کنن تا کمتر به شما پول پرداخت بکنن پس شما باید "
            "از همه ظرفیت این قرارداد استفاده بکنین که بیشترین "
            "خسارت خودت رو جبران بکنین"
        ),
        "note": "Answer retains the complete evidence-grounded rationale.",
    },
    "d70519a5-a424-538a-951e-7f3765b392f9": {
        "decision": "REPAIR",
        "question": (
            "در مثالِ حد ضرر برای سرمایه یک‌میلیون‌تومانی، "
            "حد ضرر چند درصد است؟"
        ),
        "answer": "بیست‌درصدی",
        "note": "Removed the ambiguous reference to an unstated example.",
    },
    "f969141a-6446-5a54-9045-0cd1f60ee8e4": {
        "decision": "REPAIR",
        "question": (
            "پس از ارزیابی جایگاه فعلی در پلکان "
            "توانمندی مالی، چه کاری باید انجام شود؟"
        ),
        "answer": "چند جمله‌ای درباره وضعیت خود بنویسید.",
        "note": "Question simplified without changing its supported meaning.",
    },
    "69f2ff2d-5f82-5999-8740-43d82cdb4f74": {
        "decision": "REPAIR",
        "question": (
            "تجربه‌های شرکت‌کنندگان درباره اثر نوشتن "
            "هزینه‌ها بر چه چیزی جذاب و شنیدنی بود؟"
        ),
        "answer": (
            "اثرگذاری نوشتن هزینه‌ها بر پیشگیری از ولخرجی و "
            "مدیریت بهتر هزینه‌ها، بدون کاهش کیفیت زندگی"
        ),
        "note": "Removed the ambiguous reference «امری».",
    },
}

assert len(manual_semantic_decisions) == 11

records_by_candidate_id = {
    record["candidate_id"]: record
    for record in integrity_repaired_experiment
}

assert set(manual_semantic_decisions) <= set(
    records_by_candidate_id
), "A manual decision references an unknown candidate."

grounding_failures = []

for candidate_id, decision in manual_semantic_decisions.items():
    if decision["decision"] != "REPAIR":
        continue

    source_evidence = records_by_candidate_id[
        candidate_id
    ]["source_evidence"]

    if canon(decision["answer"]) not in canon(source_evidence):
        grounding_failures.append({
            "candidate_id": candidate_id,
            "answer": decision["answer"],
        })

assert not grounding_failures, (
    "Ungrounded replacement answers: "
    + json.dumps(
        grounding_failures,
        ensure_ascii=False,
        indent=2,
    )
)

semantic_repaired_experiment = []
semantic_repair_log = []

for record in integrity_repaired_experiment:
    source_candidate_id = record["candidate_id"]
    decision = manual_semantic_decisions.get(
        source_candidate_id
    )

    if decision is not None and decision["decision"] == "REMOVE":
        semantic_repair_log.append({
            "source_candidate_id": source_candidate_id,
            "source_seed_id": record["source_seed_id"],
            "decision": "REMOVE",
            "reason": decision["note"],
        })
        continue

    output_record = deepcopy(record)

    # Critical: every output record belongs to the same
    # semantic-repaired experiment version.
    output_record["dataset_version"] = (
        SEMANTIC_REPAIRED_EXPERIMENT_NAME
    )

    if decision is None:
        output_record["manual_decision"] = "RETAIN"
        output_record["manual_notes"] = (
            "Retained unchanged after semantic findings review."
        )
        semantic_repaired_experiment.append(output_record)
        continue

    output_record["candidate_id"] = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            (
                f"{SEMANTIC_REPAIR_VERSION}|"
                f"{source_candidate_id}|"
                f"{canon(decision['question'])}|"
                f"{canon(decision['answer'])}"
            ),
        )
    )
    output_record["question"] = decision["question"]
    output_record["expected_answer"] = decision["answer"]

    output_record["curation"] = {
        "curation_version": SEMANTIC_REPAIR_VERSION,
        "decision": "SEMANTIC_REPAIR",
        "source_candidate_id": source_candidate_id,
        "supersedes_candidate_id": source_candidate_id,
        "reason": decision["note"],
        "active_dataset_record": False,
    }
    output_record["manual_decision"] = "REPAIR"
    output_record["replacement_question"] = decision["question"]
    output_record["replacement_answer"] = decision["answer"]
    output_record["manual_notes"] = decision["note"]

    semantic_repaired_experiment.append(output_record)

    semantic_repair_log.append({
        "source_candidate_id": source_candidate_id,
        "candidate_id": output_record["candidate_id"],
        "source_seed_id": record["source_seed_id"],
        "decision": "REPAIR",
        "question": output_record["question"],
        "answer": output_record["expected_answer"],
        "reason": decision["note"],
    })

assert len(semantic_repaired_experiment) == 47

repair_count = sum(
    row["decision"] == "REPAIR"
    for row in semantic_repair_log
)
remove_count = sum(
    row["decision"] == "REMOVE"
    for row in semantic_repair_log
)

assert repair_count == 9
assert remove_count == 2

assert {
    record["dataset_version"]
    for record in semantic_repaired_experiment
} == {SEMANTIC_REPAIRED_EXPERIMENT_NAME}

semantic_candidate_ids = [
    record["candidate_id"]
    for record in semantic_repaired_experiment
]
semantic_question_keys = [
    canon(record["question"])
    for record in semantic_repaired_experiment
]

assert len(semantic_candidate_ids) == len(
    set(semantic_candidate_ids)
), "Duplicate candidate IDs after semantic repair."

assert len(semantic_question_keys) == len(
    set(semantic_question_keys)
), "Duplicate normalized questions after semantic repair."

semantic_repaired_experiment_path = (
    golden_dir
    / f"{SEMANTIC_REPAIRED_EXPERIMENT_NAME}.jsonl"
)

write_jsonl_atomic(
    semantic_repaired_experiment_path,
    semantic_repaired_experiment,
)

semantic_repair_report = {
    "repair_version": SEMANTIC_REPAIR_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "READY_FOR_POST_REPAIR_AUDIT",
    "active_dataset_created": False,
    "active_v1_modified": False,
    "input_experiment_path": str(
        integrity_repaired_experiment_path
    ),
    "input_experiment_sha256": sha256_file(
        integrity_repaired_experiment_path
    ),
    "output_experiment_path": str(
        semantic_repaired_experiment_path
    ),
    "output_experiment_sha256": sha256_file(
        semantic_repaired_experiment_path
    ),
    "input_record_count": 49,
    "output_record_count": 47,
    "repair_count": repair_count,
    "remove_count": remove_count,
    "log": semantic_repair_log,
}

semantic_repair_report_path = (
    golden_dir
    / (
        f"{SEMANTIC_REPAIRED_EXPERIMENT_NAME}"
        "_repair_report.json"
    )
)

write_json_atomic(
    semantic_repair_report_path,
    semantic_repair_report,
)

print("=== SEMANTIC REPAIR APPLICATION ===")
print("Input records:", 49)
print("Repaired records:", repair_count)
print("Removed records:", remove_count)
print("Output records:", len(semantic_repaired_experiment))
print("Output dataset version:", (
    SEMANTIC_REPAIRED_EXPERIMENT_NAME
))
print("Active Golden Dataset created:", False)
print("Golden Dataset v1 modified:", False)
print("\nExperiment:", semantic_repaired_experiment_path)
print("Report:", semantic_repair_report_path)
print("\nSEMANTIC REPAIR APPLICATION: COMPLETE")

=== SEMANTIC REPAIR APPLICATION ===
Input records: 49
Repaired records: 9
Removed records: 2
Output records: 47
Output dataset version: golden_dataset_quote_grounded_semantic_repaired_experiment
Active Golden Dataset created: False
Golden Dataset v1 modified: False

Experiment: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_semantic_repaired_experiment.jsonl
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_semantic_repaired_experiment_repair_report.json

SEMANTIC REPAIR APPLICATION: COMPLETE


In [41]:
# Code Cell 34
# NB12G — Post-semantic-repair integrity audit

assert "semantic_repaired_experiment" in globals(), (
    "Run Code Cell 33 first."
)
assert len(semantic_repaired_experiment) == 47

POST_SEMANTIC_AUDIT_VERSION = (
    "nb12g_post_semantic_repair_integrity_audit_v1"
)

ALLOWED_FALLBACK_SEED_IDS = {16}
ALLOWED_FALLBACK_LABEL_BASIS = {
    "EXACT_REVIEWED_EVIDENCE_FALLBACK",
}

post_semantic_integrity_rows = []

for record in semantic_repaired_experiment:
    errors = []
    accepted_exceptions = []

    candidate_id = record.get("candidate_id")
    seed_id = record.get("source_seed_id")

    missing_fields = sorted(
        REQUIRED_RECORD_FIELDS - set(record)
    )

    if missing_fields:
        errors.append({
            "code": "missing_required_fields",
            "fields": missing_fields,
        })

    try:
        uuid.UUID(str(candidate_id))
    except (
        ValueError,
        TypeError,
        AttributeError,
    ):
        errors.append({
            "code": "invalid_candidate_uuid",
        })

    if record.get("dataset_version") != (
        SEMANTIC_REPAIRED_EXPERIMENT_NAME
    ):
        errors.append({
            "code": "invalid_dataset_version",
        })

    if seed_id not in seeds:
        errors.append({
            "code": "unknown_source_seed_id",
        })

    if record.get("evaluation_split") not in {
        "train",
        "dev",
        "test",
    }:
        errors.append({
            "code": "invalid_evaluation_split",
        })

    question = str(record.get("question", "")).strip()
    expected_answer = str(
        record.get("expected_answer", "")
    ).strip()
    source_evidence = str(
        record.get("source_evidence", "")
    ).strip()
    evidence_quote = str(
        record.get("evidence_quote", "")
    ).strip()

    if not question.endswith("؟"):
        errors.append({
            "code": "invalid_question_ending",
        })

    if canon(expected_answer) not in canon(source_evidence):
        errors.append({
            "code": "answer_not_in_source_evidence",
        })

    if canon(source_evidence) != canon(evidence_quote):
        errors.append({
            "code": "evidence_quote_mismatch",
        })

    labels = record.get("retrieval_labels", {})
    candidate_name = labels.get("candidate")
    label_basis = labels.get("label_basis")

    positive_chunk_ids = labels.get(
        "positive_chunk_ids",
        [],
    )
    positive_parent_chunk_ids = labels.get(
        "positive_parent_chunk_ids",
        [],
    )

    if candidate_name not in {
        "parent_child_children",
        "structure_aware_medium",
    }:
        errors.append({
            "code": "invalid_retrieval_candidate",
        })

    if not positive_chunk_ids:
        errors.append({
            "code": "empty_positive_chunk_ids",
        })

    missing_positive_chunk_ids = [
        chunk_id
        for chunk_id in positive_chunk_ids
        if chunk_id not in all_retrieval_chunk_by_id
    ]

    if missing_positive_chunk_ids:
        errors.append({
            "code": "missing_positive_chunks",
            "chunk_ids": missing_positive_chunk_ids,
        })

    available_positive_chunks = [
        all_retrieval_chunk_by_id[chunk_id]
        for chunk_id in positive_chunk_ids
        if chunk_id in all_retrieval_chunk_by_id
    ]

    evidence_found_in_positive_chunk = any(
        canon(source_evidence) in canon(chunk["content"])
        for chunk in available_positive_chunks
    )

    if not evidence_found_in_positive_chunk:
        errors.append({
            "code": "evidence_not_in_positive_chunk",
        })

    missing_parent_ids = [
        parent_id
        for parent_id in positive_parent_chunk_ids
        if parent_id not in all_parent_chunk_ids
    ]

    if missing_parent_ids:
        errors.append({
            "code": "missing_positive_parents",
            "parent_ids": missing_parent_ids,
        })

    record_document_ids = set(
        record.get("document_ids", [])
    )

    positive_document_ids = set()

    for chunk in available_positive_chunks:
        positive_document_ids.update(
            chunk_identifiers(chunk)
        )

    document_overlap = sorted(
        record_document_ids & positive_document_ids
    )

    if not document_overlap:
        fallback_is_allowed = (
            seed_id in ALLOWED_FALLBACK_SEED_IDS
            and label_basis in ALLOWED_FALLBACK_LABEL_BASIS
            and evidence_found_in_positive_chunk
        )

        if fallback_is_allowed:
            accepted_exceptions.append({
                "code": (
                    "accepted_exact_reviewed_evidence_fallback"
                ),
                "label_basis": label_basis,
            })
        else:
            errors.append({
                "code": (
                    "unapproved_missing_document_overlap"
                ),
                "label_basis": label_basis,
            })

    post_semantic_integrity_rows.append({
        "candidate_id": candidate_id,
        "source_seed_id": seed_id,
        "evaluation_split": record.get(
            "evaluation_split"
        ),
        "errors": errors,
        "accepted_exceptions": accepted_exceptions,
        "document_overlap": document_overlap,
        "status": (
            "FAIL"
            if errors
            else "PASS_WITH_ACCEPTED_EXCEPTION"
            if accepted_exceptions
            else "PASS"
        ),
    })

post_semantic_failure_rows = [
    row
    for row in post_semantic_integrity_rows
    if row["errors"]
]

post_semantic_exception_rows = [
    row
    for row in post_semantic_integrity_rows
    if row["accepted_exceptions"]
]

candidate_ids = [
    record["candidate_id"]
    for record in semantic_repaired_experiment
]

question_keys = [
    canon(record["question"])
    for record in semantic_repaired_experiment
]

assert len(candidate_ids) == len(set(candidate_ids)), (
    "Duplicate candidate IDs remain."
)
assert len(question_keys) == len(set(question_keys)), (
    "Duplicate normalized questions remain."
)

splits_by_seed = defaultdict(set)

for record in semantic_repaired_experiment:
    splits_by_seed[record["source_seed_id"]].add(
        record["evaluation_split"]
    )

seed_split_failures = {
    seed_id: sorted(split_values)
    for seed_id, split_values in splits_by_seed.items()
    if len(split_values) != 1
}

assert not seed_split_failures, (
    "Seed split inconsistency remains: "
    f"{seed_split_failures}"
)

covered_seed_ids = sorted(splits_by_seed)

assert covered_seed_ids == list(range(1, 29)), (
    "One or more source seeds are no longer covered."
)

evidence_splits = defaultdict(set)

for record in semantic_repaired_experiment:
    evidence_hash = hashlib.sha256(
        canon(record["source_evidence"]).encode("utf-8")
    ).hexdigest()

    evidence_splits[evidence_hash].add(
        record["evaluation_split"]
    )

cross_split_evidence_leakage = [
    {
        "evidence_hash": evidence_hash,
        "splits": sorted(split_values),
    }
    for evidence_hash, split_values in evidence_splits.items()
    if len(split_values) > 1
]

assert not cross_split_evidence_leakage, (
    "Cross-split exact-evidence leakage remains."
)

assert len(post_semantic_exception_rows) == 3
assert {
    row["source_seed_id"]
    for row in post_semantic_exception_rows
} == {16}

assert not post_semantic_failure_rows, (
    "Post-semantic-repair failures remain: "
    + json.dumps(
        post_semantic_failure_rows,
        ensure_ascii=False,
        indent=2,
    )
)

post_semantic_integrity_report = {
    "audit_version": POST_SEMANTIC_AUDIT_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "PASS_WITH_DOCUMENTED_SEED16_FALLBACK",
    "active_dataset_created": False,
    "active_v1_modified": False,
    "experiment_path": str(
        semantic_repaired_experiment_path
    ),
    "experiment_sha256": sha256_file(
        semantic_repaired_experiment_path
    ),
    "record_count": len(semantic_repaired_experiment),
    "covered_seed_count": len(covered_seed_ids),
    "covered_seed_ids": covered_seed_ids,
    "candidate_ids_unique": True,
    "normalized_questions_unique": True,
    "seed_split_consistency": True,
    "cross_split_evidence_leakage_count": 0,
    "integrity_failure_count": 0,
    "accepted_exception_count": len(
        post_semantic_exception_rows
    ),
    "accepted_exception_seed_ids": [16],
    "rows": post_semantic_integrity_rows,
}

post_semantic_integrity_report_path = (
    golden_dir
    / (
        f"{SEMANTIC_REPAIRED_EXPERIMENT_NAME}"
        "_final_integrity_audit.json"
    )
)

write_json_atomic(
    post_semantic_integrity_report_path,
    post_semantic_integrity_report,
)

print("=== POST-SEMANTIC-REPAIR INTEGRITY AUDIT ===")
print("Records:", len(semantic_repaired_experiment))
print("Covered seeds:", len(covered_seed_ids))
print("Integrity failures:", len(post_semantic_failure_rows))
print("Accepted exceptions:", len(
    post_semantic_exception_rows
))
print("Accepted-exception seeds:", sorted({
    row["source_seed_id"]
    for row in post_semantic_exception_rows
}))
print("Candidate IDs unique:", True)
print("Normalized questions unique:", True)
print("Seed split failures:", len(seed_split_failures))
print(
    "Cross-split exact-evidence leakage:",
    len(cross_split_evidence_leakage),
)
print("Active Golden Dataset created:", False)
print("Golden Dataset v1 modified:", False)
print("\nReport:", post_semantic_integrity_report_path)
print("\nPOST-SEMANTIC-REPAIR INTEGRITY AUDIT: PASS")

=== POST-SEMANTIC-REPAIR INTEGRITY AUDIT ===
Records: 47
Covered seeds: 28
Integrity failures: 0
Accepted exceptions: 3
Accepted-exception seeds: [16]
Candidate IDs unique: True
Normalized questions unique: True
Seed split failures: 0
Cross-split exact-evidence leakage: 0
Active Golden Dataset created: False
Golden Dataset v1 modified: False

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_semantic_repaired_experiment_final_integrity_audit.json

POST-SEMANTIC-REPAIR INTEGRITY AUDIT: PASS


In [42]:
# Code Cell 35
# NB12G — Final semantic-judge evaluation after repairs

assert "semantic_repaired_experiment" in globals(), (
    "Run Code Cell 33 first."
)
assert "chat" in globals(), "Run Code Cell 08 first."
assert "chat_model" in globals()
assert len(semantic_repaired_experiment) == 47

FINAL_SEMANTIC_JUDGE_VERSION = (
    "nb12g_final_semantic_judge_v1"
)

final_semantic_cache_directory = (
    golden_dir
    / "cache"
    / SEMANTIC_REPAIRED_EXPERIMENT_NAME
    / "semantic_judge"
    / FINAL_SEMANTIC_JUDGE_VERSION
)

final_semantic_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)

def final_semantic_judge_prompt(record):
    return f"""
نقش شما داور سخت‌گیر کیفیت Golden Dataset فارسی است.

فقط بر اساس QUESTION، ANSWER و EVIDENCE تصمیم بگیرید.

معیارها:

1. سؤال باید فارسی طبیعی، روشن و مستقل باشد.
2. سؤال نباید ضمیر یا مرجع مبهم داشته باشد.
3. سؤال نباید پاسخ را در خود تکرار کند.
4. پاسخ باید مستقیماً به سؤال جواب بدهد.
5. پاسخ باید کامل، معنادار و غیرسؤالی باشد؛
   مگر آنکه سؤال صریحاً درخواست نقل یک پرسش از evidence را داشته باشد.
6. پاسخ باید کاملاً توسط EVIDENCE پشتیبانی شود.
7. برای فهم سؤال و پاسخ نباید دانش بیرونی لازم باشد.
8. صرف وجود لفظ پاسخ در evidence برای PASS کافی نیست.
9. اگر با اصلاح محدود سؤال یا پاسخ قابل حل است REVIEW بده.
10. اگر evidence برای یک سؤال مستقل و معتبر کافی نیست REJECT بده.

خروجی فقط یک JSON معتبر باشد:

{{
  "decision": "PASS|REVIEW|REJECT",
  "question_natural": true,
  "question_self_contained": true,
  "answer_directly_answers": true,
  "answer_complete": true,
  "evidence_supports_answer": true,
  "no_ambiguous_reference": true,
  "no_question_answer_leakage": true,
  "reasons": ["..."]
}}

QUESTION:
{record["question"]}

ANSWER:
{record["expected_answer"]}

EVIDENCE:
{record["source_evidence"]}
""".strip()

def valid_final_semantic_judgment(payload):
    boolean_fields = (
        "question_natural",
        "question_self_contained",
        "answer_directly_answers",
        "answer_complete",
        "evidence_supports_answer",
        "no_ambiguous_reference",
        "no_question_answer_leakage",
    )

    return (
        isinstance(payload, dict)
        and payload.get("decision")
        in {"PASS", "REVIEW", "REJECT"}
        and all(
            isinstance(payload.get(field), bool)
            for field in boolean_fields
        )
        and isinstance(payload.get("reasons"), list)
    )

def final_semantic_cache_key(record):
    material = {
        "judge_version": FINAL_SEMANTIC_JUDGE_VERSION,
        "model": chat_model,
        "candidate_id": record["candidate_id"],
        "question": canon(record["question"]),
        "answer": canon(record["expected_answer"]),
        "evidence": canon(record["source_evidence"]),
    }

    return hashlib.sha256(
        json.dumps(
            material,
            ensure_ascii=False,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()

def call_final_semantic_judge(prompt):
    response = chat.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Return only one valid JSON object. "
                    "Do not include markdown."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0,
        max_tokens=1536,
    )

    raw_response = (
        response.choices[0].message.content
        or ""
    ).strip()

    payload = parse_json_object(raw_response)

    usage = getattr(response, "usage", None)
    observed_cost = None

    if usage is not None:
        observed_cost = getattr(usage, "cost", None)

        if observed_cost is None and hasattr(
            usage,
            "model_dump",
        ):
            observed_cost = usage.model_dump().get("cost")

    return {
        "payload": payload,
        "raw_response": raw_response,
        "finish_reason": response.choices[0].finish_reason,
        "observed_cost": observed_cost,
    }

final_semantic_results = []
final_semantic_cache_reuse_count = 0
final_semantic_live_request_count = 0

print("=== FINAL SEMANTIC-JUDGE EVALUATION ===")
print("Records:", len(semantic_repaired_experiment))

for position, record in enumerate(
    semantic_repaired_experiment,
    1,
):
    cache_key = final_semantic_cache_key(record)

    cache_path = (
        final_semantic_cache_directory
        / f"{cache_key}.json"
    )

    cached_result = None

    if cache_path.is_file():
        candidate_cache = read_json(cache_path)

        if valid_final_semantic_judgment(
            candidate_cache.get("payload")
        ):
            cached_result = candidate_cache

    if cached_result is not None:
        result = {
            **cached_result,
            "cache_reused": True,
        }
        final_semantic_cache_reuse_count += 1

    else:
        result = None
        last_error = None

        for attempt in range(1, 4):
            try:
                response_record = call_final_semantic_judge(
                    final_semantic_judge_prompt(record)
                )

                final_semantic_live_request_count += 1

                payload = response_record["payload"]

                if not valid_final_semantic_judgment(payload):
                    last_error = "invalid_judgment_payload"
                    continue

                criterion_fields = (
                    "question_natural",
                    "question_self_contained",
                    "answer_directly_answers",
                    "answer_complete",
                    "evidence_supports_answer",
                    "no_ambiguous_reference",
                    "no_question_answer_leakage",
                )

                if (
                    payload["decision"] == "PASS"
                    and not all(
                        payload[field]
                        for field in criterion_fields
                    )
                ):
                    payload["decision"] = "REVIEW"

                result = {
                    "judge_version": (
                        FINAL_SEMANTIC_JUDGE_VERSION
                    ),
                    "candidate_id": record["candidate_id"],
                    "source_seed_id": record[
                        "source_seed_id"
                    ],
                    "model": chat_model,
                    "attempt": attempt,
                    "payload": payload,
                    "raw_response": response_record[
                        "raw_response"
                    ],
                    "finish_reason": response_record[
                        "finish_reason"
                    ],
                    "observed_cost": response_record[
                        "observed_cost"
                    ],
                    "request_error": None,
                    "created_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "cache_reused": False,
                }

                write_json_atomic(cache_path, result)
                break

            except Exception as error:
                final_semantic_live_request_count += 1
                last_error = (
                    f"{type(error).__name__}: {error}"
                )

        if result is None:
            result = {
                "judge_version": FINAL_SEMANTIC_JUDGE_VERSION,
                "candidate_id": record["candidate_id"],
                "source_seed_id": record["source_seed_id"],
                "model": chat_model,
                "attempt": 3,
                "payload": None,
                "raw_response": "",
                "finish_reason": None,
                "observed_cost": None,
                "request_error": last_error,
                "created_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "cache_reused": False,
            }

    final_semantic_results.append(result)

    decision = (
        result["payload"]["decision"]
        if valid_final_semantic_judgment(
            result.get("payload")
        )
        else "FAILED"
    )

    print(
        f"Record {position}/"
        f"{len(semantic_repaired_experiment)} | "
        f"seed={record['source_seed_id']} | "
        f"{decision}"
    )

assert len(final_semantic_results) == 47

final_semantic_pass_rows = [
    row
    for row in final_semantic_results
    if (
        valid_final_semantic_judgment(row.get("payload"))
        and row["payload"]["decision"] == "PASS"
    )
]

final_semantic_review_rows = [
    row
    for row in final_semantic_results
    if (
        valid_final_semantic_judgment(row.get("payload"))
        and row["payload"]["decision"] == "REVIEW"
    )
]

final_semantic_reject_rows = [
    row
    for row in final_semantic_results
    if (
        valid_final_semantic_judgment(row.get("payload"))
        and row["payload"]["decision"] == "REJECT"
    )
]

final_semantic_failed_rows = [
    row
    for row in final_semantic_results
    if not valid_final_semantic_judgment(
        row.get("payload")
    )
]

observed_costs = [
    float(row["observed_cost"])
    for row in final_semantic_results
    if row.get("observed_cost") is not None
]

final_semantic_report = {
    "judge_version": FINAL_SEMANTIC_JUDGE_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "active_v1_modified": False,
    "experiment_path": str(
        semantic_repaired_experiment_path
    ),
    "experiment_sha256": sha256_file(
        semantic_repaired_experiment_path
    ),
    "record_count": len(final_semantic_results),
    "pass_count": len(final_semantic_pass_rows),
    "review_count": len(final_semantic_review_rows),
    "reject_count": len(final_semantic_reject_rows),
    "failed_count": len(final_semantic_failed_rows),
    "cache_reuse_count": final_semantic_cache_reuse_count,
    "live_request_count": final_semantic_live_request_count,
    "total_observed_cost": (
        float(sum(observed_costs))
        if observed_costs
        else None
    ),
    "results": final_semantic_results,
}

final_semantic_report_path = (
    golden_dir
    / (
        f"{SEMANTIC_REPAIRED_EXPERIMENT_NAME}"
        "_final_semantic_judge.json"
    )
)

write_json_atomic(
    final_semantic_report_path,
    final_semantic_report,
)

print("\nSemantic PASS:", len(final_semantic_pass_rows))
print("Semantic REVIEW:", len(final_semantic_review_rows))
print("Semantic REJECT:", len(final_semantic_reject_rows))
print(
    "Request/parse failures:",
    len(final_semantic_failed_rows),
)
print(
    "Cached results reused:",
    final_semantic_cache_reuse_count,
)
print(
    "Live requests:",
    final_semantic_live_request_count,
)
print(
    "Total observed cost:",
    (
        f"{sum(observed_costs):.6f}"
        if observed_costs
        else "N/A"
    ),
)
print("\nReport:", final_semantic_report_path)
print("\nFINAL SEMANTIC-JUDGE EVALUATION: COMPLETE")

=== FINAL SEMANTIC-JUDGE EVALUATION ===
Records: 47
Record 1/47 | seed=1 | PASS
Record 2/47 | seed=2 | PASS
Record 3/47 | seed=2 | PASS
Record 4/47 | seed=3 | PASS
Record 5/47 | seed=3 | PASS
Record 6/47 | seed=3 | PASS
Record 7/47 | seed=4 | PASS
Record 8/47 | seed=4 | PASS
Record 9/47 | seed=4 | PASS
Record 10/47 | seed=5 | PASS
Record 11/47 | seed=6 | REVIEW
Record 12/47 | seed=6 | PASS
Record 13/47 | seed=7 | PASS
Record 14/47 | seed=8 | PASS
Record 15/47 | seed=8 | PASS
Record 16/47 | seed=8 | PASS
Record 17/47 | seed=9 | PASS
Record 18/47 | seed=9 | PASS
Record 19/47 | seed=10 | REVIEW
Record 20/47 | seed=11 | PASS
Record 21/47 | seed=12 | PASS
Record 22/47 | seed=12 | PASS
Record 23/47 | seed=12 | PASS
Record 24/47 | seed=13 | REJECT
Record 25/47 | seed=13 | PASS
Record 26/47 | seed=14 | PASS
Record 27/47 | seed=15 | PASS
Record 28/47 | seed=16 | PASS
Record 29/47 | seed=16 | PASS
Record 30/47 | seed=16 | PASS
Record 31/47 | seed=17 | REVIEW
Record 32/47 | seed=17 | REVIEW
Recor

In [43]:
# Code Cell 36
# NB12G — Final semantic findings packet

assert "final_semantic_results" in globals(), (
    "Run Code Cell 35 first."
)
assert "semantic_repaired_experiment" in globals(), (
    "Run Code Cell 33 first."
)
assert len(final_semantic_results) == 47
assert len(semantic_repaired_experiment) == 47

FINAL_FINDINGS_PACKET_VERSION = (
    "nb12g_final_semantic_findings_packet_v1"
)

records_by_candidate_id = {
    record["candidate_id"]: record
    for record in semantic_repaired_experiment
}

final_semantic_findings = []

for position, result in enumerate(
    final_semantic_results,
    1,
):
    payload = result.get("payload")

    if not valid_final_semantic_judgment(payload):
        final_semantic_findings.append({
            "position": position,
            "candidate_id": result["candidate_id"],
            "source_seed_id": result["source_seed_id"],
            "decision": "FAILED",
            "request_error": result.get("request_error"),
        })
        continue

    if payload["decision"] == "PASS":
        continue

    record = records_by_candidate_id[
        result["candidate_id"]
    ]

    failed_criteria = [
        field
        for field in (
            "question_natural",
            "question_self_contained",
            "answer_directly_answers",
            "answer_complete",
            "evidence_supports_answer",
            "no_ambiguous_reference",
            "no_question_answer_leakage",
        )
        if not payload[field]
    ]

    final_semantic_findings.append({
        "position": position,
        "candidate_id": record["candidate_id"],
        "source_seed_id": record["source_seed_id"],
        "decision": payload["decision"],
        "reasons": payload["reasons"],
        "failed_criteria": failed_criteria,
        "question": record["question"],
        "expected_answer": record["expected_answer"],
        "source_evidence": record["source_evidence"],
        "question_intent": record["question_intent"],
        "retrieval_labels": record["retrieval_labels"],
        "curation": record["curation"],
        "manual_decision": record.get("manual_decision"),
        "replacement_question": record.get(
            "replacement_question"
        ),
        "replacement_answer": record.get(
            "replacement_answer"
        ),
        "manual_notes": record.get("manual_notes"),
    })

assert len(final_semantic_findings) == 9

decision_counts = {
    decision: sum(
        finding["decision"] == decision
        for finding in final_semantic_findings
    )
    for decision in ("REVIEW", "REJECT", "FAILED")
}

assert decision_counts == {
    "REVIEW": 7,
    "REJECT": 2,
    "FAILED": 0,
}

final_findings_packet = {
    "packet_version": FINAL_FINDINGS_PACKET_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "active_dataset_created": False,
    "active_v1_modified": False,
    "experiment_path": str(
        semantic_repaired_experiment_path
    ),
    "experiment_sha256": sha256_file(
        semantic_repaired_experiment_path
    ),
    "finding_count": len(final_semantic_findings),
    "decision_counts": decision_counts,
    "records": final_semantic_findings,
}

final_findings_packet_path = (
    golden_dir
    / (
        f"{SEMANTIC_REPAIRED_EXPERIMENT_NAME}"
        "_final_semantic_findings_packet.json"
    )
)

write_json_atomic(
    final_findings_packet_path,
    final_findings_packet,
)

print("=== FINAL SEMANTIC FINDINGS PACKET ===")
print("Findings:", len(final_semantic_findings))
print("REVIEW:", decision_counts["REVIEW"])
print("REJECT:", decision_counts["REJECT"])
print("FAILED:", decision_counts["FAILED"])

for finding in final_semantic_findings:
    print("\n" + "=" * 80)
    print(
        f"POSITION: {finding['position']} | "
        f"SEED: {finding['source_seed_id']} | "
        f"DECISION: {finding['decision']}"
    )
    print("Candidate:", finding["candidate_id"])
    print("\nQuestion:")
    print(finding["question"])
    print("\nAnswer:")
    print(finding["expected_answer"])
    print("\nEvidence:")
    print(finding["source_evidence"])
    print("\nJudge reasons:")
    for reason in finding["reasons"]:
        print(f"- {reason}")
    print("\nFailed criteria:")
    print(
        ", ".join(finding["failed_criteria"])
        or "None"
    )

print("\n" + "=" * 80)
print("Packet:", final_findings_packet_path)
print("\nFINAL SEMANTIC FINDINGS PACKET: COMPLETE")

=== FINAL SEMANTIC FINDINGS PACKET ===
Findings: 9
REVIEW: 7
REJECT: 2
FAILED: 0

POSITION: 11 | SEED: 6 | DECISION: REVIEW
Candidate: 6164aabd-795d-5850-89e7-66bdeb99694b

Question:
گزاره‌های علمی و نمونه‌های عملی در طول رشد فرد چه تغییری می‌کنند؟

Answer:
توسعه یافته سطح قبلی خود است

Evidence:
هر حوزه از استاندارد شامل گزارهای علمی و نمونه عملی است؛ نمونه عملی، توانایی‌های متناظر با هر گزاره علمی در یک مقطع سنی را فهرست می‌کند؛ هر گزاره علمی و نمونه عملی، توسعه یافته سطح قبلی خود است؛ به بیان دیگر، یک گزاره دانشی و چارچوب راهنمای آن در طول رشد فرد توسعه می‌یابد.

Judge reasons:
- پاسخ کوتاه و غامض است و جزئیات لازم را ارائه نمی‌کند (مثلاً چگونگی تغییر یا توسعه گزاره‌ها و نمونه‌ها).
- پاسخ به طور کامل توسط محتوای اثبات پشتیبانی نمی‌شود، زیرا اثبات به توسعه یافته بودن گزاره‌ها و نمونه‌ها اشاره دارد، اما پاسخ به طور دقیق چه چیزی توسعه یافته است را مشخص نمی‌کند.

Failed criteria:
answer_complete

POSITION: 19 | SEED: 10 | DECISION: REVIEW
Candidate: a1c77a2e-3eb7-5708-be3b-e8e3775b5510


In [44]:
# Code Cell 37
# NB12G — Final adjudicated semantic repair

from copy import deepcopy

assert "semantic_repaired_experiment" in globals(), (
    "Run Code Cell 33 first."
)
assert len(semantic_repaired_experiment) == 47

FINAL_ADJUDICATED_EXPERIMENT_NAME = (
    "golden_dataset_quote_grounded_final_adjudicated_experiment"
)
FINAL_ADJUDICATION_VERSION = (
    "nb12g_final_semantic_adjudication_v1"
)

final_adjudications = {
    "6164aabd-795d-5850-89e7-66bdeb99694b": {
        "decision": "REPAIR",
        "question": (
            "گزاره‌های علمی و نمونه‌های عملی در طول "
            "رشد فرد چه تغییری می‌کنند؟"
        ),
        "answer": (
            "هر گزاره علمی و نمونه عملی، توسعه یافته "
            "سطح قبلی خود است"
        ),
        "note": "Expanded the answer into a complete proposition.",
    },
    "a1c77a2e-3eb7-5708-be3b-e8e3775b5510": {
        "decision": "REPAIR",
        "question": (
            "از دیدگاه کلان، آموزش سواد مالی چگونه "
            "توصیف شده است؟"
        ),
        "answer": (
            "آموزش سواد مالی وقتی از بالا بهش نگاه "
            "می‌کنیم یکی از ضرورتهای مهمه"
        ),
        "note": "Used the full evidence-grounded proposition.",
    },
    "c7903509-29cf-5a27-b714-4de21195b14f": {
        "decision": "REMOVE",
        "note": (
            "The only supported answer is itself an interrogative "
            "financial-check question, which fails the final "
            "non-question answer criterion."
        ),
    },
    "ac9f8f16-32c7-515c-a35a-635d11cbc43e": {
        "decision": "REPAIR",
        "question": (
            "اگر فرد در سه ماه اول هزینه‌های خود را ثبت نکند "
            "و از الگوی رفتاری مالی خود آگاه نباشد، "
            "چه اتفاقی می‌افتد؟"
        ),
        "answer": "احتمالاً بودجه‌بندی شدنی نباشه.",
        "note": "Removed the ambiguous second-person reference.",
    },
    "d80b0947-6810-5b4c-98c6-072ed6ea3bc4": {
        "decision": "REPAIR",
        "question": (
            "در سه ماه اول، چه روشی برای ثبت و دسته‌بندی "
            "هزینه‌ها پیشنهاد شد؟"
        ),
        "answer": (
            "تو اون سه ماه اول پیشنهادمون این بود که شما "
            "هزینه‌ها رو می‌نویسی و در آخر ماه می‌آیی "
            "دسته‌بندی از هزینه‌ها انجام می‌دیم."
        ),
        "note": "Made the three-month time scope explicit in answer.",
    },
    "6a6f6fac-f333-5a37-8390-be1504901e8f": {
        "decision": "REPAIR",
        "question": (
            "دانستن اصطلاحات قراردادهای بیمه چه کمکی "
            "به بیمه‌گذار می‌کند؟"
        ),
        "answer": (
            "از همه ظرفیت این قرارداد استفاده بکنین که "
            "بیشترین خسارت خودت رو جبران بکنین"
        ),
        "note": "Replaced the repetitive explanation with the direct benefit.",
    },
    "748c4efa-afc0-5833-9394-9c181b2658ef": {
        "decision": "REPAIR",
        "question": (
            "در خودارزیابی موقعیت فرد در پلکان توانمندی مالی، "
            "از او چه خواسته شده است؟"
        ),
        "answer": "چند جمله‌ای درباره وضعیت خود بنویسید.",
        "note": "Question now asks for the exact instruction in evidence.",
    },
    "c54f4c4a-c75c-5021-baf2-f307181016ef": {
        "decision": "REPAIR",
        "question": (
            "تجربه‌های شرکت‌کنندگان از اثرگذاری نوشتن هزینه‌ها "
            "بر چه نتایجی جذاب و شنیدنی بود؟"
        ),
        "answer": (
            "اثرگذاری نوشتن هزینه‌ها بر پیشگیری از ولخرجی و "
            "مدیریت بهتر هزینه‌ها، بدون کاهش کیفیت زندگی"
        ),
        "note": "Replaced the unnatural question formulation.",
    },
    "a864f7bb-249a-5986-9cdc-de072d8559c8": {
        "decision": "REPAIR",
        "question": (
            "چه وضعیتی ممکن است باعث فراموش‌شدن چیزهای "
            "ارزشمند اطراف‌مان شود؟"
        ),
        "answer": (
            "رسیدن به خواسته‌ها و فکر کردن به نداشته‌ها "
            "آنقدر به تار و پود زندگی‌مان گره می‌خورد که "
            "وجود چیزهای ارزشمند اطراف‌مان را یادمان می‌رود"
        ),
        "note": "Answer now states the direct causal relation.",
    },
}

assert len(final_adjudications) == 9

records_by_candidate_id = {
    record["candidate_id"]: record
    for record in semantic_repaired_experiment
}

assert set(final_adjudications) <= set(
    records_by_candidate_id
), "An adjudication references an unknown candidate."

grounding_failures = []

for candidate_id, decision in final_adjudications.items():
    if decision["decision"] != "REPAIR":
        continue

    source_evidence = records_by_candidate_id[
        candidate_id
    ]["source_evidence"]

    if canon(decision["answer"]) not in canon(source_evidence):
        grounding_failures.append({
            "candidate_id": candidate_id,
            "answer": decision["answer"],
        })

assert not grounding_failures, (
    "Ungrounded replacement answers: "
    + json.dumps(
        grounding_failures,
        ensure_ascii=False,
        indent=2,
    )
)

final_adjudicated_experiment = []
final_adjudication_log = []

for record in semantic_repaired_experiment:
    source_candidate_id = record["candidate_id"]
    decision = final_adjudications.get(source_candidate_id)

    if decision is not None and decision["decision"] == "REMOVE":
        final_adjudication_log.append({
            "source_candidate_id": source_candidate_id,
            "source_seed_id": record["source_seed_id"],
            "decision": "REMOVE",
            "reason": decision["note"],
        })
        continue

    output_record = deepcopy(record)

    output_record["dataset_version"] = (
        FINAL_ADJUDICATED_EXPERIMENT_NAME
    )

    if decision is None:
        output_record["manual_decision"] = "RETAIN"
        output_record["manual_notes"] = (
            "Retained after final semantic adjudication."
        )
        final_adjudicated_experiment.append(output_record)
        continue

    output_record["candidate_id"] = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            (
                f"{FINAL_ADJUDICATION_VERSION}|"
                f"{source_candidate_id}|"
                f"{canon(decision['question'])}|"
                f"{canon(decision['answer'])}"
            ),
        )
    )
    output_record["question"] = decision["question"]
    output_record["expected_answer"] = decision["answer"]

    output_record["curation"] = {
        "curation_version": FINAL_ADJUDICATION_VERSION,
        "decision": "FINAL_SEMANTIC_REPAIR",
        "source_candidate_id": source_candidate_id,
        "supersedes_candidate_id": source_candidate_id,
        "reason": decision["note"],
        "active_dataset_record": False,
    }
    output_record["manual_decision"] = "REPAIR"
    output_record["replacement_question"] = decision["question"]
    output_record["replacement_answer"] = decision["answer"]
    output_record["manual_notes"] = decision["note"]

    final_adjudicated_experiment.append(output_record)

    final_adjudication_log.append({
        "source_candidate_id": source_candidate_id,
        "candidate_id": output_record["candidate_id"],
        "source_seed_id": record["source_seed_id"],
        "decision": "REPAIR",
        "question": output_record["question"],
        "answer": output_record["expected_answer"],
        "reason": decision["note"],
    })

assert len(final_adjudicated_experiment) == 46

repair_count = sum(
    row["decision"] == "REPAIR"
    for row in final_adjudication_log
)
remove_count = sum(
    row["decision"] == "REMOVE"
    for row in final_adjudication_log
)

assert repair_count == 8
assert remove_count == 1

assert {
    record["dataset_version"]
    for record in final_adjudicated_experiment
} == {FINAL_ADJUDICATED_EXPERIMENT_NAME}

candidate_ids = [
    record["candidate_id"]
    for record in final_adjudicated_experiment
]
question_keys = [
    canon(record["question"])
    for record in final_adjudicated_experiment
]

assert len(candidate_ids) == len(set(candidate_ids))
assert len(question_keys) == len(set(question_keys))

final_adjudicated_experiment_path = (
    golden_dir
    / f"{FINAL_ADJUDICATED_EXPERIMENT_NAME}.jsonl"
)

write_jsonl_atomic(
    final_adjudicated_experiment_path,
    final_adjudicated_experiment,
)

final_adjudication_report = {
    "adjudication_version": FINAL_ADJUDICATION_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "READY_FOR_FINAL_AUDIT",
    "active_dataset_created": False,
    "active_v1_modified": False,
    "input_experiment_path": str(
        semantic_repaired_experiment_path
    ),
    "input_experiment_sha256": sha256_file(
        semantic_repaired_experiment_path
    ),
    "output_experiment_path": str(
        final_adjudicated_experiment_path
    ),
    "output_experiment_sha256": sha256_file(
        final_adjudicated_experiment_path
    ),
    "input_record_count": 47,
    "output_record_count": 46,
    "repair_count": repair_count,
    "remove_count": remove_count,
    "log": final_adjudication_log,
}

final_adjudication_report_path = (
    golden_dir
    / (
        f"{FINAL_ADJUDICATED_EXPERIMENT_NAME}"
        "_adjudication_report.json"
    )
)

write_json_atomic(
    final_adjudication_report_path,
    final_adjudication_report,
)

print("=== FINAL SEMANTIC ADJUDICATION ===")
print("Input records:", 47)
print("Repaired records:", repair_count)
print("Removed records:", remove_count)
print("Output records:", len(final_adjudicated_experiment))
print("Active Golden Dataset created:", False)
print("Golden Dataset v1 modified:", False)
print("\nExperiment:", final_adjudicated_experiment_path)
print("Report:", final_adjudication_report_path)
print("\nFINAL SEMANTIC ADJUDICATION: COMPLETE")

=== FINAL SEMANTIC ADJUDICATION ===
Input records: 47
Repaired records: 8
Removed records: 1
Output records: 46
Active Golden Dataset created: False
Golden Dataset v1 modified: False

Experiment: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_final_adjudicated_experiment.jsonl
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_final_adjudicated_experiment_adjudication_report.json

FINAL SEMANTIC ADJUDICATION: COMPLETE


In [45]:
# Code Cell 38
# NB12G — Final-adjudicated experiment integrity audit

assert "final_adjudicated_experiment" in globals(), (
    "Run Code Cell 37 first."
)
assert len(final_adjudicated_experiment) == 46

FINAL_ADJUDICATED_AUDIT_VERSION = (
    "nb12g_final_adjudicated_integrity_audit_v1"
)

ALLOWED_FALLBACK_SEED_IDS = {16}
ALLOWED_FALLBACK_LABEL_BASIS = {
    "EXACT_REVIEWED_EVIDENCE_FALLBACK",
}

final_adjudicated_integrity_rows = []

for record in final_adjudicated_experiment:
    errors = []
    accepted_exceptions = []

    candidate_id = record.get("candidate_id")
    seed_id = record.get("source_seed_id")

    missing_fields = sorted(
        REQUIRED_RECORD_FIELDS - set(record)
    )

    if missing_fields:
        errors.append({
            "code": "missing_required_fields",
            "fields": missing_fields,
        })

    try:
        uuid.UUID(str(candidate_id))
    except (
        ValueError,
        TypeError,
        AttributeError,
    ):
        errors.append({
            "code": "invalid_candidate_uuid",
        })

    if record.get("dataset_version") != (
        FINAL_ADJUDICATED_EXPERIMENT_NAME
    ):
        errors.append({
            "code": "invalid_dataset_version",
        })

    if seed_id not in seeds:
        errors.append({
            "code": "unknown_source_seed_id",
        })

    if record.get("evaluation_split") not in {
        "train",
        "dev",
        "test",
    }:
        errors.append({
            "code": "invalid_evaluation_split",
        })

    question = str(record.get("question", "")).strip()
    answer = str(
        record.get("expected_answer", "")
    ).strip()
    evidence = str(
        record.get("source_evidence", "")
    ).strip()
    evidence_quote = str(
        record.get("evidence_quote", "")
    ).strip()

    if not question.endswith("؟"):
        errors.append({
            "code": "invalid_question_ending",
        })

    if canon(answer) not in canon(evidence):
        errors.append({
            "code": "answer_not_in_source_evidence",
        })

    if canon(evidence) != canon(evidence_quote):
        errors.append({
            "code": "evidence_quote_mismatch",
        })

    labels = record.get("retrieval_labels", {})
    positive_chunk_ids = labels.get(
        "positive_chunk_ids",
        [],
    )
    positive_parent_chunk_ids = labels.get(
        "positive_parent_chunk_ids",
        [],
    )

    if labels.get("candidate") not in {
        "parent_child_children",
        "structure_aware_medium",
    }:
        errors.append({
            "code": "invalid_retrieval_candidate",
        })

    if not positive_chunk_ids:
        errors.append({
            "code": "empty_positive_chunk_ids",
        })

    missing_chunk_ids = [
        chunk_id
        for chunk_id in positive_chunk_ids
        if chunk_id not in all_retrieval_chunk_by_id
    ]

    if missing_chunk_ids:
        errors.append({
            "code": "missing_positive_chunks",
            "chunk_ids": missing_chunk_ids,
        })

    available_chunks = [
        all_retrieval_chunk_by_id[chunk_id]
        for chunk_id in positive_chunk_ids
        if chunk_id in all_retrieval_chunk_by_id
    ]

    evidence_in_positive_chunk = any(
        canon(evidence) in canon(chunk["content"])
        for chunk in available_chunks
    )

    if not evidence_in_positive_chunk:
        errors.append({
            "code": "evidence_not_in_positive_chunk",
        })

    missing_parent_ids = [
        parent_id
        for parent_id in positive_parent_chunk_ids
        if parent_id not in all_parent_chunk_ids
    ]

    if missing_parent_ids:
        errors.append({
            "code": "missing_positive_parents",
            "parent_ids": missing_parent_ids,
        })

    record_document_ids = set(
        record.get("document_ids", [])
    )
    positive_document_ids = set()

    for chunk in available_chunks:
        positive_document_ids.update(
            chunk_identifiers(chunk)
        )

    document_overlap = sorted(
        record_document_ids & positive_document_ids
    )

    if not document_overlap:
        fallback_is_allowed = (
            seed_id in ALLOWED_FALLBACK_SEED_IDS
            and labels.get("label_basis")
            in ALLOWED_FALLBACK_LABEL_BASIS
            and evidence_in_positive_chunk
        )

        if fallback_is_allowed:
            accepted_exceptions.append({
                "code": (
                    "accepted_exact_reviewed_evidence_fallback"
                ),
                "label_basis": labels.get("label_basis"),
            })
        else:
            errors.append({
                "code": (
                    "unapproved_missing_document_overlap"
                ),
                "label_basis": labels.get("label_basis"),
            })

    final_adjudicated_integrity_rows.append({
        "candidate_id": candidate_id,
        "source_seed_id": seed_id,
        "evaluation_split": record.get(
            "evaluation_split"
        ),
        "errors": errors,
        "accepted_exceptions": accepted_exceptions,
        "document_overlap": document_overlap,
        "status": (
            "FAIL"
            if errors
            else "PASS_WITH_ACCEPTED_EXCEPTION"
            if accepted_exceptions
            else "PASS"
        ),
    })

failure_rows = [
    row
    for row in final_adjudicated_integrity_rows
    if row["errors"]
]

exception_rows = [
    row
    for row in final_adjudicated_integrity_rows
    if row["accepted_exceptions"]
]

candidate_ids = [
    record["candidate_id"]
    for record in final_adjudicated_experiment
]
question_keys = [
    canon(record["question"])
    for record in final_adjudicated_experiment
]

assert len(candidate_ids) == len(set(candidate_ids))
assert len(question_keys) == len(set(question_keys))

splits_by_seed = defaultdict(set)

for record in final_adjudicated_experiment:
    splits_by_seed[record["source_seed_id"]].add(
        record["evaluation_split"]
    )

seed_split_failures = {
    seed_id: sorted(split_values)
    for seed_id, split_values in splits_by_seed.items()
    if len(split_values) != 1
}

covered_seed_ids = sorted(splits_by_seed)

assert covered_seed_ids == list(range(1, 29))
assert not seed_split_failures

evidence_splits = defaultdict(set)

for record in final_adjudicated_experiment:
    evidence_hash = hashlib.sha256(
        canon(record["source_evidence"]).encode("utf-8")
    ).hexdigest()

    evidence_splits[evidence_hash].add(
        record["evaluation_split"]
    )

cross_split_evidence_leakage = [
    {
        "evidence_hash": evidence_hash,
        "splits": sorted(split_values),
    }
    for evidence_hash, split_values in evidence_splits.items()
    if len(split_values) > 1
]

assert not cross_split_evidence_leakage

assert len(exception_rows) == 3
assert {
    row["source_seed_id"]
    for row in exception_rows
} == {16}

assert not failure_rows, (
    "Final integrity failures: "
    + json.dumps(
        failure_rows,
        ensure_ascii=False,
        indent=2,
    )
)

final_adjudicated_integrity_report = {
    "audit_version": FINAL_ADJUDICATED_AUDIT_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "PASS_WITH_DOCUMENTED_SEED16_FALLBACK",
    "active_dataset_created": False,
    "active_v1_modified": False,
    "experiment_path": str(
        final_adjudicated_experiment_path
    ),
    "experiment_sha256": sha256_file(
        final_adjudicated_experiment_path
    ),
    "record_count": len(final_adjudicated_experiment),
    "covered_seed_count": len(covered_seed_ids),
    "covered_seed_ids": covered_seed_ids,
    "candidate_ids_unique": True,
    "normalized_questions_unique": True,
    "seed_split_consistency": True,
    "cross_split_evidence_leakage_count": 0,
    "integrity_failure_count": 0,
    "accepted_exception_count": len(exception_rows),
    "accepted_exception_seed_ids": [16],
    "rows": final_adjudicated_integrity_rows,
}

final_adjudicated_integrity_report_path = (
    golden_dir
    / (
        f"{FINAL_ADJUDICATED_EXPERIMENT_NAME}"
        "_final_integrity_audit.json"
    )
)

write_json_atomic(
    final_adjudicated_integrity_report_path,
    final_adjudicated_integrity_report,
)

print("=== FINAL-ADJUDICATED INTEGRITY AUDIT ===")
print("Records:", len(final_adjudicated_experiment))
print("Covered seeds:", len(covered_seed_ids))
print("Integrity failures:", len(failure_rows))
print("Accepted exceptions:", len(exception_rows))
print("Accepted-exception seeds:", sorted({
    row["source_seed_id"]
    for row in exception_rows
}))
print("Candidate IDs unique:", True)
print("Normalized questions unique:", True)
print("Seed split failures:", len(seed_split_failures))
print(
    "Cross-split exact-evidence leakage:",
    len(cross_split_evidence_leakage),
)
print("Active Golden Dataset created:", False)
print("Golden Dataset v1 modified:", False)
print("\nReport:", final_adjudicated_integrity_report_path)
print("\nFINAL-ADJUDICATED INTEGRITY AUDIT: PASS")

=== FINAL-ADJUDICATED INTEGRITY AUDIT ===
Records: 46
Covered seeds: 28
Integrity failures: 0
Accepted exceptions: 3
Accepted-exception seeds: [16]
Candidate IDs unique: True
Normalized questions unique: True
Seed split failures: 0
Cross-split exact-evidence leakage: 0
Active Golden Dataset created: False
Golden Dataset v1 modified: False

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_final_adjudicated_experiment_final_integrity_audit.json

FINAL-ADJUDICATED INTEGRITY AUDIT: PASS


In [46]:
# Code Cell 39
# NB12G — Targeted final semantic re-judge

assert "final_adjudicated_experiment" in globals(), (
    "Run Code Cell 37 first."
)
assert "final_semantic_results" in globals(), (
    "Run Code Cell 35 first."
)
assert "chat" in globals(), "Run Code Cell 08 first."
assert len(final_adjudicated_experiment) == 46
assert len(final_semantic_results) == 47

TARGETED_FINAL_JUDGE_VERSION = (
    "nb12g_targeted_final_semantic_judge_v1"
)

targeted_judge_cache_directory = (
    golden_dir
    / "cache"
    / FINAL_ADJUDICATED_EXPERIMENT_NAME
    / "semantic_judge"
    / TARGETED_FINAL_JUDGE_VERSION
)

targeted_judge_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)

previous_results_by_candidate_id = {
    result["candidate_id"]: result
    for result in final_semantic_results
}

changed_records = [
    record
    for record in final_adjudicated_experiment
    if record.get("curation", {}).get(
        "curation_version"
    ) == FINAL_ADJUDICATION_VERSION
]

unchanged_records = [
    record
    for record in final_adjudicated_experiment
    if record not in changed_records
]

assert len(changed_records) == 8
assert len(unchanged_records) == 38

def targeted_final_judge_prompt(record):
    return f"""
نقش شما داور نهایی و سخت‌گیر Golden Dataset فارسی است.

فقط بر اساس QUESTION، ANSWER و EVIDENCE تصمیم بگیر.

معیارها:

1. سؤال باید فارسی طبیعی، روشن و مستقل باشد.
2. سؤال نباید مرجع یا ضمیر مبهم داشته باشد.
3. پاسخ باید مستقیماً به سؤال جواب بدهد.
4. پاسخ باید کامل و معنادار باشد.
5. پاسخ باید کاملاً توسط EVIDENCE پشتیبانی شود.
6. سؤال نباید پاسخ را در خود تکرار کند.
7. دانش بیرونی نباید لازم باشد.
8. اگر همه معیارها برقرارند PASS بده.
9. اگر اصلاح محدود لازم است REVIEW بده.
10. اگر evidence برای سؤال مستقل کافی نیست REJECT بده.

خروجی فقط JSON معتبر:

{{
  "decision": "PASS|REVIEW|REJECT",
  "question_natural": true,
  "question_self_contained": true,
  "answer_directly_answers": true,
  "answer_complete": true,
  "evidence_supports_answer": true,
  "no_ambiguous_reference": true,
  "no_question_answer_leakage": true,
  "reasons": ["..."]
}}

QUESTION:
{record["question"]}

ANSWER:
{record["expected_answer"]}

EVIDENCE:
{record["source_evidence"]}
""".strip()

def targeted_cache_key(record):
    material = {
        "judge_version": TARGETED_FINAL_JUDGE_VERSION,
        "model": chat_model,
        "candidate_id": record["candidate_id"],
        "question": canon(record["question"]),
        "answer": canon(record["expected_answer"]),
        "evidence": canon(record["source_evidence"]),
    }

    return hashlib.sha256(
        json.dumps(
            material,
            ensure_ascii=False,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()

def call_targeted_judge(record):
    response = chat.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Return only one valid JSON object. "
                    "Do not include markdown."
                ),
            },
            {
                "role": "user",
                "content": targeted_final_judge_prompt(record),
            },
        ],
        temperature=0,
        max_tokens=1536,
    )

    raw_response = (
        response.choices[0].message.content
        or ""
    ).strip()

    payload = parse_json_object(raw_response)

    usage = getattr(response, "usage", None)
    observed_cost = None

    if usage is not None:
        observed_cost = getattr(usage, "cost", None)

        if observed_cost is None and hasattr(
            usage,
            "model_dump",
        ):
            observed_cost = usage.model_dump().get("cost")

    return {
        "payload": payload,
        "raw_response": raw_response,
        "finish_reason": response.choices[0].finish_reason,
        "observed_cost": observed_cost,
    }

targeted_final_results = []
baseline_reused_count = 0
targeted_cache_reused_count = 0
targeted_live_request_count = 0

print("=== TARGETED FINAL SEMANTIC RE-JUDGE ===")
print("Unchanged prior-PASS records:", len(unchanged_records))
print("Records requiring re-judge:", len(changed_records))

for record in unchanged_records:
    previous_result = previous_results_by_candidate_id.get(
        record["candidate_id"]
    )

    assert previous_result is not None, (
        "Missing prior result for unchanged record: "
        f"{record['candidate_id']}"
    )

    assert valid_final_semantic_judgment(
        previous_result.get("payload")
    ), (
        "Invalid prior semantic result for unchanged record: "
        f"{record['candidate_id']}"
    )

    assert previous_result["payload"]["decision"] == "PASS", (
        "Unchanged record was not PASS before final adjudication: "
        f"{record['candidate_id']}"
    )

    targeted_final_results.append({
        **deepcopy(previous_result),
        "judge_version": TARGETED_FINAL_JUDGE_VERSION,
        "reused_from_prior_pass": True,
        "cache_reused": True,
    })

    baseline_reused_count += 1

for position, record in enumerate(changed_records, 1):
    cache_path = (
        targeted_judge_cache_directory
        / f"{targeted_cache_key(record)}.json"
    )

    result = None

    if cache_path.is_file():
        cached_result = read_json(cache_path)

        if valid_final_semantic_judgment(
            cached_result.get("payload")
        ):
            result = {
                **cached_result,
                "cache_reused": True,
            }
            targeted_cache_reused_count += 1

    if result is None:
        last_error = None

        for attempt in range(1, 4):
            try:
                response_record = call_targeted_judge(record)
                targeted_live_request_count += 1

                payload = response_record["payload"]

                if not valid_final_semantic_judgment(payload):
                    last_error = "invalid_judgment_payload"
                    continue

                criterion_fields = (
                    "question_natural",
                    "question_self_contained",
                    "answer_directly_answers",
                    "answer_complete",
                    "evidence_supports_answer",
                    "no_ambiguous_reference",
                    "no_question_answer_leakage",
                )

                if (
                    payload["decision"] == "PASS"
                    and not all(
                        payload[field]
                        for field in criterion_fields
                    )
                ):
                    payload["decision"] = "REVIEW"

                result = {
                    "judge_version": (
                        TARGETED_FINAL_JUDGE_VERSION
                    ),
                    "candidate_id": record["candidate_id"],
                    "source_seed_id": record[
                        "source_seed_id"
                    ],
                    "model": chat_model,
                    "attempt": attempt,
                    "payload": payload,
                    "raw_response": response_record[
                        "raw_response"
                    ],
                    "finish_reason": response_record[
                        "finish_reason"
                    ],
                    "observed_cost": response_record[
                        "observed_cost"
                    ],
                    "request_error": None,
                    "created_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "cache_reused": False,
                    "reused_from_prior_pass": False,
                }

                write_json_atomic(cache_path, result)
                break

            except Exception as error:
                targeted_live_request_count += 1
                last_error = (
                    f"{type(error).__name__}: {error}"
                )

        if result is None:
            result = {
                "judge_version": TARGETED_FINAL_JUDGE_VERSION,
                "candidate_id": record["candidate_id"],
                "source_seed_id": record["source_seed_id"],
                "model": chat_model,
                "attempt": 3,
                "payload": None,
                "raw_response": "",
                "finish_reason": None,
                "observed_cost": None,
                "request_error": last_error,
                "created_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "cache_reused": False,
                "reused_from_prior_pass": False,
            }

    targeted_final_results.append(result)

    decision = (
        result["payload"]["decision"]
        if valid_final_semantic_judgment(
            result.get("payload")
        )
        else "FAILED"
    )

    print(
        f"Re-judged {position}/{len(changed_records)} | "
        f"seed={record['source_seed_id']} | {decision}"
    )

assert len(targeted_final_results) == 46

targeted_pass_rows = [
    row
    for row in targeted_final_results
    if (
        valid_final_semantic_judgment(row.get("payload"))
        and row["payload"]["decision"] == "PASS"
    )
]

targeted_review_rows = [
    row
    for row in targeted_final_results
    if (
        valid_final_semantic_judgment(row.get("payload"))
        and row["payload"]["decision"] == "REVIEW"
    )
]

targeted_reject_rows = [
    row
    for row in targeted_final_results
    if (
        valid_final_semantic_judgment(row.get("payload"))
        and row["payload"]["decision"] == "REJECT"
    )
]

targeted_failed_rows = [
    row
    for row in targeted_final_results
    if not valid_final_semantic_judgment(
        row.get("payload")
    )
]

observed_costs = [
    float(row["observed_cost"])
    for row in targeted_final_results
    if (
        row.get("observed_cost") is not None
        and not row.get("reused_from_prior_pass")
    )
]

targeted_final_status = (
    "PASS"
    if (
        not targeted_review_rows
        and not targeted_reject_rows
        and not targeted_failed_rows
    )
    else "FINDINGS_REQUIRING_REVIEW"
)

targeted_final_report = {
    "judge_version": TARGETED_FINAL_JUDGE_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": targeted_final_status,
    "active_dataset_created": False,
    "active_v1_modified": False,
    "experiment_path": str(
        final_adjudicated_experiment_path
    ),
    "experiment_sha256": sha256_file(
        final_adjudicated_experiment_path
    ),
    "record_count": len(targeted_final_results),
    "prior_pass_reuse_count": baseline_reused_count,
    "targeted_rejudge_count": len(changed_records),
    "cache_reuse_count": targeted_cache_reused_count,
    "live_request_count": targeted_live_request_count,
    "pass_count": len(targeted_pass_rows),
    "review_count": len(targeted_review_rows),
    "reject_count": len(targeted_reject_rows),
    "failed_count": len(targeted_failed_rows),
    "total_observed_cost": (
        float(sum(observed_costs))
        if observed_costs
        else None
    ),
    "results": targeted_final_results,
}

targeted_final_report_path = (
    golden_dir
    / (
        f"{FINAL_ADJUDICATED_EXPERIMENT_NAME}"
        "_targeted_final_semantic_judge.json"
    )
)

write_json_atomic(
    targeted_final_report_path,
    targeted_final_report,
)

print("\nSemantic PASS:", len(targeted_pass_rows))
print("Semantic REVIEW:", len(targeted_review_rows))
print("Semantic REJECT:", len(targeted_reject_rows))
print("Request/parse failures:", len(targeted_failed_rows))
print("Prior PASS results reused:", baseline_reused_count)
print("Targeted records re-judged:", len(changed_records))
print("Cached results reused:", targeted_cache_reused_count)
print("Live requests:", targeted_live_request_count)
print(
    "Total observed cost:",
    (
        f"{sum(observed_costs):.6f}"
        if observed_costs
        else "N/A"
    ),
)
print("\nReport:", targeted_final_report_path)
print("\nStatus:", targeted_final_status)
print("\nTARGETED FINAL SEMANTIC RE-JUDGE: COMPLETE")

=== TARGETED FINAL SEMANTIC RE-JUDGE ===
Unchanged prior-PASS records: 38
Records requiring re-judge: 8
Re-judged 1/8 | seed=6 | PASS
Re-judged 2/8 | seed=10 | PASS
Re-judged 3/8 | seed=17 | PASS
Re-judged 4/8 | seed=17 | PASS
Re-judged 5/8 | seed=21 | PASS
Re-judged 6/8 | seed=22 | PASS
Re-judged 7/8 | seed=25 | PASS
Re-judged 8/8 | seed=28 | PASS

Semantic PASS: 46
Semantic REVIEW: 0
Semantic REJECT: 0
Request/parse failures: 0
Prior PASS results reused: 38
Targeted records re-judged: 8
Cached results reused: 0
Live requests: 8
Total observed cost: 0.003858

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_final_adjudicated_experiment_targeted_final_semantic_judge.json

Status: PASS

TARGETED FINAL SEMANTIC RE-JUDGE: COMPLETE


In [47]:
# Code Cell 40
# NB12G — Formal final-experiment acceptance record

assert "final_adjudicated_integrity_report" in globals(), (
    "Run Code Cell 38 first."
)
assert "targeted_final_report" in globals(), (
    "Run Code Cell 39 first."
)
assert len(final_adjudicated_experiment) == 46

assert (
    final_adjudicated_integrity_report["integrity_failure_count"]
    == 0
)
assert (
    final_adjudicated_integrity_report[
        "accepted_exception_count"
    ]
    == 3
)
assert targeted_final_report["status"] == "PASS"
assert targeted_final_report["pass_count"] == 46
assert targeted_final_report["review_count"] == 0
assert targeted_final_report["reject_count"] == 0
assert targeted_final_report["failed_count"] == 0

assert (
    final_adjudicated_integrity_report["experiment_sha256"]
    == targeted_final_report["experiment_sha256"]
)

formal_reports_dir = (
    golden_dir.parents[2]
    / "reports"
    / "phase2"
)
formal_reports_dir.mkdir(
    parents=True,
    exist_ok=True,
)

final_acceptance_record = {
    "acceptance_version": (
        "golden_dataset_final_adjudicated_acceptance_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        "ACCEPTED_EXPERIMENT_NOT_ACTIVE_DATASET"
    ),
    "active_dataset_created": False,
    "active_v1_modified": False,
    "accepted_experiment_path": str(
        final_adjudicated_experiment_path
    ),
    "accepted_experiment_sha256": sha256_file(
        final_adjudicated_experiment_path
    ),
    "record_count": 46,
    "covered_seed_count": 28,
    "covered_seed_ids": list(range(1, 29)),
    "integrity": {
        "status": final_adjudicated_integrity_report[
            "status"
        ],
        "failure_count": 0,
        "accepted_exception_count": 3,
        "accepted_exception_seed_ids": [16],
        "audit_path": str(
            final_adjudicated_integrity_report_path
        ),
    },
    "semantic_judgment": {
        "status": targeted_final_report["status"],
        "pass_count": 46,
        "review_count": 0,
        "reject_count": 0,
        "failed_count": 0,
        "prior_pass_reuse_count": 38,
        "targeted_rejudge_count": 8,
        "report_path": str(targeted_final_report_path),
    },
    "lineage": {
        "initial_integrity_repaired_record_count": 49,
        "first_semantic_repair_output_count": 47,
        "final_adjudicated_output_count": 46,
        "first_semantic_repair_count": 9,
        "first_semantic_remove_count": 2,
        "final_adjudication_repair_count": 8,
        "final_adjudication_remove_count": 1,
    },
    "activation_rule": (
        "This is an accepted experimental artifact only. "
        "It must not replace an active Golden Dataset version "
        "without an explicit activation decision."
    ),
}

final_acceptance_path = (
    formal_reports_dir
    / "golden_dataset_final_adjudicated_acceptance_v1.json"
)

write_json_atomic(
    final_acceptance_path,
    final_acceptance_record,
)

print("=== FINAL EXPERIMENT ACCEPTANCE ===")
print("Status:", final_acceptance_record["status"])
print("Accepted records:", final_acceptance_record["record_count"])
print(
    "Integrity failures:",
    final_acceptance_record["integrity"]["failure_count"],
)
print(
    "Semantic PASS:",
    final_acceptance_record[
        "semantic_judgment"
    ]["pass_count"],
)
print("Active Golden Dataset created:", False)
print("Golden Dataset v1 modified:", False)
print("\nAcceptance record:", final_acceptance_path)
print("\nFINAL EXPERIMENT ACCEPTANCE: PASS")

=== FINAL EXPERIMENT ACCEPTANCE ===
Status: ACCEPTED_EXPERIMENT_NOT_ACTIVE_DATASET
Accepted records: 46
Integrity failures: 0
Semantic PASS: 46
Active Golden Dataset created: False
Golden Dataset v1 modified: False

Acceptance record: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/reports/phase2/golden_dataset_final_adjudicated_acceptance_v1.json

FINAL EXPERIMENT ACCEPTANCE: PASS
